In [ ]:
import sys
from pathlib import Path

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Notebook directory:", Path.cwd())

try:
    import flask
    print("Flask version:", flask.__version__)
except ImportError:
    print("Flask is not available in this kernel.")

In [ ]:
# SIGNALHER — CELL 2
# Project configuration and incident data model

from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from pathlib import Path
from uuid import uuid4


# --------------------------------------------------
# 1. Resolve the project root
# --------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

EXPECTED_DIRECTORIES = [
    "src/signalher",
    "data/synthetic",
    "outputs",
    "tests",
]

print("Project root:", PROJECT_ROOT)

for directory in EXPECTED_DIRECTORIES:
    path = PROJECT_ROOT / directory
    print(f"{directory}: {'OK' if path.is_dir() else 'MISSING'}")


# --------------------------------------------------
# 2. Supported incident source platforms
# --------------------------------------------------

class Platform(str, Enum):
    WHATSAPP = "whatsapp"
    INSTAGRAM = "instagram"
    EMAIL = "email"
    LINKEDIN = "linkedin"
    OTHER = "other"


# --------------------------------------------------
# 3. Incident data model
# --------------------------------------------------

@dataclass(slots=True)
class Incident:
    """
    Represents one user-selected digital safety incident.

    This model does not independently determine whether
    abuse has occurred. It stores information for later,
    explainable analysis.
    """

    platform: Platform
    occurred_at: datetime
    message_text: str

    user_note: str = ""
    source_reference: str | None = None

    incident_id: str = field(
        default_factory=lambda: str(uuid4())
    )

    def __post_init__(self):
        # Normalize platform input.
        if isinstance(self.platform, str):
            try:
                self.platform = Platform(
                    self.platform.strip().lower()
                )
            except ValueError as exc:
                raise ValueError(
                    "Unsupported platform. Use WhatsApp, "
                    "Instagram, Email, LinkedIn, or Other."
                ) from exc

        if not isinstance(self.platform, Platform):
            raise TypeError("platform must be a Platform value.")

        # Require a valid, timezone-aware timestamp.
        if not isinstance(self.occurred_at, datetime):
            raise TypeError(
                "occurred_at must be a datetime object."
            )

        if (
            self.occurred_at.tzinfo is None
            or self.occurred_at.utcoffset() is None
        ):
            raise ValueError(
                "occurred_at must include a timezone."
            )

        # Validate message text.
        if not isinstance(self.message_text, str):
            raise TypeError("message_text must be a string.")

        self.message_text = self.message_text.strip()

        if not self.message_text:
            raise ValueError(
                "Incident message cannot be empty."
            )

        if len(self.message_text) > 10_000:
            raise ValueError(
                "Message exceeds the 10,000-character limit."
            )

        # Validate optional note.
        self.user_note = self.user_note.strip()

        if len(self.user_note) > 1_500:
            raise ValueError(
                "User note exceeds the 1,500-character limit."
            )

    def to_dict(self, include_sensitive=False):
        """
        Create a dictionary for processing or export.

        By default, raw messages and user notes are excluded.
        Include them only when explicitly required by the user.
        """

        record = {
            "incident_id": self.incident_id,
            "platform": self.platform.value,
            "occurred_at": self.occurred_at.isoformat(),
            "source_reference": self.source_reference,
        }

        if include_sensitive:
            record["message_text"] = self.message_text
            record["user_note"] = self.user_note

        return record


print("\nIncident model defined successfully.")

In [ ]:
# SIGNALHER — CELL 3
# Validate our first incident record

test_incident = Incident(
    platform="WhatsApp",
    occurred_at=datetime(
        2026, 10, 24, 12, 0,
        tzinfo=timezone.utc
    ),
    message_text=(
        "[SYNTHETIC TEST] Please stop contacting me."
    ),
    user_note="Fictional example for backend testing."
)

safe_record = test_incident.to_dict()

assert test_incident.platform == Platform.WHATSAPP
assert test_incident.incident_id
assert "message_text" not in safe_record
assert "user_note" not in safe_record

print("Platform:", test_incident.platform.value)
print("Incident ID:", test_incident.incident_id)
print("Timestamp:", test_incident.occurred_at.isoformat())
print("Default export:", safe_record)
print("\nAll initial incident-model tests passed.")

In [ ]:
# SIGNALHER — CELL 4
# Explainable Digital Safety Signal Analyser
# Version 0.1 — Transparent, rule-based baseline

import re
from dataclasses import dataclass


@dataclass(frozen=True, slots=True)
class SignalRule:
    rule_id: str
    label: str
    category: str
    patterns: tuple[str, ...]
    explanation: str
    suggested_action: str


# Explicit patterns only. These are initial engineering rules,
# not a clinically validated or research-validated classifier.

SIGNAL_RULES = (
    SignalRule(
        rule_id="EXTORTION_THREAT",
        label="Possible coercion or extortion",
        category="coercion",
        patterns=(
            r"\b(?:pay me|send me money|send money|"
            r"transfer (?:me )?money|give me money)\b"
            r".{0,100}\b(?:or else|otherwise|or i'll|"
            r"or i will|i'll|i will)\b"
            r".{0,100}\b(?:leak|expose|post|publish|"
            r"share|send|distribute)\b",
            r"\bif you (?:don't|do not) pay\b"
            r".{0,100}\b(?:leak|expose|post|publish|"
            r"share|send|distribute)\b",
        ),
        explanation=(
            "The message appears to combine a demand for money "
            "with a possible threat or consequence. Context matters."
        ),
        suggested_action=(
            "Consider seeking trusted support, preserving relevant "
            "messages if safe, and using the appropriate platform "
            "reporting or official support channel."
        ),
    ),

    SignalRule(
        rule_id="THREAT_TO_EXPOSE_IMAGES",
        label="Possible threat to expose private images",
        category="image_based_abuse",
        patterns=(
            r"\b(?:i will|i'll|i am going to|i'm going to)\s+"
            r"(?:post|publish|share|send|leak|upload|expose|"
            r"distribute)\s+(?:your|the)\s+"
            r"(?:(?:private|intimate|personal)\s+)?"
            r"(?:photos?|pictures?|images?|videos?|nudes?|pics?)\b",
        ),
        explanation=(
            "The message contains language that may threaten "
            "to distribute someone's images without consent."
        ),
        suggested_action=(
            "If safe, preserve the message and relevant account "
            "details. Consider platform reporting and trusted "
            "support. Do not feel pressured to comply with threats."
        ),
    ),

    SignalRule(
        rule_id="COERCIVE_IMAGE_REQUEST",
        label="Possible pressure to share intimate images",
        category="coercion",
        patterns=(
            r"\b(?:send|sending|give|giving|show|showing|"
            r"share|sharing)\s+(?:me\s+)?(?:your\s+)?"
            r"(?:nude|naked|intimate|private)\s+"
            r"(?:photos?|pictures?|images?|videos?|pics?)\b",
            r"\b(?:prove you love me|if you love me|you owe me)\b"
            r".{0,80}\b(?:send|sending|show|showing|share|"
            r"sharing|give|giving)\b"
            r".{0,50}\b(?:nude|naked|intimate|private|"
            r"photos?|pictures?|images?|videos?)\b",
        ),
        explanation=(
            "The message may be pressuring someone to share "
            "private or intimate imagery. A request alone does not "
            "establish coercion; read it in context."
        ),
        suggested_action=(
            "The user is not obligated to share intimate material. "
            "They can decline, disengage if safe, or seek trusted "
            "support if they feel pressured."
        ),
    ),

    SignalRule(
        rule_id="DIRECT_THREAT_OR_STALKING_LANGUAGE",
        label="Possible threat or stalking-related language",
        category="physical_safety",
        patterns=(
            r"\bi know where you live\b",
            r"\b(?:i will|i'll|i am going to|i'm going to)\s+"
            r"(?:find you|hurt you|come to your "
            r"(?:home|house|workplace))\b",
            r"\b(?:i am|i'm)\s+(?:outside|waiting outside)\s+"
            r"(?:your|the)\s+(?:home|house|door)\b",
        ),
        explanation=(
            "The message contains potentially threatening "
            "language about someone's location or physical safety."
        ),
        suggested_action=(
            "If you believe there may be immediate danger, "
            "prioritise physical safety and contact local emergency "
            "services or a trusted person. This tool cannot assess "
            "the actual level of danger."
        ),
    ),

    SignalRule(
        rule_id="BOUNDARY_DISREGARD",
        label="Possible disregard of a stated boundary",
        category="coercion",
        patterns=(
            r"\bi don't care that you said no\b",
            r"\byou said no but\b.{0,80}"
            r"\b(?:send|share|show|meet|come)\b",
        ),
        explanation=(
            "The message may explicitly dismiss a refusal or "
            "continue a demand after a boundary was stated."
        ),
        suggested_action=(
            "Consider the wider conversation and your own safety. "
            "You can disengage if safe and seek trusted support."
        ),
    ),

    SignalRule(
        rule_id="IDENTITY_IMPERSONATION",
        label="Possible identity impersonation",
        category="identity_misuse",
        patterns=(
            r"\bpretending to be you\b",
            r"\busing your identity\b",
            r"\b(?:i made|i created)\s+(?:a )?fake "
            r"(?:account|profile)\s+"
            r"(?:pretending to be you|in your name)\b",
        ),
        explanation=(
            "The message contains an explicit reference to "
            "pretending to be someone else or using their identity."
        ),
        suggested_action=(
            "If relevant, preserve the account name or URL if safe, "
            "and consult the platform's official impersonation "
            "reporting process."
        ),
    ),
)


def analyze_incident(incident: Incident) -> dict:
    """
    Analyse one user-selected incident.

    Returns:
        Matched rule IDs, exact matched text, explanations,
        and suggested actions.

    Important:
        This baseline does not predict violence, determine guilt,
        or establish that abuse has occurred.
    """

    if not isinstance(incident, Incident):
        raise TypeError("incident must be an Incident object.")

    text = incident.message_text
    findings = []

    for rule in SIGNAL_RULES:
        for pattern in rule.patterns:
            match = re.search(pattern, text, flags=re.IGNORECASE)

            if match:
                findings.append({
                    "rule_id": rule.rule_id,
                    "label": rule.label,
                    "category": rule.category,
                    "evidence": match.group(0),
                    "explanation": rule.explanation,
                    "suggested_action": rule.suggested_action,
                })

                # Avoid duplicate findings from the same rule.
                break

    if findings:
        status = "potential_warning_indicators_found"
        message = (
            "Review the highlighted wording and wider context. "
            "These indicators are not a determination of abuse "
            "or a prediction of future harm."
        )
    else:
        status = "no_configured_pattern_matched"
        message = (
            "No configured warning pattern matched this message. "
            "This does not mean the interaction is safe or harmless."
        )

    return {
        "incident_id": incident.incident_id,
        "platform": incident.platform.value,
        "occurred_at": incident.occurred_at.isoformat(),
        "status": status,
        "findings": findings,
        "message": message,
    }


print("Signal rules loaded:", len(SIGNAL_RULES))
print("Analyser function defined successfully.")

In [ ]:
# SIGNALHER — CELL 5
# Synthetic scenario tests

TEST_SCENARIOS = [
    {
        "name": "Ordinary boundary statement",
        "text": "Please stop contacting me.",
        "expected": set(),
    },
    {
        "name": "Threat to expose private images",
        "text": "I'll leak your private photos.",
        "expected": {"THREAT_TO_EXPOSE_IMAGES"},
    },
    {
        "name": "Coercive intimate-image request",
        "text": "Prove you love me by sending me private pictures.",
        "expected": {"COERCIVE_IMAGE_REQUEST"},
    },
    {
        "name": "Money demand combined with a threat",
        "text": "Send me money or I'll leak your private photos.",
        "expected": {
            "EXTORTION_THREAT",
            "THREAT_TO_EXPOSE_IMAGES",
        },
    },
    {
        "name": "Threatening location language",
        "text": "I know where you live. I'll find you.",
        "expected": {"DIRECT_THREAT_OR_STALKING_LANGUAGE"},
    },
    {
        "name": "Ordinary photo request",
        "text": "Can you share your vacation photos?",
        "expected": set(),
    },
    {
        "name": "Explicit impersonation statement",
        "text": "I made a fake account pretending to be you.",
        "expected": {"IDENTITY_IMPERSONATION"},
    },
]

for scenario in TEST_SCENARIOS:
    incident = Incident(
        platform="other",
        occurred_at=datetime.now(timezone.utc),
        message_text=scenario["text"],
        user_note="Synthetic test scenario.",
    )

    result = analyze_incident(incident)

    actual = {
        finding["rule_id"]
        for finding in result["findings"]
    }

    passed = scenario["expected"].issubset(actual)

    # For intentionally benign examples, require zero configured matches.
    if not scenario["expected"]:
        passed = len(actual) == 0

    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{scenario['name']} | "
        f"Matched: {sorted(actual)}"
    )

    if not passed:
        raise AssertionError(
            f"Test failed: {scenario['name']}"
        )

print("\nAll initial analyser tests passed.")

In [ ]:
# SIGNALHER — CELL 6
# Timeline context and explicit incident linking

from dataclasses import dataclass
import re


@dataclass(frozen=True, slots=True)
class TimelineEntry:
    """
    Adds user-provided context to an existing Incident.

    case_id is optional. Entries are only linked together
    when the user explicitly assigns the same case_id.
    """

    incident: Incident
    direction: str = "unknown"
    case_id: str | None = None

    def __post_init__(self):
        if not isinstance(self.incident, Incident):
            raise TypeError("incident must be an Incident object.")

        direction = self.direction.strip().lower()

        if direction not in {"incoming", "outgoing", "unknown"}:
            raise ValueError(
                "direction must be incoming, outgoing, or unknown."
            )

        object.__setattr__(self, "direction", direction)

        if self.case_id is not None:
            case_id = self.case_id.strip()

            if not case_id:
                raise ValueError(
                    "case_id must be non-empty or None."
                )

            object.__setattr__(self, "case_id", case_id)


# These patterns indicate that the user may have stated a boundary.
# They are intentionally conservative and are not comprehensive.

BOUNDARY_PATTERNS = (
    r"\bplease do not contact me\b",
    r"\bplease don't contact me\b",
    r"\bdo not contact me\b",
    r"\bdon't contact me\b",
    r"\bstop contacting me\b",
    r"\bplease leave me alone\b",
    r"\bleave me alone\b",
    r"\bdo not message me\b",
    r"\bdon't message me\b",
    r"\bi said no\b",
)


def contains_boundary_statement(message: str) -> bool:
    """Check for explicit boundary language in supplied text."""

    return any(
        re.search(pattern, message, flags=re.IGNORECASE)
        for pattern in BOUNDARY_PATTERNS
    )


print("TimelineEntry model defined.")
print("Explicit boundary indicators:", len(BOUNDARY_PATTERNS))

In [ ]:
# SIGNALHER — CELL 7
# Timeline construction and contextual observations

from collections import defaultdict


def analyze_timeline(entries: list[TimelineEntry]) -> dict:
    """
    Build a chronological timeline and generate explainable
    observations for user-linked incidents.

    This function does not calculate a danger score, predict
    violence, or determine whether a crime has occurred.
    """

    if not entries:
        raise ValueError("Provide at least one timeline entry.")

    if not all(isinstance(e, TimelineEntry) for e in entries):
        raise TypeError(
            "Every item must be a TimelineEntry object."
        )

    # Prevent duplicate incident IDs in the same timeline.
    incident_ids = [e.incident.incident_id for e in entries]

    if len(incident_ids) != len(set(incident_ids)):
        raise ValueError("Duplicate incident IDs were provided.")

    # Analyse and order events.
    evaluated = []

    for entry in entries:
        result = analyze_incident(entry.incident)
        evaluated.append((entry, result))

    evaluated.sort(
        key=lambda pair: pair[0].incident.occurred_at
    )

    # Create a structured, chronological event list.
    events = []

    for entry, result in evaluated:
        events.append({
            "incident_id": entry.incident.incident_id,
            "occurred_at": entry.incident.occurred_at.isoformat(),
            "platform": entry.incident.platform.value,
            "direction": entry.direction,
            "case_id": entry.case_id,
            "status": result["status"],
            "findings": result["findings"],
        })

    # Only explicitly linked cases can produce
    # cross-incident observations.
    linked_cases = defaultdict(list)

    for entry, result in evaluated:
        if entry.case_id is not None:
            linked_cases[entry.case_id].append((entry, result))

    observations = []

    for case_id, case_entries in linked_cases.items():
        case_entries.sort(
            key=lambda pair: pair[0].incident.occurred_at
        )

        platforms = {
            entry.incident.platform.value
            for entry, _ in case_entries
        }

        warning_events = [
            entry for entry, result in case_entries
            if result["findings"]
        ]

        categories = {
            finding["category"]
            for _, result in case_entries
            for finding in result["findings"]
        }

        # Observation 1: incidents span multiple platforms.
        if len(platforms) > 1:
            observations.append({
                "case_id": case_id,
                "type": "multiple_platforms",
                "incident_ids": [
                    entry.incident.incident_id
                    for entry, _ in case_entries
                ],
                "explanation": (
                    "The user-linked entries span multiple platforms. "
                    "This is an organisational observation, not "
                    "independent verification of identity."
                ),
            })

        # Observation 2: multiple entries have configured flags.
        if len(warning_events) >= 2:
            observations.append({
                "case_id": case_id,
                "type": "multiple_warning_bearing_incidents",
                "incident_ids": [
                    entry.incident.incident_id
                    for entry in warning_events
                ],
                "explanation": (
                    "Multiple entries contain configured warning "
                    "indicators. Review each item in context; "
                    "this is not a danger score."
                ),
            })

        # Observation 3: more than one warning category appears.
        if len(categories) >= 2:
            observations.append({
                "case_id": case_id,
                "type": "multiple_signal_categories",
                "incident_ids": [
                    entry.incident.incident_id
                    for entry, result in case_entries
                    if result["findings"]
                ],
                "categories": sorted(categories),
                "explanation": (
                    "Different configured signal categories appear "
                    "in this user-linked case. This does not prove "
                    "that abuse occurred."
                ),
            })

        # Observation 4: an incoming message was recorded after
        # the user explicitly stated a boundary in an outgoing one.
        boundary_entries = [
            entry for entry, _ in case_entries
            if (
                entry.direction == "outgoing"
                and contains_boundary_statement(
                    entry.incident.message_text
                )
            )
        ]

        incoming_entries = [
            entry for entry, _ in case_entries
            if entry.direction == "incoming"
        ]

        for boundary in boundary_entries:
            later_incoming = [
                incoming
                for incoming in incoming_entries
                if (
                    incoming.incident.occurred_at
                    > boundary.incident.occurred_at
                )
            ]

            if later_incoming:
                observations.append({
                    "case_id": case_id,
                    "type": "boundary_followed_by_later_contact",
                    "incident_ids": [
                        boundary.incident.incident_id,
                        *[
                            item.incident.incident_id
                            for item in later_incoming
                        ],
                    ],
                    "explanation": (
                        "A later incoming entry follows an outgoing "
                        "message containing explicit boundary language. "
                        "Review the context; this observation alone "
                        "does not establish harassment."
                    ),
                })

    return {
        "events": events,
        "observations": observations,
        "case_count": len(linked_cases),
        "linked_case_ids": sorted(linked_cases.keys()),
        "privacy_note": (
            "Only explicitly linked incidents are compared. "
            "No real-world identity or future behaviour is verified."
        ),
    }


def print_timeline_summary(result: dict) -> None:
    """Display a readable summary without printing full messages."""

    print(f"Timeline events: {len(result['events'])}")
    print(f"Explicitly linked cases: {result['case_count']}")
    print(f"Contextual observations: {len(result['observations'])}")
    print()

    for index, event in enumerate(result["events"], start=1):
        print(
            f"{index}. {event['occurred_at']} | "
            f"{event['platform']} | "
            f"{event['direction']} | "
            f"Findings: {len(event['findings'])}"
        )

    print("\nObservations:")

    if not result["observations"]:
        print(
            "No configured sequence-level observations were found. "
            "This does not mean the interaction is safe."
        )
    else:
        for item in result["observations"]:
            print(f"- {item['type']}: {item['explanation']}")

    print("\nPrivacy:", result["privacy_note"])


print("Timeline analysis engine defined.")

In [ ]:
# SIGNALHER — CELL 8
# Synthetic linked-incident test

from datetime import datetime, timezone


def make_synthetic_incident(platform, hour, minute, message):
    return Incident(
        platform=platform,
        occurred_at=datetime(
            2026, 10, 24, hour, minute,
            tzinfo=timezone.utc,
        ),
        message_text=message,
        user_note="Synthetic test data only.",
    )


boundary_incident = make_synthetic_incident(
    "instagram",
    10, 0,
    "Please do not contact me again."
)

coercion_incident = make_synthetic_incident(
    "whatsapp",
    10, 30,
    "I don't care that you said no. Send me your private pictures."
)

image_threat_incident = make_synthetic_incident(
    "email",
    11, 0,
    "I'll publish your private photos."
)

unlinked_benign_incident = make_synthetic_incident(
    "whatsapp",
    9, 0,
    "Can you share your vacation photos?"
)

entries = [
    TimelineEntry(
        incident=boundary_incident,
        direction="outgoing",
        case_id="synthetic_case_01",
    ),
    TimelineEntry(
        incident=coercion_incident,
        direction="incoming",
        case_id="synthetic_case_01",
    ),
    TimelineEntry(
        incident=image_threat_incident,
        direction="incoming",
        case_id="synthetic_case_01",
    ),
    # No case ID: this event must remain unlinked.
    TimelineEntry(
        incident=unlinked_benign_incident,
        direction="incoming",
    ),
]

timeline_result = analyze_timeline(entries)

print_timeline_summary(timeline_result)

observation_types = {
    observation["type"]
    for observation in timeline_result["observations"]
}

assert "multiple_platforms" in observation_types
assert "multiple_warning_bearing_incidents" in observation_types
assert "multiple_signal_categories" in observation_types
assert "boundary_followed_by_later_contact" in observation_types

assert len(timeline_result["events"]) == 4
assert timeline_result["events"][0]["incident_id"] == (
    unlinked_benign_incident.incident_id
)

# Ensure we haven't assigned the unrelated event to the case.
assert timeline_result["events"][0]["case_id"] is None

print("\nAll synthetic timeline tests passed.")

In [ ]:
# SIGNALHER — CELL 9
# Privacy-conscious, user-controlled safety-action generator

from datetime import datetime, timezone


# Official support resources.
# These are reference links; the application does not submit
# reports or contact these services automatically.

RESOURCE_DIRECTORY = {
    "emergency_112": {
        "name": "Emergency Response Support System — 112",
        "url": "https://112.gov.in/",
        "description": (
            "India's unified emergency number for immediate "
            "emergencies requiring police, health, fire, or "
            "other emergency response."
        ),
    },
    "women_helpline_181": {
        "name": "Women Helpline — 181",
        "url": "https://wcd.gov.in/women/help",
        "description": (
            "Government women-focused assistance and support "
            "information. Availability and services can vary "
            "by location; confirm local availability."
        ),
    },
    "cybercrime_portal": {
        "name": "National Cyber Crime Reporting Portal",
        "url": "https://cybercrime.gov.in/",
        "description": (
            "Official Government of India portal for reporting "
            "cybercrime, including online and social-media offences."
        ),
    },
}


def generate_action_plan(
    timeline_result: dict,
    *,
    immediate_danger: bool = False,
) -> dict:
    """
    Generate optional safety actions from a previously analysed
    timeline.

    Important:
    - Immediate danger is explicitly indicated by the user.
    - Detected signals do not establish that abuse occurred.
    - This function does not contact anyone or submit reports.
    - No numerical danger score is generated.
    """

    if not isinstance(timeline_result, dict):
        raise TypeError("timeline_result must be a dictionary.")

    if not isinstance(immediate_danger, bool):
        raise TypeError("immediate_danger must be True or False.")

    if "events" not in timeline_result:
        raise ValueError("Timeline result is missing 'events'.")

    events = timeline_result["events"]

    findings = [
        (event, finding)
        for event in events
        for finding in event.get("findings", [])
    ]

    categories = {
        finding["category"]
        for _, finding in findings
    }

    actions = []
    resource_ids = set()
    added_action_ids = set()

    def add_action(
        action_id: str,
        title: str,
        rationale: str,
        steps: list[str],
        priority: str = "consider",
    ):
        if action_id in added_action_ids:
            return

        added_action_ids.add(action_id)

        actions.append({
            "action_id": action_id,
            "title": title,
            "priority": priority,
            "rationale": rationale,
            "steps": steps,
            "user_control": (
                "Optional guidance. The user decides whether "
                "and when to act, subject to immediate safety needs."
            ),
        })

    # --------------------------------------------------
    # 1. Immediate danger is determined by the user,
    #    never inferred from the analyser.
    # --------------------------------------------------

    if immediate_danger:
        add_action(
            action_id="IMMEDIATE_SAFETY",
            title="Prioritise immediate safety",
            rationale=(
                "The user has indicated that immediate danger "
                "may be present."
            ),
            steps=[
                "If possible, move to a safer accessible place.",
                (
                    "In India, call 112 or ask a trusted person "
                    "to contact emergency services."
                ),
                (
                    "Do not wait for SignalHer to finish analysing "
                    "messages or preparing a report."
                ),
                (
                    "Do not use this application as an emergency "
                    "alert system; it sends no alerts."
                ),
            ],
            priority="urgent",
        )

        resource_ids.add("emergency_112")

    # --------------------------------------------------
    # 2. Explain what the result means.
    # --------------------------------------------------

    if findings:
        add_action(
            action_id="REVIEW_EVIDENCE",
            title="Review the highlighted evidence",
            rationale=(
                "The analyser identified one or more configured "
                "warning indicators in the supplied incidents."
            ),
            steps=[
                (
                    "Review the exact highlighted wording and "
                    "the original conversation context."
                ),
                (
                    "Check whether the incident interpretation "
                    "matches what actually happened."
                ),
                (
                    "Remember that a flag can be wrong and an "
                    "unflagged message can still be concerning."
                ),
            ],
        )

        add_action(
            action_id="PRESERVE_EVIDENCE",
            title="Consider preserving relevant evidence",
            rationale=(
                "A private record may help if the user later "
                "chooses to seek assistance or report the incident."
            ),
            steps=[
                (
                    "If safe, retain relevant messages, dates, "
                    "account names, and original URLs."
                ),
                (
                    "Keep copies in a location the other person "
                    "cannot access, if doing so is safe."
                ),
                (
                    "Do not forward intimate images or spread "
                    "sensitive material as a way of collecting evidence."
                ),
                (
                    "If the device or account may be monitored, "
                    "consider using a safer device before taking action."
                ),
            ],
        )

    # --------------------------------------------------
    # 3. Tailor actions to the configured signal categories.
    # --------------------------------------------------

    if "coercion" in categories:
        add_action(
            action_id="COERCION_SUPPORT",
            title="Consider trusted support",
            rationale=(
                "The supplied messages contain indicators of "
                "possible pressure or coercion."
            ),
            steps=[
                (
                    "You do not have to comply with demands "
                    "that make you uncomfortable."
                ),
                (
                    "Consider talking to someone you trust "
                    "if doing so feels safe."
                ),
                (
                    "Avoid responding impulsively to threats; "
                    "choose your next step based on your circumstances."
                ),
            ],
        )

    if "image_based_abuse" in categories:
        add_action(
            action_id="IMAGE_ABUSE_SUPPORT",
            title="Review image-related safety options",
            rationale=(
                "The supplied text contains a configured "
                "indicator concerning intimate or private imagery."
            ),
            steps=[
                (
                    "If an image has already been shared, "
                    "consider the relevant platform's official "
                    "reporting process."
                ),
                (
                    "Keep relevant URLs and messages if safe; "
                    "do not circulate the image further."
                ),
                (
                    "If threats or blackmail are involved, "
                    "consider seeking trusted or official support."
                ),
            ],
        )

    if "identity_misuse" in categories:
        add_action(
            action_id="IDENTITY_PROTECTION",
            title="Review identity and account security",
            rationale=(
                "The supplied text contains a configured "
                "identity-impersonation indicator."
            ),
            steps=[
                (
                    "If relevant, record the impersonating "
                    "profile URL and account name."
                ),
                (
                    "Consider reporting the profile through "
                    "the platform's official impersonation process."
                ),
                (
                    "If safe, review account sessions and security "
                    "settings from a device the other person "
                    "cannot access."
                ),
            ],
        )

    if "physical_safety" in categories:
        add_action(
            action_id="PHYSICAL_SAFETY_REVIEW",
            title="Consider your immediate circumstances",
            rationale=(
                "The message contains configured threatening "
                "or location-related language."
            ),
            steps=[
                (
                    "Consider whether the message refers to "
                    "a current and credible threat in your situation."
                ),
                (
                    "If you believe someone may harm you, "
                    "contact a trusted person or appropriate "
                    "local support if safe."
                ),
                (
                    "If danger is immediate in India, call 112; "
                    "do not rely on this application's analysis."
                ),
            ],
        )

        resource_ids.add("emergency_112")

    # --------------------------------------------------
    # 4. Add support routes without submitting a report.
    # --------------------------------------------------

    if findings:
        resource_ids.add("cybercrime_portal")

    if categories.intersection({
        "coercion",
        "image_based_abuse",
        "physical_safety",
    }):
        resource_ids.add("women_helpline_181")

    if not findings:
        add_action(
            action_id="NO_MATCH_CAUTION",
            title="Review the result cautiously",
            rationale=(
                "No configured warning pattern matched the supplied "
                "incidents. This does not establish that they are safe."
            ),
            steps=[
                (
                    "Consider the wider context and your own concerns."
                ),
                (
                    "You can add another relevant incident or seek "
                    "trusted support even if no warning was generated."
                ),
            ],
        )

    resources = [
        RESOURCE_DIRECTORY[resource_id]
        | {"resource_id": resource_id}
        for resource_id in sorted(resource_ids)
    ]

    # Do not export raw messages or user notes.
    incident_references = [
        {
            "incident_id": event["incident_id"],
            "platform": event["platform"],
            "occurred_at": event["occurred_at"],
            "matched_rule_ids": sorted({
                finding["rule_id"]
                for finding in event.get("findings", [])
            }),
        }
        for event in events
    ]

    return {
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "summary": {
            "incident_count": len(events),
            "finding_count": len(findings),
            "signal_categories": sorted(categories),
            "immediate_danger_user_indicated": immediate_danger,
        },
        "incident_references": incident_references,
        "actions": actions,
        "resources": resources,
        "limitations": [
            (
                "SignalHer does not determine whether a crime or "
                "abuse occurred."
            ),
            (
                "SignalHer does not predict violence or calculate "
                "a validated danger score."
            ),
            (
                "SignalHer does not contact trusted people, "
                "emergency services, or reporting portals."
            ),
            (
                "Resource links are informational; service availability "
                "and suitability should be checked by the user."
            ),
        ],
        "privacy_note": (
            "This action plan excludes raw message text and user notes. "
            "It is generated locally by this function and does not "
            "submit information to the listed resources."
        ),
    }


print("Safety-action generator defined.")
print("Configured official resources:", len(RESOURCE_DIRECTORY))

In [ ]:
# SIGNALHER — CELL 10
# Test action generation and privacy properties

import json


# Test the standard action plan.
action_plan = generate_action_plan(
    timeline_result,
    immediate_danger=False,
)

print("Standard action plan")
print("--------------------")
print("Summary:", action_plan["summary"])
print("Actions:", len(action_plan["actions"]))
print("Resources:", len(action_plan["resources"]))

for action in action_plan["actions"]:
    print(f"\n- {action['title']} [{action['priority']}]")
    for step in action["steps"]:
        print("  •", step)


# Verify that the expected categories are supported.
action_ids = {
    action["action_id"]
    for action in action_plan["actions"]
}

assert "REVIEW_EVIDENCE" in action_ids
assert "PRESERVE_EVIDENCE" in action_ids
assert "COERCION_SUPPORT" in action_ids
assert "IMAGE_ABUSE_SUPPORT" in action_ids

resource_ids = {
    resource["resource_id"]
    for resource in action_plan["resources"]
}

assert "cybercrime_portal" in resource_ids
assert "women_helpline_181" in resource_ids

# The sample contains no physical-safety warning, and no
# immediate danger was explicitly indicated by the user.
assert "emergency_112" not in resource_ids
assert (
    action_plan["summary"]["immediate_danger_user_indicated"]
    is False
)


# Test explicit user-indicated immediate danger.
urgent_plan = generate_action_plan(
    timeline_result,
    immediate_danger=True,
)

urgent_action = next(
    action
    for action in urgent_plan["actions"]
    if action["action_id"] == "IMMEDIATE_SAFETY"
)

assert urgent_action["priority"] == "urgent"

urgent_resource_ids = {
    resource["resource_id"]
    for resource in urgent_plan["resources"]
}

assert "emergency_112" in urgent_resource_ids


# Verify that raw message text does not appear in the action plan.
serialized_plan = json.dumps(action_plan)

assert "message_text" not in serialized_plan
assert "user_note" not in serialized_plan
assert (
    "Send me your private pictures"
    not in serialized_plan
)

print("\nAll safety-action generator tests passed.")

In [ ]:
# ============================================================
# SIGNALHER — SAFETY RESOURCE DIRECTORY
# Purpose:
# 1. Keep useful support links available locally.
# 2. Match resources to the selected incident category.
# 3. Build safe, general search queries for Google Search.
# 4. Never send private incident messages to the search engine.
# ============================================================

from dataclasses import dataclass, asdict
from urllib.parse import urlparse


@dataclass(frozen=True)
class SafetyResource:
    resource_id: str
    name: str
    categories: tuple[str, ...]
    jurisdiction: str
    url: str
    purpose: str
    eligibility_note: str
    limitations: str
    source_type: str
    priority: int = 50
    excluded_states: tuple[str, ...] = ()


# Curated resources.
# These links open external services; SignalHer does not submit reports
# or contact any organization automatically.

RESOURCE_DIRECTORY = [
    SafetyResource(
        resource_id="emergency_112",
        name="Emergency Response Support System — 112",
        categories=("emergency", "physical_safety"),
        jurisdiction="India",
        url="https://112.gov.in/",
        purpose=(
            "Official emergency assistance for situations requiring "
            "urgent police, fire, medical, or other emergency support."
        ),
        eligibility_note="For emergencies in India.",
        limitations=(
            "Use emergency services directly when immediate help is needed. "
            "SignalHer cannot dispatch assistance."
        ),
        source_type="Government of India",
        priority=1,
    ),

    SafetyResource(
        resource_id="cybercrime_portal",
        name="National Cyber Crime Reporting Portal",
        categories=(
            "cybercrime",
            "image_based_abuse",
            "identity_misuse",
            "coercion",
        ),
        jurisdiction="India",
        url="https://cybercrime.gov.in/",
        purpose=(
            "Official Indian portal for reporting cybercrime."
        ),
        eligibility_note="For relevant cybercrime reports in India.",
        limitations=(
            "The user must review the information and decide whether "
            "to report it. SignalHer does not submit reports automatically."
        ),
        source_type="Government of India",
        priority=10,
    ),

    SafetyResource(
        resource_id="women_helpline_181",
        name="Women Helpline — 181",
        categories=(
            "women_support",
            "coercion",
            "image_based_abuse",
            "physical_safety",
        ),
        jurisdiction="India",
        url="https://wcd.gov.in/women/help",
        purpose=(
            "Government women’s helpline providing support and information "
            "for women experiencing distress or violence."
        ),
        eligibility_note=(
            "Check local availability and the service's current guidance."
        ),
        limitations=(
            "The Ministry of Women and Child Development currently states "
            "that the service is operational in all States/UTs except "
            "West Bengal. Availability may change."
        ),
        source_type="Government of India",
        priority=20,
        excluded_states=("west bengal",),
    ),

    SafetyResource(
        resource_id="google_ncii_removal",
        name="Google — Request Removal of Explicit or Intimate Content",
        categories=("image_based_abuse", "identity_misuse"),
        jurisdiction="Global",
        url="https://support.google.com/accounts/answer/17045517?hl=en",
        purpose=(
            "Official guidance for requesting removal of non-consensual "
            "intimate content from Google services."
        ),
        eligibility_note=(
            "Review Google's instructions and eligibility requirements "
            "before submitting a request."
        ),
        limitations=(
            "Removal from Google services does not itself remove material "
            "from websites hosted elsewhere."
        ),
        source_type="Google official support",
        priority=30,
    ),

    SafetyResource(
        resource_id="stopncii",
        name="StopNCII.org",
        categories=("image_based_abuse",),
        jurisdiction="Global",
        url="https://stopncii.org/",
        purpose=(
            "A tool that creates a digital fingerprint of eligible intimate "
            "images or videos on the user's device."
        ),
        eligibility_note=(
            "The person depicted must meet StopNCII's age and other "
            "eligibility requirements. Check the official guidance."
        ),
        limitations=(
            "It works with participating platforms, not the entire internet. "
            "The selected image or video is not uploaded to create the hash."
        ),
        source_type="Official nonprofit-operated service",
        priority=31,
    ),

    SafetyResource(
        resource_id="take_it_down",
        name="Take It Down",
        categories=("image_based_abuse",),
        jurisdiction="Global",
        url="https://takeitdown.ncmec.org/",
        purpose=(
            "A resource for people seeking help with eligible nude or "
            "sexually explicit images or videos taken when they were "
            "under 18."
        ),
        eligibility_note=(
            "Read the service's official eligibility and safety instructions."
        ),
        limitations=(
            "Coverage depends on participating services; this does not "
            "guarantee removal from every website."
        ),
        source_type="NCMEC official service",
        priority=32,
    ),
]


def _normalise_text(value: str) -> str:
    """Normalise a category or state for consistent matching."""
    return " ".join(
        str(value).strip().casefold().replace("_", " ").replace("-", " ").split()
    )


def find_resources(
    category: str,
    state: str | None = None,
    country: str = "India",
    limit: int = 10,
) -> list[dict]:
    """
    Return curated resources matching a selected category and location.

    Important:
    - Pass a category label, not a private incident message.
    - Only matching jurisdictions are returned.
    - A state-specific exclusion prevents recommending an unavailable service.
    """
    category_key = _normalise_text(category)
    state_key = _normalise_text(state or "")
    country_key = _normalise_text(country)

    if limit < 1:
        return []

    results = []

    for resource in RESOURCE_DIRECTORY:
        resource_country = _normalise_text(resource.jurisdiction)

        # Include resources for the selected country and globally available tools.
        if resource_country not in {country_key, "global"}:
            continue

        # Apply explicit state exclusions.
        excluded_states = {
            _normalise_text(item) for item in resource.excluded_states
        }
        if state_key and state_key in excluded_states:
            continue

        # Match only the selected category, unless "all" is requested.
        resource_categories = {
            _normalise_text(item) for item in resource.categories
        }
        if category_key not in {"all", "general support"}:
            if category_key not in resource_categories:
                continue

        results.append(resource)

    results.sort(key=lambda item: item.priority)

    return [asdict(item) for item in results[:limit]]


# Only approved, general categories are used to construct search queries.
# There is deliberately NO argument for message text, names, screenshots,
# usernames, or incident history.

SAFE_SEARCH_TERMS = {
    "emergency": "official emergency assistance",
    "physical_safety": "official physical safety support",
    "cybercrime": "official cybercrime reporting and online harassment support",
    "image_based_abuse": "official non-consensual intimate image support and removal",
    "identity_misuse": "official online impersonation reporting support",
    "coercion": "official support for online threats and coercion",
    "women_support": "official women safety and support services",
    "general_support": "official online safety support resources",
}


INDIAN_STATES_AND_UTS = {
    "Andhra Pradesh", "Arunachal Pradesh", "Assam", "Bihar",
    "Chhattisgarh", "Goa", "Gujarat", "Haryana", "Himachal Pradesh",
    "Jharkhand", "Karnataka", "Kerala", "Madhya Pradesh", "Maharashtra",
    "Manipur", "Meghalaya", "Mizoram", "Nagaland", "Odisha", "Punjab",
    "Rajasthan", "Sikkim", "Tamil Nadu", "Telangana", "Tripura",
    "Uttar Pradesh", "Uttarakhand", "West Bengal",
    "Andaman and Nicobar Islands", "Chandigarh",
    "Dadra and Nagar Haveli and Daman and Diu", "Delhi",
    "Jammu and Kashmir", "Ladakh", "Lakshadweep", "Puducherry",
}


def build_safe_search_query(
    category: str,
    state: str | None = None,
    country: str = "India",
) -> str:
    """
    Build a general-purpose query for the later Google-powered interface.

    The query is made from a category and an optional recognised Indian
    State/UT. Never pass the original incident message into this function.
    """
    category_key = _normalise_text(category)
    base_query = SAFE_SEARCH_TERMS.get(
        category_key,
        SAFE_SEARCH_TERMS["general_support"],
    )

    search_parts = [base_query]

    if _normalise_text(country) == "india":
        search_parts.append("India")

        if state:
            matched_state = next(
                (
                    item
                    for item in INDIAN_STATES_AND_UTS
                    if _normalise_text(item) == _normalise_text(state)
                ),
                None,
            )
            if matched_state:
                search_parts.append(matched_state)

    return " ".join(search_parts)


# ============================================================
# TESTS
# ============================================================

# Check that every stored URL is an HTTPS URL with a hostname.
assert all(
    urlparse(resource.url).scheme == "https"
    and bool(urlparse(resource.url).netloc)
    for resource in RESOURCE_DIRECTORY
), "A resource URL is invalid."

# Check that the image-abuse category returns relevant resources.
image_resources = find_resources(
    category="image_based_abuse",
    state="Tripura",
    country="India",
)

image_resource_ids = {
    resource["resource_id"] for resource in image_resources
}

assert "google_ncii_removal" in image_resource_ids
assert "stopncii" in image_resource_ids
assert "take_it_down" in image_resource_ids
assert "cybercrime_portal" in image_resource_ids

# Check the state-specific service exclusion.
west_bengal_resources = find_resources(
    category="women_support",
    state="West Bengal",
    country="India",
)

assert "women_helpline_181" not in {
    resource["resource_id"] for resource in west_bengal_resources
}

# Check that a safe general search query can be created.
test_query = build_safe_search_query(
    category="image_based_abuse",
    state="Tripura",
    country="India",
)

assert "Tripura" in test_query
assert "India" in test_query

print("Resource directory loaded:", len(RESOURCE_DIRECTORY), "resources")
print("HTTPS URL checks: PASSED")
print("Image-abuse resource matching: PASSED")
print("State-specific availability check: PASSED")
print("Safe Google query:", test_query)
print("\nMatching resources:")

for resource in image_resources:
    print(f"\n- {resource['name']}")
    print(f"  URL: {resource['url']}")
    print(f"  Purpose: {resource['purpose']}")

In [ ]:
# ============================================================
# SIGNALHER — PAN-INDIA RESOURCE MATCHING + GOOGLE QUERY FIX
# Add this as a new cell below your existing code.
# ============================================================

def find_resources(
    category: str,
    state: str | None = None,
    country: str = "India",
    limit: int = 10,
) -> list[dict]:
    """
    Find curated support resources.

    Nationwide default:
    - Includes India-wide and global resources.
    - Excludes resources known to be unavailable in some states
      when no state has been selected.
    - Does not assume the user is from Tripura or any other state.
    """
    category_key = _normalise_text(category)
    state_key = _normalise_text(state or "")
    country_key = _normalise_text(country)

    if limit < 1:
        return []

    results = []

    for resource in RESOURCE_DIRECTORY:
        resource_country = _normalise_text(resource.jurisdiction)

        # Keep resources relevant to India or available globally.
        if resource_country not in {country_key, "global"}:
            continue

        excluded_states = {
            _normalise_text(item)
            for item in resource.excluded_states
        }

        # If the user's state is unknown, do not present a resource
        # with known state-level exclusions as universally available.
        if not state_key and excluded_states:
            continue

        # If a state is explicitly supplied, respect its exclusions.
        if state_key and state_key in excluded_states:
            continue

        resource_categories = {
            _normalise_text(item)
            for item in resource.categories
        }

        if category_key not in {"all", "general support"}:
            if category_key not in resource_categories:
                continue

        results.append(resource)

    results.sort(key=lambda item: item.priority)

    return [asdict(item) for item in results[:limit]]


# Keys use the same normalised format as _normalise_text().
SAFE_SEARCH_TERMS = {
    "emergency": "official emergency assistance",
    "physical safety": "official physical safety support",
    "cybercrime": "official cybercrime reporting and online harassment support",
    "image based abuse": "official non-consensual intimate image support and removal",
    "identity misuse": "official online impersonation reporting support",
    "coercion": "official support for online threats and coercion",
    "women support": "official women safety and support services",
    "general support": "official online safety support resources",
}


def build_safe_search_query(category: str) -> str:
    """
    Build a general, India-wide search query for the future Google
    Search interface.

    Only a category is accepted. The function deliberately does not
    accept incident text, names, usernames, screenshots, or state names.
    """
    category_key = _normalise_text(category)

    base_query = SAFE_SEARCH_TERMS.get(
        category_key,
        SAFE_SEARCH_TERMS["general support"],
    )

    return f"{base_query} India"


# ============================================================
# TESTS
# ============================================================

# 1. Category-specific query should no longer fall back to generic text.
test_query = build_safe_search_query("image_based_abuse")

assert test_query == (
    "official non-consensual intimate image support and removal India"
)

# 2. The search query must not be tied to a particular state.
assert "Tripura" not in test_query
assert "West Bengal" not in test_query

# 3. Nationwide results should not label the 181 service as universal.
nationwide_image_resources = find_resources(
    category="image_based_abuse"
)

nationwide_ids = {
    resource["resource_id"]
    for resource in nationwide_image_resources
}

assert "cybercrime_portal" in nationwide_ids
assert "google_ncii_removal" in nationwide_ids
assert "stopncii" in nationwide_ids
assert "take_it_down" in nationwide_ids
assert "women_helpline_181" not in nationwide_ids

# 4. A supplied West Bengal location must also respect the exclusion.
west_bengal_resources = find_resources(
    category="women_support",
    state="West Bengal",
)

assert "women_helpline_181" not in {
    resource["resource_id"]
    for resource in west_bengal_resources
}

print("Pan-India query generation: PASSED")
print("Category-specific search query: PASSED")
print("No state-specific query bias: PASSED")
print("National resource availability checks: PASSED")

print("\nGoogle query for the future frontend:")
print(test_query)

print("\nNationwide resources for image-based abuse:")
for resource in nationwide_image_resources:
    print(f"\n- {resource['name']}")
    print(f"  URL: {resource['url']}")
    print(f"  Purpose: {resource['purpose']}")

In [ ]:
# ============================================================
# SIGNALHER — FLASK BACKEND API
# Endpoints:
#   GET  /api/health
#   GET  /api/categories
#   GET  /api/resources
#   POST /api/search-query
#
# This cell creates an API for the future frontend.
# It does not start a public server or contact Google.
# ============================================================

from flask import Flask, jsonify, request

signalher_app = Flask("signalher_api")


# ------------------------------------------------------------
# 1. HEALTH CHECK
# ------------------------------------------------------------

@signalher_app.get("/api/health")
def api_health():
    return jsonify({
        "success": True,
        "application": "SignalHer",
        "status": "running",
        "api_version": "1.0.0",
    }), 200


# ------------------------------------------------------------
# 2. AVAILABLE SUPPORT CATEGORIES
# ------------------------------------------------------------

CATEGORY_LABELS = {
    "emergency": "Emergency support",
    "physical_safety": "Physical safety",
    "cybercrime": "Cybercrime reporting",
    "image_based_abuse": "Non-consensual intimate imagery",
    "identity_misuse": "Online impersonation",
    "coercion": "Threats and coercion",
    "women_support": "Women's support services",
    "general_support": "General online safety support",
}


@signalher_app.get("/api/categories")
def api_categories():
    categories = [
        {
            "id": category_id,
            "label": label,
        }
        for category_id, label in CATEGORY_LABELS.items()
    ]

    return jsonify({
        "success": True,
        "categories": categories,
    }), 200


# ------------------------------------------------------------
# 3. RESOURCE DIRECTORY ENDPOINT
# ------------------------------------------------------------

@signalher_app.get("/api/resources")
def api_resources():
    # General support is the default when no category is selected.
    category = request.args.get("category", "general_support")

    # State is optional. SignalHer does not assume the user's location.
    state = request.args.get("state") or None

    # Only expose categories supported by this application.
    normalised_category = _normalise_text(category)
    allowed_categories = {
        _normalise_text(category_id)
        for category_id in CATEGORY_LABELS
    }

    if normalised_category not in allowed_categories:
        return jsonify({
            "success": False,
            "error": "Unsupported category.",
            "allowed_categories": list(CATEGORY_LABELS.keys()),
        }), 400

    resources = find_resources(
        category=normalised_category,
        state=state,
        country="India",
        limit=10,
    )

    return jsonify({
        "success": True,
        "scope": "India",
        "category": normalised_category,
        "state_filter_applied": bool(state),
        "count": len(resources),
        "resources": resources,
        "notice": (
            "Check the provider's current eligibility and availability. "
            "SignalHer does not submit reports or contact services."
        ),
    }), 200


# ------------------------------------------------------------
# 4. SAFE SEARCH-QUERY ENDPOINT
# ------------------------------------------------------------

@signalher_app.post("/api/search-query")
def api_search_query():
    payload = request.get_json(silent=True)

    if not isinstance(payload, dict):
        return jsonify({
            "success": False,
            "error": "Send a JSON object containing a category.",
        }), 400

    category = payload.get("category")

    if not isinstance(category, str) or not category.strip():
        return jsonify({
            "success": False,
            "error": "A valid category is required.",
            "allowed_categories": list(CATEGORY_LABELS.keys()),
        }), 400

    normalised_category = _normalise_text(category)

    # Only category identifiers are allowed. Unknown values are rejected
    # instead of silently creating unrelated searches.
    allowed_categories = {
        _normalise_text(category_id)
        for category_id in CATEGORY_LABELS
    }

    if normalised_category not in allowed_categories:
        return jsonify({
            "success": False,
            "error": "Unsupported category.",
            "allowed_categories": list(CATEGORY_LABELS.keys()),
        }), 400

    # Only the selected category enters the generated search query.
    # Other submitted fields, including any message text, are ignored.
    safe_query = build_safe_search_query(normalised_category)

    return jsonify({
        "success": True,
        "category": normalised_category,
        "query": safe_query,
        "scope": "India",
        "privacy_note": (
            "The generated query contains only a general support category "
            "and India. This endpoint does not send the query to Google."
        ),
    }), 200


# ============================================================
# 5. TEST THE API WITHOUT STARTING A SERVER
# ============================================================

api_test_client = signalher_app.test_client()


# Test A: Health endpoint
health_response = api_test_client.get("/api/health")
assert health_response.status_code == 200
assert health_response.get_json()["status"] == "running"


# Test B: Categories endpoint
categories_response = api_test_client.get("/api/categories")
assert categories_response.status_code == 200

returned_category_ids = {
    item["id"]
    for item in categories_response.get_json()["categories"]
}
assert "image_based_abuse" in returned_category_ids
assert "cybercrime" in returned_category_ids


# Test C: India-wide resource lookup
resources_response = api_test_client.get(
    "/api/resources?category=image_based_abuse"
)
assert resources_response.status_code == 200

resource_data = resources_response.get_json()
resource_ids = {
    resource["resource_id"]
    for resource in resource_data["resources"]
}

assert "cybercrime_portal" in resource_ids
assert "google_ncii_removal" in resource_ids
assert "stopncii" in resource_ids
assert "take_it_down" in resource_ids

# State-specific service excluded when no state is selected.
assert "women_helpline_181" not in resource_ids


# Test D: Explicit state exclusion
west_bengal_response = api_test_client.get(
    "/api/resources?category=women_support&state=West%20Bengal"
)
assert west_bengal_response.status_code == 200

west_bengal_ids = {
    resource["resource_id"]
    for resource in west_bengal_response.get_json()["resources"]
}
assert "women_helpline_181" not in west_bengal_ids


# Test E: Safe, category-only search query
query_response = api_test_client.post(
    "/api/search-query",
    json={
        "category": "image_based_abuse",
        # This extra field is deliberately ignored by the query builder.
        "message": "PRIVATE TEST MESSAGE MUST NOT APPEAR IN QUERY",
    },
)
assert query_response.status_code == 200

query_data = query_response.get_json()
assert query_data["query"] == (
    "official non-consensual intimate image support and removal India"
)
assert "PRIVATE TEST MESSAGE" not in query_data["query"]


# Test F: Reject unsupported categories
invalid_response = api_test_client.post(
    "/api/search-query",
    json={"category": "unknown_category"},
)
assert invalid_response.status_code == 400


print("Flask API initialisation: PASSED")
print("Health endpoint: PASSED")
print("Category endpoint: PASSED")
print("Pan-India resource endpoint: PASSED")
print("West Bengal availability safeguard: PASSED")
print("Category-only search query: PASSED")
print("Unsupported category validation: PASSED")

print("\nAPI routes:")
for rule in signalher_app.url_map.iter_rules():
    if rule.endpoint != "static":
        print(f"{', '.join(sorted(rule.methods - {'HEAD', 'OPTIONS'}))} {rule.rule}")

print("\nExample search API response:")
print(query_data)

In [ ]:
import traceback

diagnostic_payload = {
    "platform": "WhatsApp",
    "timestamp": "2026-10-25T10:00:00+05:30",
    "occurred_at": "2026-10-25T10:00:00+05:30",
    "message_text": (
        "If you don't send money, I will expose your private images."
    ),
    "user_note": "Synthetic diagnostic test.",
    "source_reference": "",
    "incident_id": "signalher-diagnostic-test"
}

try:
    print("STEP 1: Building the incident...")
    diagnostic_incident = _build_incident_from_payload(
        diagnostic_payload
    )
    print("Incident created:", type(diagnostic_incident).__name__)

    print("\nSTEP 2: Running the existing analyzer...")
    diagnostic_result = analyze_incident(diagnostic_incident)
    print("Analyzer completed successfully.")

    print("\nSTEP 3: Checking JSON serialization...")
    diagnostic_json = _make_json_safe(diagnostic_result)
    print("JSON conversion completed successfully.")

    print("\nDIAGNOSTIC PASSED")
    print("Finding count:", len(
        diagnostic_json.get("findings", [])
        if isinstance(diagnostic_json, dict)
        else []
    ))

except Exception:
    print("\nDIAGNOSTIC FAILED — FULL EXCEPTION:")
    traceback.print_exc()

In [ ]:
required_names = [
    "Incident",
    "analyze_incident",
    "_build_incident_from_payload",
    "_make_json_safe",
    "signalher_app",
    "api_test_client",
    "_signalher_http_server",
    "_signalher_server_thread",
]

print("SIGNALHER CURRENT KERNEL CHECK")
print("=" * 45)

for name in required_names:
    exists = name in globals()
    value = globals().get(name) if exists else None

    if exists:
        print(f"[FOUND]   {name}: {type(value).__name__}")
    else:
        print(f"[MISSING] {name}")

In [ ]:
from pathlib import Path
import json
import re

notebook_path = (
    Path.cwd()
    / "01_backend_development.ipynb"
    / "01_backend_development.ipynb"
)

if not notebook_path.exists():
    notebook_path = Path(
        r"C:\Users\Ropka\OneDrive\Apps\Desktop"
        r"\RaYVention_SignalHer\01_backend_development.ipynb"
        r"\01_backend_development.ipynb"
    )

print("Notebook found:", notebook_path.exists())

if notebook_path.exists():
    notebook_data = json.loads(
        notebook_path.read_text(encoding="utf-8")
    )

    patterns = {
        "Incident payload helper": r"def\s+_build_incident_from_payload\s*\(",
        "JSON serialization helper": r"def\s+_make_json_safe\s*\(",
        "Flask server startup": r"make_server\s*\(",
    }

    for label, pattern in patterns.items():
        print(f"\n--- {label} ---")
        found = False

        for cell_number, cell in enumerate(
            notebook_data.get("cells", []), start=1
        ):
            source = "".join(cell.get("source", []))

            if re.search(pattern, source):
                print(
                    f"Cell {cell_number} "
                    f"(type: {cell.get('cell_type', 'unknown')})"
                )
                found = True

        if not found:
            print("No matching code cell found.")
else:
    print("Please check the notebook path.")

In [ ]:
from pathlib import Path
import json
import re

# Locate the existing project from the notebook path.
notebook_path = Path(
    r"C:\Users\Ropka\OneDrive\Apps\Desktop"
    r"\RaYVention_SignalHer\01_backend_development.ipynb"
    r"\01_backend_development.ipynb"
)

project_root = notebook_path.parents[1]

targets = [
    "_build_incident_from_payload",
    "_make_json_safe",
]

definition_pattern = re.compile(
    r"^\s*def\s+(" + "|".join(map(re.escape, targets)) + r")\s*\(",
    re.MULTILINE,
)

print("PROJECT ROOT:", project_root)
print("PROJECT EXISTS:", project_root.exists())

if not project_root.exists():
    print("Project path not found. Please check the path above.")
else:
    found_definitions = False

    for file_path in project_root.rglob("*"):
        if not file_path.is_file():
            continue

        if file_path.suffix not in {".py", ".ipynb"}:
            continue

        if any(
            part in {
                ".git", ".venv", "venv", "node_modules",
                ".ipynb_checkpoints", "__pycache__"
            }
            for part in file_path.parts
        ):
            continue

        try:
            if file_path.suffix == ".ipynb":
                notebook = json.loads(
                    file_path.read_text(encoding="utf-8")
                )

                for cell_number, cell in enumerate(
                    notebook.get("cells", []), start=1
                ):
                    source = "".join(cell.get("source", []))
                    matches = list(definition_pattern.finditer(source))

                    for match in matches:
                        print(
                            f"\nFOUND DEFINITION: {match.group(1)}"
                            f"\nFile: {file_path}"
                            f"\nCell: {cell_number}"
                        )
                        found_definitions = True

            else:
                source = file_path.read_text(
                    encoding="utf-8",
                    errors="replace"
                )

                for match in definition_pattern.finditer(source):
                    line_number = source.count("\n", 0, match.start()) + 1
                    print(
                        f"\nFOUND DEFINITION: {match.group(1)}"
                        f"\nFile: {file_path}"
                        f"\nLine: {line_number}"
                    )
                    found_definitions = True

        except Exception as exc:
            print(f"Could not inspect {file_path.name}: {exc}")

    if not found_definitions:
        print(
            "\nNeither helper definition was found in the project's "
            "Python files or notebook code cells."
        )

In [ ]:
import inspect

def show_source(label, obj):
    print(f"\n{'=' * 65}")
    print(label)
    print("=" * 65)

    try:
        print("Signature:", inspect.signature(obj))
    except Exception as exc:
        print("Signature unavailable:", exc)

    try:
        print(inspect.getsource(obj))
    except Exception as exc:
        print("Source unavailable:", exc)

# 1. Check the existing Incident model and analyzer.
show_source("EXISTING INCIDENT MODEL", Incident)
show_source("EXISTING INCIDENT ANALYZER", analyze_incident)

# 2. Inspect the actual registered API endpoints.
print("\n" + "=" * 65)
print("REGISTERED SIGNALHER API ROUTES")
print("=" * 65)

for rule in sorted(signalher_app.url_map.iter_rules(), key=lambda r: r.rule):
    if rule.rule.startswith("/api/"):
        endpoint_function = signalher_app.view_functions.get(rule.endpoint)
        print(f"\n{rule.methods} {rule.rule} -> {rule.endpoint}")

        if rule.rule in {
            "/api/analyze-incident",
            "/api/analyze-timeline",
            "/api/search-query",
        }:
            if endpoint_function:
                show_source(f"ROUTE SOURCE: {rule.rule}", endpoint_function)

# 3. Check whether the missing helpers are referenced by registered routes.
print("\n" + "=" * 65)
print("MISSING HELPER REFERENCES")
print("=" * 65)

for route_name in ["/api/analyze-incident", "/api/analyze-timeline"]:
    matching_rules = [
        rule for rule in signalher_app.url_map.iter_rules()
        if rule.rule == route_name
    ]

    for rule in matching_rules:
        route_function = signalher_app.view_functions.get(rule.endpoint)
        if route_function:
            code = getattr(route_function, "__code__", None)
            names = code.co_names if code else ()
            print(f"{route_name}:")
            print("  _build_incident_from_payload referenced:",
                  "_build_incident_from_payload" in names)
            print("  _make_json_safe referenced:",
                  "_make_json_safe" in names)

In [ ]:
# SignalHer — restore missing incident API helpers
# Uses the existing Incident model, Platform enum, and analyzer.

from datetime import datetime as _SignalHerDateTime
from enum import Enum as _SignalHerEnum
from dataclasses import is_dataclass as _signalher_is_dataclass
from dataclasses import fields as _signalher_fields


def _build_incident_from_payload(payload):
    """Validate an API payload and create the existing Incident model."""

    if not isinstance(payload, dict):
        raise ValueError("Send the incident as a JSON object.")

    # Resolve the platform against the existing enum.
    raw_platform = payload.get("platform")

    if isinstance(raw_platform, Platform):
        platform = raw_platform
    else:
        platform_text = str(raw_platform or "").strip()
        platform = None

        for candidate in Platform:
            if (
                str(candidate.value).casefold() == platform_text.casefold()
                or candidate.name.casefold() == platform_text.casefold()
            ):
                platform = candidate
                break

        if platform is None:
            allowed = ", ".join(
                f"{item.name} ({item.value})" for item in Platform
            )
            raise ValueError(
                "Unsupported platform. Allowed values: "
                + (allowed or "none configured")
            )

    # Accept either occurred_at or timestamp from the frontend.
    raw_time = payload.get("occurred_at") or payload.get("timestamp")

    if isinstance(raw_time, _SignalHerDateTime):
        occurred_at = raw_time
    elif raw_time is None or not str(raw_time).strip():
        occurred_at = _SignalHerDateTime.now().astimezone()
    else:
        time_text = str(raw_time).strip()

        if time_text.endswith("Z"):
            time_text = time_text[:-1] + "+00:00"

        try:
            occurred_at = _SignalHerDateTime.fromisoformat(time_text)
        except ValueError as exc:
            raise ValueError(
                "The incident timestamp must use ISO 8601 format."
            ) from exc

    # Accept the frontend's message_text field, with a message fallback.
    message_text = payload.get("message_text")

    if message_text is None:
        message_text = payload.get("message")

    if not isinstance(message_text, str) or not message_text.strip():
        raise ValueError("Please enter the message you want to analyse.")

    user_note = payload.get("user_note") or ""
    source_reference = payload.get("source_reference")
    incident_id = payload.get("incident_id")

    incident_arguments = {
        "platform": platform,
        "occurred_at": occurred_at,
        "message_text": message_text,
        "user_note": str(user_note),
        "source_reference": (
            str(source_reference).strip()
            if source_reference is not None
            and str(source_reference).strip()
            else None
        ),
    }

    if incident_id is not None and str(incident_id).strip():
        incident_arguments["incident_id"] = str(incident_id).strip()

    return Incident(**incident_arguments)


def _make_json_safe(value):
    """Convert common Python values into JSON-serializable values."""

    if value is None or isinstance(value, (str, int, float, bool)):
        return value

    if isinstance(value, _SignalHerEnum):
        return _make_json_safe(value.value)

    if isinstance(value, _SignalHerDateTime):
        return value.isoformat()

    if isinstance(value, dict):
        return {
            str(key): _make_json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple, set)):
        return [_make_json_safe(item) for item in value]

    if _signalher_is_dataclass(value) and not isinstance(value, type):
        return {
            field.name: _make_json_safe(getattr(value, field.name))
            for field in _signalher_fields(value)
        }

    # Support NumPy-like scalar values, if they appear in results.
    if hasattr(value, "item") and callable(value.item):
        try:
            return _make_json_safe(value.item())
        except (ValueError, TypeError):
            pass

    raise TypeError(
        f"Cannot convert {type(value).__name__} to a JSON-safe value."
    )


print("SignalHer helper functions defined.")
print("Incident payload helper:", callable(_build_incident_from_payload))
print("JSON serialization helper:", callable(_make_json_safe))


# Test the existing API with a synthetic message.
if not hasattr(Platform, "__iter__"):
    raise RuntimeError("Could not inspect the existing Platform enum.")

_test_platform = next(iter(Platform))

_test_payload = {
    "platform": _test_platform.value,
    "occurred_at": "2026-10-25T10:00:00+05:30",
    "timestamp": "2026-10-25T10:00:00+05:30",
    "message_text": (
        "If you don't send money, I will expose your private images."
    ),
    "user_note": "Synthetic backend diagnostic.",
    "source_reference": "",
    "incident_id": "signalher-diagnostic-test",
}

try:
    _test_response = api_test_client.post(
        "/api/analyze-incident",
        json=_test_payload,
    )

    print("\nIncident endpoint HTTP status:", _test_response.status_code)
    print("Incident endpoint response:")
    print(_test_response.get_json())

    if _test_response.status_code == 200:
        print("\nPASS: Incident analysis endpoint responded successfully.")
    else:
        print("\nThe endpoint still needs investigation. See the response above.")

except Exception:
    import traceback
    traceback.print_exc()

In [ ]:
# SignalHer — start the existing Flask API and test the live connection

from threading import Thread as _SignalHerThread
from werkzeug.serving import make_server as _signalher_make_server
import urllib.request as _signalher_urlrequest
import urllib.error as _signalher_urlerror

try:
    # Reuse the server if it is already running in this kernel.
    if (
        "_signalher_server_thread" in globals()
        and _signalher_server_thread.is_alive()
    ):
        print("SignalHer server thread is already running.")

    else:
        _signalher_http_server = _signalher_make_server(
            host="127.0.0.1",
            port=5000,
            app=signalher_app,
            threaded=True,
        )

        _signalher_server_thread = _SignalHerThread(
            target=_signalher_http_server.serve_forever,
            daemon=True,
            name="SignalHerLocalServer",
        )
        _signalher_server_thread.start()

        print("SignalHer server started at http://127.0.0.1:5000")

    # Test the actual HTTP server, not just Flask's in-memory test client.
    with _signalher_urlrequest.urlopen(
        "http://127.0.0.1:5000/api/health",
        timeout=5,
    ) as response:
        print("Live health endpoint HTTP status:", response.status)
        print("Live health endpoint response:")
        print(response.read().decode("utf-8"))

        if response.status == 200:
            print("\nPASS: The live backend is reachable.")

except Exception as exc:
    print("\nBACKEND STARTUP OR CONNECTION CHECK FAILED")
    print("Error type:", type(exc).__name__)
    print("Error details:", str(exc))
    print(
        "\nSend me this error output before trying another startup command."
    )

In [ ]:
# ============================================================
# SIGNALHER — CLEAN FLASK APP REGISTRATION
# Fixes: Flask routes cannot be added after the first request.
#
# IMPORTANT:
# Register every route first.
# Run test requests only after registration is complete.
# ============================================================

from flask import Flask, jsonify, request

# Create a fresh app instance instead of modifying the locked one.
signalher_app = Flask("signalher_api_v2")


# ------------------------------------------------------------
# 1. Resolve category names consistently
# ------------------------------------------------------------

def resolve_category(category):
    """Return the canonical category ID, or None if unsupported."""
    if not isinstance(category, str):
        return None

    normalised = _normalise_text(category)

    for category_id in CATEGORY_LABELS:
        if _normalise_text(category_id) == normalised:
            return category_id

    return None


# ------------------------------------------------------------
# 2. Register all API routes BEFORE making any requests
# ------------------------------------------------------------

@signalher_app.get("/api/health")
def api_health():
    return jsonify({
        "success": True,
        "application": "SignalHer",
        "status": "running",
        "api_version": "1.0.0",
    }), 200


@signalher_app.get("/api/categories")
def api_categories():
    return jsonify({
        "success": True,
        "categories": [
            {"id": category_id, "label": label}
            for category_id, label in CATEGORY_LABELS.items()
        ],
    }), 200


@signalher_app.get("/api/resources")
def api_resources():
    requested_category = request.args.get(
        "category", "general_support"
    )
    category_id = resolve_category(requested_category)

    if category_id is None:
        return jsonify({
            "success": False,
            "error": "Unsupported category.",
            "allowed_categories": list(CATEGORY_LABELS.keys()),
        }), 400

    state = request.args.get("state") or None

    resources = find_resources(
        category=category_id,
        state=state,
        country="India",
        limit=10,
    )

    return jsonify({
        "success": True,
        "scope": "India",
        "category": category_id,
        "state_filter_applied": bool(state),
        "count": len(resources),
        "resources": resources,
        "notice": (
            "Check the provider's current eligibility and availability. "
            "SignalHer does not submit reports or contact services."
        ),
    }), 200


@signalher_app.post("/api/search-query")
def api_search_query():
    payload = request.get_json(silent=True)

    if not isinstance(payload, dict):
        return jsonify({
            "success": False,
            "error": "Send a JSON object containing a category.",
        }), 400

    category_id = resolve_category(payload.get("category"))

    if category_id is None:
        return jsonify({
            "success": False,
            "error": "Unsupported category.",
            "allowed_categories": list(CATEGORY_LABELS.keys()),
        }), 400

    # Only the selected category is used. The message and other fields
    # are deliberately not passed into the query builder.
    safe_query = build_safe_search_query(category_id)

    return jsonify({
        "success": True,
        "category": category_id,
        "query": safe_query,
        "scope": "India",
        "privacy_note": (
            "This endpoint generates a general query only. "
            "It does not send the query or incident to Google."
        ),
    }), 200


@signalher_app.post("/api/analyze-incident")
def api_analyze_incident():
    payload = request.get_json(silent=True)

    if not isinstance(payload, dict):
        return jsonify({
            "success": False,
            "error": "Send the incident as a JSON object.",
        }), 400

    try:
        incident = _build_incident_from_payload(payload)

        # Use the existing rule-based incident analysis.
        analysis_result = analyze_incident(incident)

        return jsonify({
            "success": True,
            "analysis": _make_json_safe(analysis_result),
            "privacy": {
                "message_persisted": False,
                "sent_to_google": False,
                "notice": (
                    "This endpoint analyses the incident in memory. "
                    "It does not intentionally save the incident or "
                    "send its message to Google."
                ),
            },
        }), 200

    except ValueError as exc:
        return jsonify({
            "success": False,
            "error": str(exc),
        }), 400

    except Exception:
        # Do not expose submitted text or internal exception details.
        return jsonify({
            "success": False,
            "error": (
                "Incident analysis could not be completed. "
                "Check the incident model and analysis function."
            ),
        }), 500


# ============================================================
# 3. ALL ROUTES ARE NOW REGISTERED.
# Only now create a test client and send requests.
# ============================================================

api_test_client = signalher_app.test_client()

# Test A — Health
response = api_test_client.get("/api/health")
assert response.status_code == 200
assert response.get_json()["status"] == "running"

# Test B — Categories
response = api_test_client.get("/api/categories")
assert response.status_code == 200
assert len(response.get_json()["categories"]) == len(CATEGORY_LABELS)

# Test C — Pan-India resources
response = api_test_client.get(
    "/api/resources?category=image_based_abuse"
)
assert response.status_code == 200

resource_data = response.get_json()
resource_ids = {
    item["resource_id"] for item in resource_data["resources"]
}

assert "cybercrime_portal" in resource_ids
assert "google_ncii_removal" in resource_ids
assert "stopncii" in resource_ids
assert "take_it_down" in resource_ids
assert "women_helpline_181" not in resource_ids

# Test D — Category-only search query
response = api_test_client.post(
    "/api/search-query",
    json={
        "category": "image_based_abuse",
        "message": "PRIVATE TEST MESSAGE",
    },
)
assert response.status_code == 200

query_data = response.get_json()
assert query_data["category"] == "image_based_abuse"
assert "PRIVATE TEST MESSAGE" not in query_data["query"]

# Test E — Incident analysis with synthetic content
response = api_test_client.post(
    "/api/analyze-incident",
    json={
        "platform": "WhatsApp",
        "timestamp": "2026-10-24T10:00:00+05:30",
        "message_text": (
            "If you don't send money, I will expose your private images."
        ),
    },
)

print("Incident endpoint HTTP status:", response.status_code)

if response.status_code != 200:
    print("Incident endpoint response:", response.get_json())

assert response.status_code == 200
analysis_data = response.get_json()

assert analysis_data["success"] is True
assert "analysis" in analysis_data
assert analysis_data["privacy"]["message_persisted"] is False
assert analysis_data["privacy"]["sent_to_google"] is False

# Test F — Reject an empty message
response = api_test_client.post(
    "/api/analyze-incident",
    json={
        "platform": "WhatsApp",
        "message_text": "   ",
    },
)
assert response.status_code == 400

# Test G — Reject unsupported categories
response = api_test_client.post(
    "/api/search-query",
    json={"category": "unknown_category"},
)
assert response.status_code == 400


print("\nAll API checks passed:")
print("1. Health endpoint")
print("2. Categories endpoint")
print("3. Pan-India resource endpoint")
print("4. Privacy-safe search-query endpoint")
print("5. Incident analysis endpoint")
print("6. Empty-message validation")
print("7. Invalid-category validation")

print("\nRegistered routes:")
for rule in signalher_app.url_map.iter_rules():
    if rule.endpoint != "static":
        print(rule.rule, sorted(rule.methods - {"HEAD", "OPTIONS"}))

print("\nSynthetic incident analysis result:")
print(analysis_data["analysis"])

In [ ]:
# ============================================================
# SIGNALHER — TIMELINE ENDPOINT DIAGNOSTICS
# Find the underlying error without changing existing routes.
# ============================================================

import traceback
import inspect

print("1. Previous HTTP status:")
print(timeline_response.status_code)

print("\n2. Previous API response:")
print(timeline_response.get_json())

print("\n3. TimelineEntry model fields:")
print(list(TimelineEntry.__dataclass_fields__.keys()))

print("\n4. Existing function signatures:")
print("analyze_timeline:", inspect.signature(analyze_timeline))
print("generate_action_plan:", inspect.signature(generate_action_plan))

print("\n5. Testing each stage separately...")

try:
    diagnostic_entries = [
        _build_timeline_entry_from_payload(entry)
        for entry in synthetic_timeline_payload["entries"]
    ]
    print("PASS: TimelineEntry objects created")
    print("Number of entries:", len(diagnostic_entries))

    diagnostic_timeline = analyze_timeline(diagnostic_entries)
    print("\nPASS: Timeline analysis completed")
    print("Timeline result:", _make_json_safe(diagnostic_timeline))

    diagnostic_plan = generate_action_plan(
        diagnostic_timeline,
        immediate_danger=False,
    )
    print("\nPASS: Action plan generated")
    print("Action plan:", _make_json_safe(diagnostic_plan))

except Exception as exc:
    print("\nFAILED STAGE:", type(exc).__name__)
    print("Error:", str(exc))
    print("\nTraceback:")
    traceback.print_exc()

In [ ]:
# ============================================================
# SIGNALHER — RESOURCE DIRECTORY COMPATIBILITY FIX
#
# Supports both:
#   RESOURCE_DIRECTORY iteration for resource search
#   RESOURCE_DIRECTORY["resource_id"] for action plans
# ============================================================

from dataclasses import asdict
import traceback


class CompatibleResourceDirectory(list):
    """
    Behaves like a list when iterating over resources, while also
    supporting dictionary-style access by resource_id.

    This preserves compatibility with the existing action-plan code.
    """

    def __init__(self, resources):
        super().__init__(resources)
        self._resource_map = {
            resource.resource_id: resource
            for resource in resources
        }

    def _as_mapping(self, resource):
        data = asdict(resource)

        # Preserve common keys used by older resource-directory code.
        data["description"] = resource.purpose
        data["when_to_use"] = list(resource.categories)
        data["official_url"] = resource.url

        return data

    def __getitem__(self, key):
        if isinstance(key, str):
            resource = self._resource_map.get(key)

            if resource is None:
                raise KeyError(
                    f"Unknown resource ID: {key}"
                )

            return self._as_mapping(resource)

        # Normal integer and slice access remains list-compatible.
        return super().__getitem__(key)

    def get(self, key, default=None):
        """Dictionary-style safe lookup."""
        resource = self._resource_map.get(key)

        if resource is None:
            return default

        return self._as_mapping(resource)

    def keys(self):
        return self._resource_map.keys()

    def values(self):
        return (
            self._as_mapping(resource)
            for resource in self._resource_map.values()
        )

    def items(self):
        return (
            (resource_id, self._as_mapping(resource))
            for resource_id, resource in self._resource_map.items()
        )


# Preserve the actual resource objects already in the directory.
_existing_resources = list(RESOURCE_DIRECTORY)

assert _existing_resources, "Resource directory is unexpectedly empty."

assert all(
    isinstance(resource, SafetyResource)
    for resource in _existing_resources
), "The directory contains an unexpected resource type."

RESOURCE_DIRECTORY = CompatibleResourceDirectory(
    _existing_resources
)


# ------------------------------------------------------------
# TEST 1 — Both access patterns work
# ------------------------------------------------------------

assert len(RESOURCE_DIRECTORY) == 6

# List-style access: returns a SafetyResource object.
assert isinstance(RESOURCE_DIRECTORY[0], SafetyResource)

# Dictionary-style access: returns a mapping.
emergency_resource = RESOURCE_DIRECTORY["emergency_112"]

assert emergency_resource["url"] == "https://112.gov.in/"
assert emergency_resource["name"]
assert emergency_resource["description"]

print("List-style resource access: PASSED")
print("ID-based resource lookup: PASSED")
print("Legacy action-plan field compatibility: PASSED")


# ------------------------------------------------------------
# TEST 2 — Re-run timeline and action-plan logic directly
# ------------------------------------------------------------

try:
    _test_entries = [
        _build_timeline_entry_from_payload(entry)
        for entry in synthetic_timeline_payload["entries"]
    ]

    _test_timeline = analyze_timeline(_test_entries)

    _test_action_plan = generate_action_plan(
        _test_timeline,
        immediate_danger=False,
    )

    assert isinstance(_test_timeline, dict)
    assert isinstance(_test_action_plan, dict)

    print("\nTimeline analysis: PASSED")
    print("Action-plan generation: PASSED")

except Exception:
    print("\nThe action-plan generator still needs attention.")
    traceback.print_exc()
    raise


# ------------------------------------------------------------
# TEST 3 — Re-test the existing timeline API route
# ------------------------------------------------------------

# Reuse the existing Flask app and route registration.
# Do not register any more routes in this cell.

api_test_client = signalher_app.test_client()

_timeline_api_response = api_test_client.post(
    "/api/analyze-timeline",
    json=synthetic_timeline_payload,
)

print("\nTimeline API HTTP status:", _timeline_api_response.status_code)

if _timeline_api_response.status_code != 200:
    print("API response:", _timeline_api_response.get_json())

assert _timeline_api_response.status_code == 200

_timeline_api_data = _timeline_api_response.get_json()

assert _timeline_api_data["success"] is True
assert "timeline" in _timeline_api_data
assert "action_plan" in _timeline_api_data

print("Timeline API integration: PASSED")
print("Combined timeline and action plan: PASSED")

print("\nAction-plan response preview:")
print(_timeline_api_data["action_plan"])

In [ ]:
# ============================================================
# SIGNALHER — LOCAL FRONTEND ACCESS (CORS)
# Allowed development origins:
#   WebStorm:      localhost:63342
#   Vite:          localhost:5173
#   Live Server:   localhost:5500
#
# No arbitrary website origins or credentials are allowed.
# ============================================================

from flask import Flask, request

# Create a fresh app because the previous app has already handled requests.
_previous_app = signalher_app
_signalher_v4 = Flask("signalher_api_v4")

# Copy all existing routes before making any new requests.
for rule in _previous_app.url_map.iter_rules():
    if rule.endpoint == "static":
        continue

    methods = sorted(rule.methods - {"HEAD", "OPTIONS"})

    _signalher_v4.add_url_rule(
        rule.rule,
        endpoint=rule.endpoint,
        view_func=_previous_app.view_functions[rule.endpoint],
        methods=methods,
    )

# Local development origins only.
ALLOWED_DEV_ORIGINS = {
    "http://localhost:63342",
    "http://127.0.0.1:63342",
    "http://localhost:5173",
    "http://127.0.0.1:5173",
    "http://localhost:5500",
    "http://127.0.0.1:5500",
}


@_signalher_v4.after_request
def add_local_cors_headers(response):
    origin = request.headers.get("Origin")

    # Only grant browser access to explicitly allowed local origins.
    if origin in ALLOWED_DEV_ORIGINS:
        response.headers["Access-Control-Allow-Origin"] = origin
        response.headers["Access-Control-Allow-Methods"] = (
            "GET, POST, OPTIONS"
        )
        response.headers["Access-Control-Allow-Headers"] = (
            "Content-Type"
        )
        response.headers["Access-Control-Max-Age"] = "600"

    response.headers["Vary"] = "Origin"
    return response


# Switch to the configured app only after all routes and hooks are ready.
signalher_app = _signalher_v4

# ============================================================
# TESTS — NO LIVE SERVER REQUIRED YET
# ============================================================

cors_test_client = signalher_app.test_client()

# 1. Health endpoint still works.
health = cors_test_client.get("/api/health")
assert health.status_code == 200

# 2. An allowed WebStorm origin receives CORS permission.
allowed_origin = "http://localhost:63342"

allowed_response = cors_test_client.get(
    "/api/health",
    headers={"Origin": allowed_origin},
)

assert allowed_response.status_code == 200
assert (
    allowed_response.headers.get("Access-Control-Allow-Origin")
    == allowed_origin
)

# 3. An arbitrary origin is not granted permission.
unapproved_response = cors_test_client.get(
    "/api/health",
    headers={"Origin": "https://unapproved-example.com"},
)

assert (
    unapproved_response.headers.get("Access-Control-Allow-Origin")
    is None
)

# 4. Simulate the browser's preflight request for a POST.
preflight_response = cors_test_client.options(
    "/api/analyze-incident",
    headers={
        "Origin": allowed_origin,
        "Access-Control-Request-Method": "POST",
        "Access-Control-Request-Headers": "Content-Type",
    },
)

assert preflight_response.status_code == 200
assert (
    preflight_response.headers.get("Access-Control-Allow-Origin")
    == allowed_origin
)
assert "POST" in preflight_response.headers.get(
    "Access-Control-Allow-Methods", ""
)

# 5. Timeline route still works.
timeline_check = cors_test_client.post(
    "/api/analyze-timeline",
    json=synthetic_timeline_payload,
)
assert timeline_check.status_code == 200

# 6. Incident endpoint still works.
incident_check = cors_test_client.post(
    "/api/analyze-incident",
    json={
        "platform": "WhatsApp",
        "message_text": (
            "If you don't send money, I will expose your private images."
        ),
    },
)
assert incident_check.status_code == 200

print("Local CORS configuration: PASSED")
print("Allowed WebStorm origin: PASSED")
print("Unapproved origin blocked: PASSED")
print("Browser POST preflight: PASSED")
print("Timeline endpoint preserved: PASSED")
print("Incident endpoint preserved: PASSED")

print("\nAllowed development origins:")
for origin in sorted(ALLOWED_DEV_ORIGINS):
    print("-", origin)

print("\nAvailable API routes:")
for rule in signalher_app.url_map.iter_rules():
    if rule.endpoint != "static":
        print(
            f"{', '.join(sorted(rule.methods - {'HEAD', 'OPTIONS'}))} "
            f"{rule.rule}"
        )

In [ ]:
# ============================================================
# SIGNALHER — START LOCAL BACKEND AND TEST OVER HTTP
#
# Local address: http://127.0.0.1:5000
# Binds only to this computer, not to the public network.
# ============================================================

import json
import time
import urllib.request
import urllib.error
from threading import Thread
from werkzeug.serving import make_server


# ------------------------------------------------------------
# 1. Start the server only if it is not already running
# ------------------------------------------------------------

if (
    "_signalher_server_thread" in globals()
    and _signalher_server_thread.is_alive()
):
    print("SignalHer local server is already running.")

else:
    _signalher_http_server = make_server(
        host="127.0.0.1",
        port=5000,
        app=signalher_app,
        threaded=True,
    )

    _signalher_server_thread = Thread(
        target=_signalher_http_server.serve_forever,
        daemon=True,
        name="SignalHerLocalServer",
    )

    _signalher_server_thread.start()
    print("SignalHer local server started.")


BASE_URL = "http://127.0.0.1:5000"


# ------------------------------------------------------------
# 2. Helper for making real HTTP requests
# ------------------------------------------------------------

def signalher_http_request(method, path, payload=None):
    request_data = None
    headers = {
        "Accept": "application/json",
        "Origin": "http://localhost:63342",
    }

    if payload is not None:
        request_data = json.dumps(payload).encode("utf-8")
        headers["Content-Type"] = "application/json"

    req = urllib.request.Request(
        url=BASE_URL + path,
        data=request_data,
        headers=headers,
        method=method,
    )

    try:
        with urllib.request.urlopen(req, timeout=10) as response:
            body = response.read().decode("utf-8")
            return (
                response.status,
                json.loads(body),
                response.headers,
            )

    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8")
        try:
            parsed_body = json.loads(body)
        except json.JSONDecodeError:
            parsed_body = {"response": body}

        return exc.code, parsed_body, exc.headers


# ------------------------------------------------------------
# 3. Wait until the server responds
# ------------------------------------------------------------

health_status = None
health_data = None
health_headers = None

for attempt in range(20):
    try:
        health_status, health_data, health_headers = (
            signalher_http_request("GET", "/api/health")
        )
        break
    except Exception:
        time.sleep(0.25)

assert health_status == 200, (
    "The local server did not respond successfully. "
    "Check whether port 5000 is available."
)

assert health_data["status"] == "running"

print("Local HTTP health check: PASSED")


# ------------------------------------------------------------
# 4. Test categories
# ------------------------------------------------------------

status, categories_data, _ = signalher_http_request(
    "GET", "/api/categories"
)

assert status == 200
assert categories_data["success"] is True
assert len(categories_data["categories"]) > 0

print("HTTP categories endpoint: PASSED")


# ------------------------------------------------------------
# 5. Test nationwide resource lookup
# ------------------------------------------------------------

status, resources_data, _ = signalher_http_request(
    "GET",
    "/api/resources?category=image_based_abuse",
)

assert status == 200
assert resources_data["success"] is True

print("HTTP nationwide resources endpoint: PASSED")


# ------------------------------------------------------------
# 6. Test safe search-query generation
# ------------------------------------------------------------

status, query_data, _ = signalher_http_request(
    "POST",
    "/api/search-query",
    {
        "category": "image_based_abuse",
    },
)

assert status == 200
assert query_data["success"] is True
assert "India" in query_data["query"]

print("HTTP search-query endpoint: PASSED")


# ------------------------------------------------------------
# 7. Test incident analysis with synthetic content
# ------------------------------------------------------------

status, incident_data, _ = signalher_http_request(
    "POST",
    "/api/analyze-incident",
    {
        "platform": "WhatsApp",
        "message_text": (
            "If you don't send money, I will expose your private images."
        ),
    },
)

assert status == 200
assert incident_data["success"] is True
assert "analysis" in incident_data

print("HTTP incident-analysis endpoint: PASSED")


# ------------------------------------------------------------
# 8. Test timeline and action-plan generation
# ------------------------------------------------------------

status, timeline_data, _ = signalher_http_request(
    "POST",
    "/api/analyze-timeline",
    synthetic_timeline_payload,
)

assert status == 200
assert timeline_data["success"] is True
assert "timeline" in timeline_data
assert "action_plan" in timeline_data

print("HTTP timeline and action-plan endpoint: PASSED")


# ------------------------------------------------------------
# 9. Verify CORS on the actual HTTP response
# ------------------------------------------------------------

cors_origin = health_headers.get(
    "Access-Control-Allow-Origin"
)

assert cors_origin == "http://localhost:63342"

print("HTTP CORS response header: PASSED")

print("\nAll local HTTP checks passed.")
print("Backend address:", BASE_URL)
print("API is listening on this computer only.")

In [ ]:

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <meta
        name="description"
        content="SignalHer helps users review possible online coercion indicators, organise incidents, and explore safety resources."
    >
    <title>SignalHer — Your Digital Safety Companion</title>

    <link rel="stylesheet" href="styles.css">
    <script src="app.js" defer></script>
</head>

<body>
    <header class="site-header">
        <a class="brand" href="#" aria-label="SignalHer home">
            <span class="brand-mark" aria-hidden="true">S</span>
            <span>SignalHer</span>
        </a>

        <div id="backend-status" class="status-pill" role="status">
            Checking backend…
        </div>
    </header>

    <main class="container">
        <section class="hero">
            <p class="eyebrow">YOUR DIGITAL SAFETY, YOUR CHOICES</p>
            <h1>Understand the signals.<br>Choose your next step.</h1>
            <p class="hero-description">
                Review potentially concerning messages, organise incidents
                you choose to link, and explore support resources across India.
            </p>
        </section>

        <section class="notice notice-safety" aria-label="Safety information">
            <strong>Your safety comes first.</strong>
            <p>
                SignalHer provides preliminary rule-based indicators, not
                proof of abuse, a legal assessment, or a prediction of danger.
                If you believe you are in immediate danger in India, call 112
                rather than relying on this application.
            </p>
        </section>

        <section class="panel" id="incident-section">
            <div class="section-heading">
                <div>
                    <p class="eyebrow">01 / REVIEW A MESSAGE</p>
                    <h2>Incident analysis</h2>
                    <p class="section-description">
                        Paste a message you choose to review. The analysis
                        runs through your local SignalHer backend.
                    </p>
                </div>
            </div>

            <form id="incident-form">
                <div class="form-grid">
                    <div class="field">
                        <label for="platform">Platform</label>
                        <select id="platform" name="platform" required>
                            <option value="WhatsApp">WhatsApp</option>
                            <option value="Instagram">Instagram</option>
                            <option value="Email">Email</option>
                            <option value="LinkedIn">LinkedIn</option>
                            <option value="Other">Other</option>
                        </select>
                    </div>

                    <div class="field">
                        <label for="direction">Message direction</label>
                        <select id="direction" name="direction" required>
                            <option value="incoming">Received by me</option>
                            <option value="outgoing">Sent by me</option>
                            <option value="unknown">Not sure</option>
                        </select>
                    </div>
                </div>

                <div class="field">
                    <label for="message-text">Message text</label>
                    <textarea
                        id="message-text"
                        name="message_text"
                        rows="6"
                        maxlength="5000"
                        placeholder="Paste only the text you want to analyse…"
                        required
                    ></textarea>
                    <div class="field-hint">
                        <span>Maximum 5,000 characters. Remove unnecessary personal details.</span>
                        <span id="character-count">0 / 5000</span>
                    </div>
                </div>

                <div class="button-row">
                    <button class="button button-primary" type="submit">
                        Analyse message
                    </button>
                    <button
                        class="button button-secondary"
                        type="button"
                        id="add-to-timeline-btn"
                    >
                        Add to timeline
                    </button>
                </div>
            </form>

            <div
                id="incident-results"
                class="results-area"
                aria-live="polite"
                aria-atomic="false"
            ></div>
        </section>

        <section class="panel" id="timeline-section">
            <div class="section-heading">
                <div>
                    <p class="eyebrow">02 / SEE THE CONTEXT</p>
                    <h2>Incident timeline</h2>
                    <p class="section-description">
                        Add individual incidents and review their sequence.
                        Nothing is automatically linked by identity.
                    </p>
                </div>

                <span class="count-badge" id="timeline-count">0 entries</span>
            </div>

            <div class="notice">
                <strong>Link incidents only when appropriate.</strong>
                <p>
                    By default, entries remain unlinked. Select the option
                    below only if you want SignalHer to compare the entries
                    under one case label that you provide.
                </p>
            </div>

            <div class="grouping-box">
                <label class="checkbox-label" for="group-case">
                    <input type="checkbox" id="group-case">
                    I confirm these entries belong to the same user-selected case
                </label>

                <div class="field">
                    <label for="case-label">Your case label (optional)</label>
                    <input
                        type="text"
                        id="case-label"
                        maxlength="100"
                        placeholder="For example, Case A"
                    >
                    <span class="field-hint">
                        This is a label you choose, not a verified identity.
                    </span>
                </div>
            </div>

            <ol id="timeline-list" class="timeline-list">
                <li id="empty-timeline" class="empty-state">
                    No incidents added yet. Analyse a message or add it
                    to the timeline to begin.
                </li>
            </ol>

            <div class="field danger-toggle">
                <label class="checkbox-label" for="immediate-danger">
                    <input type="checkbox" id="immediate-danger">
                    I believe someone may harm me imminently
                </label>
                <p class="field-hint">
                    This is your own selection, not a risk score generated
                    by SignalHer.
                </p>
            </div>

            <div class="button-row">
                <button
                    class="button button-primary"
                    type="button"
                    id="analyze-timeline-btn"
                >
                    Analyse timeline
                </button>
                <button
                    class="button button-secondary"
                    type="button"
                    id="clear-timeline-btn"
                >
                    Clear entries
                </button>
            </div>

            <div
                id="timeline-results"
                class="results-area"
                aria-live="polite"
                aria-atomic="false"
            ></div>
        </section>

        <section class="panel" id="resources-section">
            <div class="section-heading">
                <div>
                    <p class="eyebrow">03 / EXPLORE SUPPORT</p>
                    <h2>Safety resource navigator</h2>
                    <p class="section-description">
                        Explore curated resources relevant to your selected
                        category. Check each provider's eligibility and guidance.
                    </p>
                </div>
            </div>

            <div class="resource-search-row">
                <div class="field">
                    <label for="resource-category">What do you need help with?</label>
                    <select id="resource-category">
                        <option value="image_based_abuse">
                            Non-consensual intimate imagery
                        </option>
                        <option value="cybercrime">
                            Cybercrime reporting
                        </option>
                        <option value="coercion">
                            Online threats and coercion
                        </option>
                        <option value="identity_misuse">
                            Online impersonation
                        </option>
                        <option value="physical_safety">
                            Physical safety
                        </option>
                        <option value="women_support">
                            Women's support services
                        </option>
                        <option value="emergency">
                            Emergency support
                        </option>
                    </select>
                </div>
            </div>

            <div class="button-row">
                <button
                    class="button button-primary"
                    type="button"
                    id="load-resources-btn"
                >
                    Find relevant resources
                </button>

                <a
                    id="google-search-link"
                    class="button button-secondary"
                    href="https://www.google.com/"
                    target="_blank"
                    rel="noopener noreferrer"
                    hidden
                >
                    Search Google
                </a>
            </div>

            <p class="field-hint">
                Google searches use only a general support category and India,
                never the message you submitted. Search results can include
                third-party sites; verify websites before sharing information.
            </p>

            <div
                id="resource-list"
                class="resource-list"
                aria-live="polite"
                aria-atomic="false"
            ></div>
        </section>
    </main>

    <footer class="site-footer">
        <div class="container">
            <p><strong>SignalHer</strong> — a privacy-conscious digital safety prototype.</p>
            <p>
                During this prototype, incident analysis runs on your local
                backend. Timeline entries remain in page memory until cleared
                or the page is refreshed, and are sent to the local backend
                when you request timeline analysis. They are not sent to Google.
            </p>
            <p class="footer-small">
                This tool does not replace emergency services, professional
                advice, or official reporting channels.
            </p>
        </div>
    </footer>
</body>
</html>


In [ ]:
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>SignalHer — Your Digital Safety Companion</title>
</head>
<body>
</body>
</html>

In [ ]:
############################################################################################################################
############################################################################################################################

In [ ]:
# ============================================================
# SIGNALHER — BACKEND AUDIT
# Inspect source handling, resource URLs and availability.
# This is diagnostic only; it does not modify the application.
# ============================================================

from enum import Enum
from dataclasses import fields
from urllib.parse import urlparse


# ------------------------------------------------------------
# 1. Inspect the current Incident schema
# ------------------------------------------------------------

print("=" * 60)
print("AUDIT 1 — INCIDENT MODEL")
print("=" * 60)

for field in fields(Incident):
    annotation = field.type
    print(f"{field.name}: {annotation}")

    # If the platform is represented by an enum, show its values.
    if field.name == "platform":
        enum_candidates = [
            annotation,
            *getattr(annotation, "__args__", ()),
        ]

        for candidate in enum_candidates:
            if isinstance(candidate, type) and issubclass(candidate, Enum):
                print(
                    "Supported platform values:",
                    [member.value for member in candidate],
                )
                break


# ------------------------------------------------------------
# 2. Test the same message with varied source metadata
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("AUDIT 2 — SOURCE-INDEPENDENT ANALYSIS")
print("=" * 60)

audit_client = signalher_app.test_client()

sample_message = "I will expose your private images"

source_test_cases = [
    ("unspecified", {}),
    ("WhatsApp", {"platform": "WhatsApp"}),
    ("Discord", {"platform": "Discord"}),
    ("X", {"platform": "X"}),
    ("LinkedIn", {"platform": "LinkedIn"}),
    ("Email", {"platform": "Email"}),
    ("Instagram", {"platform": "Instagram"}),
]

source_audit_results = {}

for source_name, extra_fields in source_test_cases:
    payload = {
        **extra_fields,
        "message_text": sample_message,
    }

    response = audit_client.post(
        "/api/analyze-incident",
        json=payload,
    )

    if response.status_code == 200:
        body = response.get_json()
        analysis = body.get("analysis", {})
        findings = analysis.get("findings", [])

        matched_rule_ids = sorted(
            finding.get("rule_id")
            for finding in findings
            if finding.get("rule_id")
        )

        source_audit_results[source_name] = {
            "status_code": response.status_code,
            "rule_ids": matched_rule_ids,
        }

        print(
            f"{source_name:15} HTTP {response.status_code} | "
            f"rules: {matched_rule_ids}"
        )

    else:
        source_audit_results[source_name] = {
            "status_code": response.status_code,
            "error": response.get_json(),
        }

        print(
            f"{source_name:15} HTTP {response.status_code} | "
            f"response: {response.get_json()}"
        )


# ------------------------------------------------------------
# 3. Inspect resource URLs and jurisdiction metadata
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("AUDIT 3 — ACTION-PLAN RESOURCE INTEGRITY")
print("=" * 60)

audit_entries = [
    _build_timeline_entry_from_payload(entry)
    for entry in synthetic_timeline_payload["entries"]
]

audit_timeline = analyze_timeline(audit_entries)

audit_action_plan = generate_action_plan(
    audit_timeline,
    immediate_danger=False,
)

audit_resources = audit_action_plan.get("resources", [])

for resource in audit_resources:
    resource_id = resource.get("resource_id", "unknown")
    resource_url = resource.get("url", "")
    exclusions = resource.get("excluded_states", [])

    is_plain_https_url = (
        isinstance(resource_url, str)
        and urlparse(resource_url).scheme == "https"
        and bool(urlparse(resource_url).netloc)
        and not resource_url.startswith("[")
        and "](" not in resource_url
    )

    print(f"\nResource: {resource.get('name', resource_id)}")
    print("ID:", resource_id)
    print("Returned URL:", resource_url)
    print("Plain HTTPS URL:", is_plain_https_url)
    print("State exclusions:", exclusions)

    if exclusions:
        print(
            "Review required: this resource has jurisdiction restrictions."
        )


# ------------------------------------------------------------
# 4. Summarise what requires repair
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("AUDIT SUMMARY")
print("=" * 60)

rejected_sources = [
    name
    for name, result in source_audit_results.items()
    if result["status_code"] != 200
]

print("Sources rejected by current API:", rejected_sources)
print("Resources returned by action plan:", len(audit_resources))

urls_needing_repair = [
    resource.get("resource_id", "unknown")
    for resource in audit_resources
    if not (
        isinstance(resource.get("url"), str)
        and urlparse(resource.get("url", "")).scheme == "https"
        and bool(urlparse(resource.get("url", "")).netloc)
        and not resource.get("url", "").startswith("[")
        and "](" not in resource.get("url", "")
    )
]

print("Resources with URLs needing repair:", urls_needing_repair)
print("Diagnostic audit completed; no application settings were changed.")

In [ ]:
# ============================================================
# SIGNALHER — SOURCE-INDEPENDENCE AND RESOURCE FIXES
# 1. Accept optional and unfamiliar platform labels.
# 2. Preserve the supplied source label as separate metadata.
# 3. Exclude geographically restricted resources when location
#    is unknown in the general action plan.
# 4. Test the changes using synthetic messages.
# ============================================================

from enum import Enum
from urllib.parse import urlparse
from flask import jsonify, request


# ------------------------------------------------------------
# FIX 1 — Unfamiliar platforms map to Other internally
# ------------------------------------------------------------

def _coerce_platform(value, annotation):
    """
    Platform is optional contextual metadata, not a condition
    for analysing the message text.

    Recognised labels map to the existing enum.
    Unrecognised labels, such as Discord or X, map to Other.
    The original label is preserved separately in API metadata.
    """
    enum_types = []

    candidates = [annotation]
    try:
        from typing import get_args
        candidates.extend(get_args(annotation))
    except Exception:
        pass

    for candidate in candidates:
        if (
            isinstance(candidate, type)
            and issubclass(candidate, Enum)
            and candidate not in enum_types
        ):
            enum_types.append(candidate)

    raw_label = (
        str(value).strip()
        if value is not None and str(value).strip()
        else "Other"
    )
    normalised_label = _normalise_text(raw_label)

    for enum_type in enum_types:
        for member in enum_type:
            recognised_labels = {
                _normalise_text(member.name),
                _normalise_text(str(member.value)),
            }

            if normalised_label in recognised_labels:
                return member

    # Unknown labels do not stop message analysis.
    for enum_type in enum_types:
        for member in enum_type:
            if (
                _normalise_text(member.name) == "other"
                or _normalise_text(str(member.value)) == "other"
            ):
                return member

    # Preserve previous failure behaviour if no suitable enum exists.
    if enum_types:
        raise ValueError("The platform model has no 'Other' option.")

    return raw_label


# ------------------------------------------------------------
# FIX 2 — Preserve source labels outside the analysis model
# ------------------------------------------------------------

def _source_label_from_payload(payload):
    """
    Prefer an explicit source_label. Otherwise use platform.
    A missing or blank label remains None.
    """
    raw_label = payload.get("source_label")

    if raw_label is None:
        raw_label = payload.get("platform")

    if not isinstance(raw_label, str) or not raw_label.strip():
        return None

    # Bound the displayed metadata length.
    return raw_label.strip()[:80]


def _attach_source_context(result, source_context):
    """
    Add source metadata to a successful JSON response without
    changing the actual analysis findings or error responses.
    """
    is_tuple = isinstance(result, tuple)

    if is_tuple:
        response = result[0]
        status_code = (
            result[1]
            if len(result) > 1 and isinstance(result[1], int)
            else response.status_code
        )
    else:
        response = result
        status_code = response.status_code

    if status_code != 200:
        return result

    body = response.get_json(silent=True)

    if not isinstance(body, dict):
        return result

    body["source_context"] = source_context
    updated_response = jsonify(body)

    if is_tuple:
        return updated_response, status_code

    return updated_response


# Wrap the existing incident endpoint without adding a new route.
if "_signalher_original_incident_view" not in globals():
    _signalher_original_incident_view = (
        signalher_app.view_functions["api_analyze_incident"]
    )


def _incident_view_with_source_context():
    payload = request.get_json(silent=True)
    result = _signalher_original_incident_view()

    if not isinstance(payload, dict):
        return result

    label = _source_label_from_payload(payload)

    return _attach_source_context(
        result,
        {
            "provided": label is not None,
            "label": label,
            "role": (
                "Optional context only. Rule matching is based on "
                "message text, not on the source label."
            ),
        },
    )


signalher_app.view_functions[
    "api_analyze_incident"
] = _incident_view_with_source_context


# Add source labels to timeline responses in submitted-entry order.
if "_signalher_original_timeline_view" not in globals():
    _signalher_original_timeline_view = (
        signalher_app.view_functions["api_analyze_timeline"]
    )


def _timeline_view_with_source_context():
    payload = request.get_json(silent=True)
    result = _signalher_original_timeline_view()

    if not isinstance(payload, dict):
        return result

    raw_entries = payload.get("entries", [])
    source_entries = []

    if isinstance(raw_entries, list):
        for index, item in enumerate(raw_entries):
            if not isinstance(item, dict):
                continue

            label = _source_label_from_payload(item)

            source_entries.append({
                "entry_number": index + 1,
                "source_label_provided": label is not None,
                "source_label": label,
            })

    return _attach_source_context(
        result,
        {
            "entries_in_submitted_order": source_entries,
            "role": (
                "Optional source metadata only. Entry labels do not "
                "verify identity or determine rule matches."
            ),
        },
    )


signalher_app.view_functions[
    "api_analyze_timeline"
] = _timeline_view_with_source_context


# ------------------------------------------------------------
# FIX 3 — Filter state-restricted resources from general plans
# ------------------------------------------------------------

if "_signalher_original_action_plan" not in globals():
    _signalher_original_action_plan = generate_action_plan


def generate_action_plan(timeline_result, *, immediate_danger=False):
    """
    Preserve the existing action-plan generator while ensuring that
    resources known to exclude certain states aren't presented as
    universally available when no location was provided.
    """
    plan = _signalher_original_action_plan(
        timeline_result,
        immediate_danger=immediate_danger,
    )

    kept_resources = []

    for item in plan.get("resources", []):
        if not isinstance(item, dict):
            continue

        # Jurisdiction-restricted resources need a location check.
        # The current general action-plan generator receives no state.
        if item.get("excluded_states"):
            continue

        # Obtain the canonical URL from the curated directory.
        resource_id = item.get("resource_id")
        canonical_resource = RESOURCE_DIRECTORY.get(resource_id)

        if canonical_resource:
            canonical_url = canonical_resource.get("url")

            if canonical_url:
                item["url"] = canonical_url
                item["official_url"] = canonical_url

        kept_resources.append(item)

    plan["resources"] = kept_resources
    plan["resource_availability_note"] = (
        "Resources with known geographic restrictions are omitted "
        "from this general plan when no location is specified. "
        "Check official services for current eligibility and availability."
    )

    return plan


# ------------------------------------------------------------
# TEST 1 — Source-independent analysis
# ------------------------------------------------------------

source_test_cases = [
    ("unspecified", {}),
    ("WhatsApp", {"platform": "WhatsApp"}),
    ("Instagram", {"platform": "Instagram"}),
    ("Email", {"platform": "Email"}),
    ("LinkedIn", {"platform": "LinkedIn"}),
    ("Discord", {"platform": "Discord"}),
    ("X", {"platform": "X"}),
]

source_test_results = {}
api_client = signalher_app.test_client()

for source_name, source_fields in source_test_cases:
    response = api_client.post(
        "/api/analyze-incident",
        json={
            **source_fields,
            "message_text": "I will expose your private images",
        },
    )

    assert response.status_code == 200, (
        f"{source_name} was rejected: {response.get_json()}"
    )

    body = response.get_json()
    finding_ids = sorted(
        finding["rule_id"]
        for finding in body["analysis"]["findings"]
    )

    source_test_results[source_name] = finding_ids

    expected_label = (
        source_fields.get("source_label")
        or source_fields.get("platform")
    )

    assert body["source_context"]["label"] == expected_label

# The same message should produce the same configured findings
# regardless of platform metadata.
unique_findings = {
    tuple(rule_ids)
    for rule_ids in source_test_results.values()
}

assert len(unique_findings) == 1, (
    "The same message produced different findings across sources."
)

print("Optional source handling: PASSED")
print("Discord acceptance: PASSED")
print("X acceptance: PASSED")
print("Source labels preserved separately: PASSED")
print("Same message, same rule findings: PASSED")


# ------------------------------------------------------------
# TEST 2 — Resource filtering and URL integrity
# ------------------------------------------------------------

audit_entries = [
    _build_timeline_entry_from_payload(entry)
    for entry in synthetic_timeline_payload["entries"]
]

audit_timeline = analyze_timeline(audit_entries)

fixed_plan = generate_action_plan(
    audit_timeline,
    immediate_danger=False,
)

fixed_resources = fixed_plan.get("resources", [])
fixed_resource_ids = {
    item["resource_id"] for item in fixed_resources
}

assert "women_helpline_181" not in fixed_resource_ids

for item in fixed_resources:
    url = item.get("url", "")
    parsed_url = urlparse(url)

    assert parsed_url.scheme == "https", (
        f"Invalid HTTPS resource URL: {item.get('resource_id')}"
    )
    assert parsed_url.netloc, (
        f"Missing hostname: {item.get('resource_id')}"
    )
    assert "](" not in url, (
        f"URL appears to contain Markdown formatting: {url}"
    )

print("Geographically restricted resource filtering: PASSED")
print("Action-plan HTTPS URL validation: PASSED")


# ------------------------------------------------------------
# TEST 3 — Location-aware resource directory still works
# ------------------------------------------------------------

tripura_resources = find_resources(
    category="women_support",
    state="Tripura",
    country="India",
)

west_bengal_resources = find_resources(
    category="women_support",
    state="West Bengal",
    country="India",
)

assert "women_helpline_181" in {
    item["resource_id"] for item in tripura_resources
}

assert "women_helpline_181" not in {
    item["resource_id"] for item in west_bengal_resources
}

print("Explicit state-aware resource matching: PASSED")


# ------------------------------------------------------------
# TEST 4 — Timeline endpoint accepts varied source labels
# ------------------------------------------------------------

timeline_test_payload = {
    "immediate_danger": False,
    "entries": [
        {
            "platform": "Discord",
            "timestamp": "2026-10-24T10:00:00+05:30",
            "message_text": "I will expose your private images",
            "direction": "incoming",
            "case_id": "synthetic-source-test",
        },
        {
            "platform": "X",
            "timestamp": "2026-10-24T10:10:00+05:30",
            "message_text": "I know where you live",
            "direction": "incoming",
            "case_id": "synthetic-source-test",
        },
        {
            "timestamp": "2026-10-24T10:20:00+05:30",
            "message_text": "Please stop contacting me",
            "direction": "unknown",
        },
    ],
}

timeline_response = api_client.post(
    "/api/analyze-timeline",
    json=timeline_test_payload,
)

assert timeline_response.status_code == 200, (
    timeline_response.get_json()
)

timeline_body = timeline_response.get_json()

returned_labels = [
    item["source_label"]
    for item in timeline_body["source_context"]["entries_in_submitted_order"]
]

assert returned_labels == ["Discord", "X", None]
assert "action_plan" in timeline_body

# The plan returned through the API must also omit the restricted service.
assert "women_helpline_181" not in {
    item["resource_id"]
    for item in timeline_body["action_plan"].get("resources", [])
}

print("Timeline with Discord, X and unspecified source: PASSED")
print("Timeline source metadata: PASSED")
print("Timeline action-plan resource filtering: PASSED")


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\nAll audit repairs passed.")

print("\nSource-label test results:")
for source_name, rule_ids in source_test_results.items():
    print(f"- {source_name}: {rule_ids}")

print("\nGeneral action-plan resources:")
for resource in fixed_resources:
    print(f"- {resource['name']}: {resource['url']}")

In [ ]:
# ============================================================
# SIGNALHER — RULE AUDIT BEFORE EVALUATION
# Read-only inspection of the active detection rules.
# ============================================================

import inspect
from dataclasses import fields, is_dataclass


# ------------------------------------------------------------
# 1. Inspect the actual incident-analysis function
# ------------------------------------------------------------

print("=" * 70)
print("1. ACTIVE INCIDENT ANALYSIS FUNCTION")
print("=" * 70)

try:
    print(inspect.getsource(analyze_incident))
except (OSError, TypeError) as exc:
    print("Could not retrieve function source:", exc)


# ------------------------------------------------------------
# 2. Find collections containing configured rule objects
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2. CONFIGURED RULE COLLECTIONS")
print("=" * 70)

rule_collections_found = 0

for collection_name, collection in list(globals().items()):

    if collection_name.startswith("__"):
        continue

    if isinstance(collection, dict):
        possible_rules = list(collection.values())
    elif isinstance(collection, (list, tuple, set)):
        possible_rules = list(collection)
    else:
        continue

    possible_rules = [
        item for item in possible_rules
        if hasattr(item, "rule_id")
    ]

    if not possible_rules:
        continue

    rule_collections_found += 1

    print(
        f"\nCollection: {collection_name} "
        f"({len(possible_rules)} rule objects)"
    )

    for rule in possible_rules:
        print("\nRule ID:", getattr(rule, "rule_id", "unknown"))

        if is_dataclass(rule):
            rule_data = {
                field.name: getattr(rule, field.name)
                for field in fields(rule)
            }
        elif hasattr(rule, "__dict__"):
            rule_data = vars(rule)
        else:
            rule_data = {"representation": repr(rule)}

        for key, value in rule_data.items():
            print(f"  {key}: {value}")


if rule_collections_found == 0:
    print(
        "No rule collection was found through simple inspection. "
        "Use the function source above to identify where its rules "
        "are stored."
    )


# ------------------------------------------------------------
# 3. Inspect the existing rule dataclass, if defined
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("3. SIGNAL RULE MODEL")
print("=" * 70)

rule_class = globals().get("SignalRule")

if rule_class is None:
    print("SignalRule class was not found in the current namespace.")
elif is_dataclass(rule_class):
    for field in fields(rule_class):
        print(f"{field.name}: {field.type}")
else:
    print(inspect.getsource(rule_class))


print("\nRule audit completed. No application settings were changed.")

In [ ]:
# ============================================================
# SIGNALHER — SYNTHETIC EVALUATION SET V1
#
# 18 labelled positive examples
# 18 labelled negative/benign examples
#  6 ambiguous examples (reported separately, excluded from metrics)
#
# These are manually authored synthetic examples, not real-world
# validation data. Labels reflect intended interpretation of each
# example for this initial baseline.
# ============================================================

from collections import defaultdict
from datetime import datetime, timezone


SYNTHETIC_EVALUATION_V1 = [
    # --------------------------------------------------------
    # A. Labelled positive examples
    # expected_rule_ids records the intended rule labels.
    # --------------------------------------------------------

    {
        "case_id": "P01",
        "group": "labelled",
        "expected_rule_ids": ["EXTORTION_THREAT"],
        "message_text": "Send me money or I'll publish the documents tonight.",
        "platform": "Discord",
    },
    {
        "case_id": "P02",
        "group": "labelled",
        "expected_rule_ids": ["EXTORTION_THREAT"],
        "message_text": "If you do not pay, I will leak the files.",
        "platform": "X",
    },
    {
        "case_id": "P03",
        "group": "labelled",
        "expected_rule_ids": ["EXTORTION_THREAT"],
        "message_text": "Pay me or else I will distribute the records.",
        # No platform field: test unspecified source.
    },
    {
        "case_id": "P04",
        "group": "labelled",
        "expected_rule_ids": ["THREAT_TO_EXPOSE_IMAGES"],
        "message_text": "I will expose your private images tonight.",
        "platform": "WhatsApp",
    },
    {
        "case_id": "P05",
        "group": "labelled",
        "expected_rule_ids": ["THREAT_TO_EXPOSE_IMAGES"],
        "message_text": "I'm going to share the videos.",
        "platform": "Instagram",
    },
    {
        "case_id": "P06",
        "group": "labelled",
        "expected_rule_ids": ["THREAT_TO_EXPOSE_IMAGES"],
        "message_text": "I'll upload your photos tomorrow.",
        "platform": "LinkedIn",
    },
    {
        "case_id": "P07",
        "group": "labelled",
        "expected_rule_ids": ["COERCIVE_IMAGE_REQUEST"],
        "message_text": "Send me your nude photos.",
        "platform": "Email",
    },
    {
        "case_id": "P08",
        "group": "labelled",
        "expected_rule_ids": ["COERCIVE_IMAGE_REQUEST"],
        "message_text": "If you love me, show me your intimate pictures.",
        "platform": "Discord",
    },
    {
        "case_id": "P09",
        "group": "labelled",
        "expected_rule_ids": ["COERCIVE_IMAGE_REQUEST"],
        "message_text": "You owe me, share private images now.",
        # No platform field.
    },
    {
        "case_id": "P10",
        "group": "labelled",
        "expected_rule_ids": ["DIRECT_THREAT_OR_STALKING_LANGUAGE"],
        "message_text": "I know where you live.",
        "platform": "X",
    },
    {
        "case_id": "P11",
        "group": "labelled",
        "expected_rule_ids": ["DIRECT_THREAT_OR_STALKING_LANGUAGE"],
        "message_text": "I will hurt you tonight.",
        "platform": "WhatsApp",
    },
    {
        "case_id": "P12",
        "group": "labelled",
        "expected_rule_ids": ["DIRECT_THREAT_OR_STALKING_LANGUAGE"],
        "message_text": "I'm waiting outside your house.",
        # No platform field.
    },
    {
        "case_id": "P13",
        "group": "labelled",
        "expected_rule_ids": ["BOUNDARY_DISREGARD"],
        "message_text": "I don't care that you said no.",
        "platform": "LinkedIn",
    },
    {
        "case_id": "P14",
        "group": "labelled",
        "expected_rule_ids": ["BOUNDARY_DISREGARD"],
        "message_text": "You said no but meet me tonight.",
        "platform": "Instagram",
    },
    {
        "case_id": "P15",
        "group": "labelled",
        "expected_rule_ids": ["BOUNDARY_DISREGARD"],
        "message_text": "You said no but share the photos.",
        "platform": "X",
    },
    {
        "case_id": "P16",
        "group": "labelled",
        "expected_rule_ids": ["IDENTITY_IMPERSONATION"],
        "message_text": "Someone is pretending to be you online.",
        "platform": "Discord",
    },
    {
        "case_id": "P17",
        "group": "labelled",
        "expected_rule_ids": ["IDENTITY_IMPERSONATION"],
        "message_text": "I created a fake profile in your name.",
        "platform": "Email",
    },
    {
        "case_id": "P18",
        "group": "labelled",
        "expected_rule_ids": ["IDENTITY_IMPERSONATION"],
        "message_text": "I made a fake account pretending to be you.",
        # No platform field.
    },

    # --------------------------------------------------------
    # B. Labelled negative / benign examples
    # Deliberately includes cases that may expose false positives.
    # --------------------------------------------------------

    {
        "case_id": "N01",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": (
            "I will share your photos with you after editing them."
        ),
        "platform": "Instagram",
    },
    {
        "case_id": "N02",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": (
            "Please send me your private photos from the holiday album."
        ),
        "platform": "WhatsApp",
    },
    {
        "case_id": "N03",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": "If you don't pay, I will call customer support.",
        "platform": "Email",
    },
    {
        "case_id": "N04",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": "I will publish the meeting notes tomorrow.",
        # No platform field.
    },
    {
        "case_id": "N05",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": "I don't care that you said no to the movie.",
        "platform": "Discord",
    },
    {
        "case_id": "N06",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": "I know where you live; I can deliver your parcel.",
        "platform": "X",
    },
    {
        "case_id": "N07",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": "I made a fake account to test my own website.",
        "platform": "LinkedIn",
    },
    {
        "case_id": "N08",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": (
            "Please don't share my private pictures without permission."
        ),
        "platform": "Email",
    },
    {
        "case_id": "N09",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": "If you love me, send me the concert tickets.",
        "platform": "WhatsApp",
    },
    {
        "case_id": "N10",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": (
            "You said no but I understand and won't ask again."
        ),
        # No platform field.
    },
    {
        "case_id": "N11",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": "I will send the files you requested.",
        "platform": "Discord",
    },
    {
        "case_id": "N12",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": (
            "The phrase 'I will expose your private images' "
            "appears in this safety guide."
        ),
        "platform": "X",
    },
    {
        "case_id": "N13",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": "Pay me by Friday; I'll send the invoice.",
        "platform": "Email",
    },
    {
        "case_id": "N14",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": "If you don't pay, I will send the receipt.",
        "platform": "LinkedIn",
    },
    {
        "case_id": "N15",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": (
            "The phrase 'I know where you live' is from a play."
        ),
        "platform": "Instagram",
    },
    {
        "case_id": "N16",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": (
            "I'd like to post your holiday photos after you approve."
        ),
        # No platform field.
    },
    {
        "case_id": "N17",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": "I created a fake profile to test a new design.",
        "platform": "Discord",
    },
    {
        "case_id": "N18",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": (
            "You can share your intimate photos with me "
            "only if you want to."
        ),
        "platform": "X",
    },

    # --------------------------------------------------------
    # C. Ambiguous / context-dependent cases.
    # These cases are measured separately and excluded from metrics.
    # expected_rule_ids=None means "not adjudicated".
    # --------------------------------------------------------

    {
        "case_id": "A01",
        "group": "ambiguous",
        "expected_rule_ids": None,
        "message_text": "The sender wrote, 'I will expose your private images'.",
        "platform": "Discord",
    },
    {
        "case_id": "A02",
        "group": "ambiguous",
        "expected_rule_ids": None,
        "message_text": "Someone sent me the words 'I know where you live'.",
        "platform": "X",
    },
    {
        "case_id": "A03",
        "group": "ambiguous",
        "expected_rule_ids": None,
        "message_text": "Could you send me your private photos for the archive?",
        "platform": "Email",
    },
    {
        "case_id": "A04",
        "group": "ambiguous",
        "expected_rule_ids": None,
        "message_text": "If you don't pay, I will expose everything.",
        # No platform field.
    },
    {
        "case_id": "A05",
        "group": "ambiguous",
        "expected_rule_ids": None,
        "message_text": "The report says someone is using your identity.",
        "platform": "LinkedIn",
    },
    {
        "case_id": "A06",
        "group": "ambiguous",
        "expected_rule_ids": None,
        "message_text": (
            "The sender wrote, 'You said no but send the photos anyway'."
        ),
        "platform": "WhatsApp",
    },
]


# ============================================================
# RUN THE EXISTING RULES VIA THE FLASK TEST CLIENT
# ============================================================

evaluation_client = signalher_app.test_client()
RULE_IDS = [rule.rule_id for rule in SIGNAL_RULES]

evaluation_results = []

for case in SYNTHETIC_EVALUATION_V1:
    payload = {
        "message_text": case["message_text"],
    }

    # Platform/source is optional.
    # Unknown labels should not prevent message analysis.
    if "platform" in case:
        payload["platform"] = case["platform"]

    response = evaluation_client.post(
        "/api/analyze-incident",
        json=payload,
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"{case['case_id']} failed with HTTP "
            f"{response.status_code}: {response.get_json()}"
        )

    body = response.get_json()

    predicted_rule_ids = {
        finding["rule_id"]
        for finding in body["analysis"].get("findings", [])
    }

    expected = case["expected_rule_ids"]

    evaluation_results.append({
        "case_id": case["case_id"],
        "group": case["group"],
        "message_text": case["message_text"],
        "expected_rule_ids": (
            None if expected is None else set(expected)
        ),
        "predicted_rule_ids": predicted_rule_ids,
        "status_code": response.status_code,
        "source_context": body.get("source_context"),
    })


# ============================================================
# CALCULATE METRICS ON LABELLED EXAMPLES ONLY
# ============================================================

labelled_results = [
    result
    for result in evaluation_results
    if result["expected_rule_ids"] is not None
]

ambiguous_results = [
    result
    for result in evaluation_results
    if result["expected_rule_ids"] is None
]

rule_confusion = {
    rule_id: {
        "TP": 0,
        "FP": 0,
        "FN": 0,
        "TN": 0,
    }
    for rule_id in RULE_IDS
}

exact_match_count = 0
false_positive_cases = []
false_negative_cases = []

for result in labelled_results:
    expected = result["expected_rule_ids"]
    predicted = result["predicted_rule_ids"]

    if expected == predicted:
        exact_match_count += 1

    extra_rules = predicted - expected
    missed_rules = expected - predicted

    if extra_rules:
        false_positive_cases.append({
            "case_id": result["case_id"],
            "unexpected_rules": sorted(extra_rules),
            "message_text": result["message_text"],
        })

    if missed_rules:
        false_negative_cases.append({
            "case_id": result["case_id"],
            "missed_rules": sorted(missed_rules),
            "message_text": result["message_text"],
        })

    for rule_id in RULE_IDS:
        expected_positive = rule_id in expected
        predicted_positive = rule_id in predicted

        if expected_positive and predicted_positive:
            rule_confusion[rule_id]["TP"] += 1
        elif not expected_positive and predicted_positive:
            rule_confusion[rule_id]["FP"] += 1
        elif expected_positive and not predicted_positive:
            rule_confusion[rule_id]["FN"] += 1
        else:
            rule_confusion[rule_id]["TN"] += 1


def _safe_metric(numerator, denominator):
    return numerator / denominator if denominator else None


def _format_metric(value):
    return "N/A" if value is None else f"{100 * value:.1f}%"


print("=" * 72)
print("SIGNALHER — SYNTHETIC RULE EVALUATION V1")
print("=" * 72)

print("Total test cases:", len(evaluation_results))
print("Labelled cases used for metrics:", len(labelled_results))
print("Ambiguous cases excluded from metrics:", len(ambiguous_results))
print("API requests returning HTTP 200:", len(evaluation_results))

print("\nPER-RULE METRICS")
print("-" * 72)
print(
    f"{'Rule ID':42} {'Prec.':>9} {'Recall':>9} "
    f"{'F1':>9} {'TP':>4} {'FP':>4} {'FN':>4}"
)

micro_tp = micro_fp = micro_fn = micro_tn = 0

for rule_id, counts in rule_confusion.items():
    tp, fp, fn, tn = (
        counts["TP"],
        counts["FP"],
        counts["FN"],
        counts["TN"],
    )

    precision = _safe_metric(tp, tp + fp)
    recall = _safe_metric(tp, tp + fn)

    if precision is not None and recall is not None:
        f1 = _safe_metric(
            2 * precision * recall,
            precision + recall,
        )
    else:
        f1 = None

    micro_tp += tp
    micro_fp += fp
    micro_fn += fn
    micro_tn += tn

    print(
        f"{rule_id:42} "
        f"{_format_metric(precision):>9} "
        f"{_format_metric(recall):>9} "
        f"{_format_metric(f1):>9} "
        f"{tp:>4} {fp:>4} {fn:>4}"
    )


micro_precision = _safe_metric(
    micro_tp, micro_tp + micro_fp
)
micro_recall = _safe_metric(
    micro_tp, micro_tp + micro_fn
)

if micro_precision is not None and micro_recall is not None:
    micro_f1 = _safe_metric(
        2 * micro_precision * micro_recall,
        micro_precision + micro_recall,
    )
else:
    micro_f1 = None

exact_match_rate = _safe_metric(
    exact_match_count, len(labelled_results)
)

print("\nOVERALL LABELLED-CASE SUMMARY")
print("-" * 72)
print("Exact set match rate:", _format_metric(exact_match_rate))
print("Micro precision:", _format_metric(micro_precision))
print("Micro recall:", _format_metric(micro_recall))
print("Micro F1:", _format_metric(micro_f1))


# ============================================================
# SHOW MISCLASSIFICATIONS FOR RULE REVIEW
# ============================================================

print("\nFALSE-POSITIVE CASES")
print("-" * 72)

if not false_positive_cases:
    print("None in this synthetic test set.")
else:
    for item in false_positive_cases:
        print(
            f"{item['case_id']} | "
            f"Unexpected: {item['unexpected_rules']} | "
            f"Message: {item['message_text']}"
        )


print("\nFALSE-NEGATIVE CASES")
print("-" * 72)

if not false_negative_cases:
    print("None in this synthetic test set.")
else:
    for item in false_negative_cases:
        print(
            f"{item['case_id']} | "
            f"Missed: {item['missed_rules']} | "
            f"Message: {item['message_text']}"
        )


print("\nAMBIGUOUS CASES — REPORTED SEPARATELY")
print("-" * 72)

for result in ambiguous_results:
    print(
        f"{result['case_id']} | "
        f"Detected: {sorted(result['predicted_rule_ids'])} | "
        f"Message: {result['message_text']}"
    )


print("\nEVALUATION CAVEAT")
print(
    "These metrics describe this manually labelled synthetic test set only. "
    "They are not estimates of real-world performance, and the labels "
    "require review before the set is used for model training."
)

In [ ]:
# ============================================================
# SIGNALHER — EXPERIMENTAL RULES V2
#
# This is a shadow evaluation only.
# It does NOT change SIGNAL_RULES, the existing analyzer,
# Flask routes, or the active backend.
# ============================================================

import copy
import re


# ------------------------------------------------------------
# 1. Prepare an audited copy of the evaluation set
# ------------------------------------------------------------

SYNTHETIC_EVALUATION_V1_1 = copy.deepcopy(
    SYNTHETIC_EVALUATION_V1
)

# These messages lack enough context to label confidently as
# harmful or benign. Report them separately instead.
AMBIGUOUS_CASE_IDS = {"P05", "P06", "P07"}

for case in SYNTHETIC_EVALUATION_V1_1:
    if case["case_id"] in AMBIGUOUS_CASE_IDS:
        case["group"] = "ambiguous"
        case["expected_rule_ids"] = None
        case["adjudication_note"] = (
            "Insufficient context to establish the intended rule label."
        )

# The boundary-disregard rule targets refusal being dismissed,
# not proof of criminal abuse or serious harm.
for case in SYNTHETIC_EVALUATION_V1_1:
    if case["case_id"] == "N05":
        case["expected_rule_ids"] = ["BOUNDARY_DISREGARD"]
        case["adjudication_note"] = (
            "The text explicitly dismisses a stated refusal; "
            "severity and wider context remain separate questions."
        )


# ------------------------------------------------------------
# 2. Draft revised patterns, without replacing the live rules
# ------------------------------------------------------------

V2_PATTERN_OVERRIDES = {
    "EXTORTION_THREAT": (
        (
            r"\b(?:pay me|send me money|send money|"
            r"transfer (?:me )?money|give me money)\b"
            r".{0,100}"
            r"\b(?:or else|otherwise|or i'll|or i will|i'll|i will)\b"
            r".{0,100}"
            r"\b(?:leak|expose|post|publish|share|distribute)\b"
        ),
        (
            r"\bif you (?:don't|do not) pay\b"
            r".{0,100}"
            r"\b(?:leak|expose|post|publish|share|distribute)\b"
        ),
    ),

    "COERCIVE_IMAGE_REQUEST": (
        # A request for intimate imagery is kept as a possible
        # indicator, but ordinary "private photos" requests
        # no longer qualify by themselves.
        (
            r"\b(?:send|sending|give|giving|show|showing|share|sharing)\b"
            r"\s+(?:me\s+)?(?:your\s+)?"
            r"(?:nude|naked|intimate)\s+"
            r"(?:photos?|pictures?|images?|videos?|pics?)\b"
        ),
        # This pattern requires explicit pressure language.
        (
            r"\b(?:prove you love me|if you love me|you owe me)\b"
            r".{0,80}"
            r"\b(?:send|sending|show|showing|share|sharing|give|giving)\b"
            r".{0,50}"
            r"\b(?:nude|naked|intimate|private)\s+"
            r"(?:photos?|pictures?|images?|videos?|pics?)\b"
        ),
    ),
}


def v2_context_suppression(rule_id, text):
    """
    Return a reason when an explicitly recognisable context
    should suppress a rule match.

    These are conservative experimental heuristics, not a
    general-purpose natural-language understanding system.
    """
    text_lower = text.casefold()

    # Explicitly metalinguistic or educational context:
    # for example, discussing a phrase in a safety guide or play.
    meta_context = re.search(
        r"\b(?:the phrase|the wording|the sentence|the expression)\b"
        r".{0,120}"
        r"\b(?:appears|is|was|comes from)\b"
        r".{0,80}"
        r"\b(?:safety guide|guide|article|play|script|"
        r"training material|example|fiction)\b",
        text_lower,
    )

    if meta_context:
        return "explicit educational or fictional phrase context"

    # Sharing one's photos with the recipient is different from
    # threatening to expose private imagery.
    if rule_id == "THREAT_TO_EXPOSE_IMAGES":
        with_recipient = re.search(
            r"\b(?:share|send|show|upload|post|publish)\b"
            r"\s+(?:your|the)\s+"
            r"(?:(?:private|intimate|personal)\s+)?"
            r"(?:photos?|pictures?|images?|videos?|nudes?|pics?)"
            r"\s+with you\b",
            text_lower,
        )

        if with_recipient:
            return "the message describes sharing the material with the recipient"

    # Explicit consent language can reduce a pressure-related
    # match where no contradictory pressure indicator is present.
    if rule_id == "COERCIVE_IMAGE_REQUEST":
        explicit_consent = re.search(
            r"\b(?:only if you want to|if you want to|"
            r"if you feel comfortable|no pressure|"
            r"you can say no|it's okay to say no|"
            r"it is okay to say no|with your permission)\b",
            text_lower,
        )

        explicit_pressure = re.search(
            r"\b(?:you owe me|prove you love me|if you love me|"
            r"or else|i will expose|i'll expose|"
            r"you have to|you must)\b",
            text_lower,
        )

        if explicit_consent and not explicit_pressure:
            return "explicit voluntary/consent language without a pressure cue"

    # The bare phrase "I know where you live" can be benign in
    # a clear parcel-delivery context when no stronger threat
    # phrase is present.
    if rule_id == "DIRECT_THREAT_OR_STALKING_LANGUAGE":
        delivery_context = re.search(
            r"\b(?:deliver your parcel|deliver your package|"
            r"parcel delivery|package delivery|"
            r"drop off your order)\b",
            text_lower,
        )

        stronger_threat = re.search(
            r"\b(?:hurt you|find you|come to your home|"
            r"waiting outside your|outside your house)\b",
            text_lower,
        )

        if (
            "i know where you live" in text_lower
            and delivery_context
            and not stronger_threat
        ):
            return "clear delivery context with no additional threat phrase"

    return None


# ------------------------------------------------------------
# 3. Experimental V2 analyzer
# ------------------------------------------------------------

def analyze_incident_v2(incident):
    """
    Evaluate experimental patterns and context filters.
    This function does not alter the active analyzer.
    """
    if not isinstance(incident, Incident):
        raise TypeError("incident must be an Incident object.")

    text = incident.message_text
    findings = []
    suppressed = []

    for rule in SIGNAL_RULES:
        patterns = V2_PATTERN_OVERRIDES.get(
            rule.rule_id,
            rule.patterns,
        )

        reason = v2_context_suppression(rule.rule_id, text)

        if reason:
            # Record suppression for error analysis instead of
            # silently hiding the experimental decision.
            if any(
                re.search(pattern, text, flags=re.IGNORECASE)
                for pattern in patterns
            ):
                suppressed.append({
                    "rule_id": rule.rule_id,
                    "reason": reason,
                })
                continue

        for pattern in patterns:
            match = re.search(pattern, text, flags=re.IGNORECASE)

            if match:
                findings.append({
                    "rule_id": rule.rule_id,
                    "label": rule.label,
                    "category": rule.category,
                    "evidence": match.group(0),
                    "explanation": rule.explanation,
                    "suggested_action": rule.suggested_action,
                })
                break

    return {
        "incident_id": incident.incident_id,
        "platform": incident.platform.value,
        "occurred_at": incident.occurred_at.isoformat(),
        "status": (
            "potential_warning_indicators_found"
            if findings
            else "no_configured_pattern_matched"
        ),
        "findings": findings,
        "message": (
            "Review the highlighted wording and wider context. "
            "Indicators are not proof of abuse."
            if findings
            else
            "No configured V2 pattern matched. This is not a "
            "determination that the message is safe."
        ),
        "experimental_suppressions": suppressed,
    }


# ------------------------------------------------------------
# 4. Run baseline and V2 over the same audited cases
# ------------------------------------------------------------

def _predict_rule_ids(analyzer, case):
    payload = {"message_text": case["message_text"]}

    if "platform" in case:
        payload["platform"] = case["platform"]

    incident = _build_incident_from_payload(payload)
    result = analyzer(incident)

    return {
        finding["rule_id"]
        for finding in result.get("findings", [])
    }


def _calculate_suite_metrics(suite, analyzer):
    labelled = [
        case
        for case in suite
        if case["expected_rule_ids"] is not None
    ]

    confusion = {
        rule.rule_id: {"TP": 0, "FP": 0, "FN": 0, "TN": 0}
        for rule in SIGNAL_RULES
    }

    exact_matches = 0
    false_positives = []
    false_negatives = []
    differences = []
    suppressions = []

    for case in suite:
        baseline_ids = _predict_rule_ids(analyze_incident, case)
        predicted_ids = _predict_rule_ids(analyzer, case)

        expected = case["expected_rule_ids"]

        if predicted_ids != baseline_ids:
            differences.append({
                "case_id": case["case_id"],
                "baseline": sorted(baseline_ids),
                "experimental": sorted(predicted_ids),
                "expected": (
                    None if expected is None else sorted(expected)
                ),
            })

        # Capture explicit suppressions if this is V2.
        if analyzer is analyze_incident_v2:
            payload = {"message_text": case["message_text"]}

            if "platform" in case:
                payload["platform"] = case["platform"]

            incident = _build_incident_from_payload(payload)
            v2_result = analyzer(incident)

            if v2_result["experimental_suppressions"]:
                suppressions.append({
                    "case_id": case["case_id"],
                    "suppressions": v2_result["experimental_suppressions"],
                })

        if expected is None:
            continue

        expected = set(expected)

        if expected == predicted_ids:
            exact_matches += 1

        if predicted_ids - expected:
            false_positives.append({
                "case_id": case["case_id"],
                "unexpected_rules": sorted(predicted_ids - expected),
                "message_text": case["message_text"],
            })

        if expected - predicted_ids:
            false_negatives.append({
                "case_id": case["case_id"],
                "missed_rules": sorted(expected - predicted_ids),
                "message_text": case["message_text"],
            })

        for rule_id in confusion:
            actual_positive = rule_id in expected
            predicted_positive = rule_id in predicted_ids

            if actual_positive and predicted_positive:
                confusion[rule_id]["TP"] += 1
            elif predicted_positive and not actual_positive:
                confusion[rule_id]["FP"] += 1
            elif actual_positive and not predicted_positive:
                confusion[rule_id]["FN"] += 1
            else:
                confusion[rule_id]["TN"] += 1

    total_tp = sum(item["TP"] for item in confusion.values())
    total_fp = sum(item["FP"] for item in confusion.values())
    total_fn = sum(item["FN"] for item in confusion.values())

    precision = (
        total_tp / (total_tp + total_fp)
        if total_tp + total_fp else None
    )

    recall = (
        total_tp / (total_tp + total_fn)
        if total_tp + total_fn else None
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision is not None
        and recall is not None
        and precision + recall
        else None
    )

    exact_match_rate = (
        exact_matches / len(labelled)
        if labelled else None
    )

    return {
        "labelled_count": len(labelled),
        "ambiguous_count": len(suite) - len(labelled),
        "confusion": confusion,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "exact_match_rate": exact_match_rate,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "differences": differences,
        "suppressions": suppressions,
    }


baseline_v2_audit = _calculate_suite_metrics(
    SYNTHETIC_EVALUATION_V1_1,
    analyze_incident,
)

experimental_v2_audit = _calculate_suite_metrics(
    SYNTHETIC_EVALUATION_V1_1,
    analyze_incident_v2,
)


# ------------------------------------------------------------
# 5. Print a side-by-side development-set comparison
# ------------------------------------------------------------

def _percent(value):
    return "N/A" if value is None else f"{100 * value:.1f}%"


print("=" * 72)
print("SIGNALHER — V1 vs EXPERIMENTAL V2")
print("=" * 72)

print(
    "Labelled cases:",
    baseline_v2_audit["labelled_count"],
)
print(
    "Ambiguous cases excluded:",
    baseline_v2_audit["ambiguous_count"],
)

print("\nDEVELOPMENT-SET METRICS")
print("-" * 72)
print(
    f"{'Metric':24} {'V1 baseline':>15} {'V2 experiment':>18}"
)

for key, label in [
    ("precision", "Micro precision"),
    ("recall", "Micro recall"),
    ("f1", "Micro F1"),
    ("exact_match_rate", "Exact set match"),
]:
    print(
        f"{label:24} "
        f"{_percent(baseline_v2_audit[key]):>15} "
        f"{_percent(experimental_v2_audit[key]):>18}"
    )


print("\nV2 RULE-LEVEL METRICS")
print("-" * 72)
print(
    f"{'Rule ID':42} {'Prec.':>9} {'Recall':>9} {'F1':>9}"
)

for rule_id, counts in experimental_v2_audit["confusion"].items():
    tp, fp, fn = counts["TP"], counts["FP"], counts["FN"]

    precision = tp / (tp + fp) if tp + fp else None
    recall = tp / (tp + fn) if tp + fn else None

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision is not None
        and recall is not None
        and precision + recall
        else None
    )

    print(
        f"{rule_id:42} "
        f"{_percent(precision):>9} "
        f"{_percent(recall):>9} "
        f"{_percent(f1):>9}"
    )


print("\nV2 FALSE POSITIVES ON AUDITED CASES")
print("-" * 72)

for item in experimental_v2_audit["false_positives"]:
    print(
        f"{item['case_id']} | {item['unexpected_rules']} | "
        f"{item['message_text']}"
    )

if not experimental_v2_audit["false_positives"]:
    print("None in this small audited development set.")


print("\nV2 FALSE NEGATIVES ON AUDITED CASES")
print("-" * 72)

for item in experimental_v2_audit["false_negatives"]:
    print(
        f"{item['case_id']} | {item['missed_rules']} | "
        f"{item['message_text']}"
    )

if not experimental_v2_audit["false_negatives"]:
    print("None in this small audited development set.")


print("\nCASES WHERE V2 CHANGES THE BASELINE")
print("-" * 72)

for item in experimental_v2_audit["differences"]:
    print(
        f"{item['case_id']} | "
        f"V1={item['baseline']} | "
        f"V2={item['experimental']} | "
        f"Expected={item['expected']}"
    )

print("\nEXPLICIT CONTEXT SUPPRESSIONS")
print("-" * 72)

for item in experimental_v2_audit["suppressions"]:
    print(item)

print("\nIMPORTANT")
print(
    "These are development-set results only. V2 was designed after "
    "reviewing V1 errors, so improved scores here do not establish "
    "generalisation. Do not replace the active backend rules yet. "
    "The next gate is a separate challenge set with new paraphrases, "
    "negation, benign contexts, and unseen wording."
)

In [ ]:
# ============================================================
# SIGNALHER — CHALLENGE SET V1
# 12 intended-positive cases
# 16 benign/negative cases
#  8 context-dependent cases, excluded from metrics
#
# This is a manually authored challenge set, not a blind or
# externally validated benchmark.
# ============================================================

CHALLENGE_SET_V1 = [
    # --------------------------------------------------------
    # A. Intended-positive cases
    # Each rule has one pattern-aligned and one paraphrased case.
    # --------------------------------------------------------

    {
        "case_id": "C01",
        "group": "labelled",
        "expected_rule_ids": ["EXTORTION_THREAT"],
        "message_text": (
            "Transfer me money or I'll publish the confidential records."
        ),
        "platform": "Discord",
    },
    {
        "case_id": "C02",
        "group": "labelled",
        "expected_rule_ids": ["EXTORTION_THREAT"],
        "message_text": (
            "Pay the amount by tonight, or I will release the records."
        ),
        "platform": "X",
    },
    {
        "case_id": "C03",
        "group": "labelled",
        "expected_rule_ids": ["THREAT_TO_EXPOSE_IMAGES"],
        "message_text": "I'm going to publish your intimate pictures.",
        "platform": "Email",
    },
    {
        "case_id": "C04",
        "group": "labelled",
        "expected_rule_ids": ["THREAT_TO_EXPOSE_IMAGES"],
        "message_text": "I will circulate your private photos.",
        "platform": "LinkedIn",
    },
    {
        "case_id": "C05",
        "group": "labelled",
        "expected_rule_ids": ["COERCIVE_IMAGE_REQUEST"],
        "message_text": "You owe me. Send your intimate images.",
        "platform": "Other",
    },
    {
        "case_id": "C06",
        "group": "labelled",
        "expected_rule_ids": ["COERCIVE_IMAGE_REQUEST"],
        "message_text": (
            "You must provide intimate imagery to prove you care about me."
        ),
        # Source intentionally unspecified.
    },
    {
        "case_id": "C07",
        "group": "labelled",
        "expected_rule_ids": ["DIRECT_THREAT_OR_STALKING_LANGUAGE"],
        "message_text": "I'll find you after your shift.",
        "platform": "Discord",
    },
    {
        "case_id": "C08",
        "group": "labelled",
        "expected_rule_ids": ["DIRECT_THREAT_OR_STALKING_LANGUAGE"],
        "message_text": "I'm coming to hurt you tonight.",
        "platform": "X",
    },
    {
        "case_id": "C09",
        "group": "labelled",
        "expected_rule_ids": ["BOUNDARY_DISREGARD"],
        "message_text": "You said no but come to the meeting anyway.",
        "platform": "Email",
    },
    {
        "case_id": "C10",
        "group": "labelled",
        "expected_rule_ids": ["BOUNDARY_DISREGARD"],
        "message_text": (
            "I heard your refusal, but I'm going to keep pushing you to meet me."
        ),
        # Source intentionally unspecified.
    },
    {
        "case_id": "C11",
        "group": "labelled",
        "expected_rule_ids": ["IDENTITY_IMPERSONATION"],
        "message_text": "I made a fake account pretending to be you.",
        "platform": "Other",
    },
    {
        "case_id": "C12",
        "group": "labelled",
        "expected_rule_ids": ["IDENTITY_IMPERSONATION"],
        "message_text": "Someone has opened an account impersonating you.",
        "platform": "LinkedIn",
    },

    # --------------------------------------------------------
    # B. Intended-negative cases
    # These include benign context, negation, and quotations.
    # --------------------------------------------------------

    {
        "case_id": "N01",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": (
            "Transfer me money for the concert ticket, "
            "and I'll share the receipt."
        ),
        "platform": "Email",
    },
    {
        "case_id": "N02",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": "If you don't pay, I will send your receipt.",
        "platform": "X",
    },
    {
        "case_id": "N03",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": "I will not expose your private images.",
        "platform": "Discord",
    },
    {
        "case_id": "N04",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": (
            "I'm going to share your private photos with you "
            "in the shared album."
        ),
        "platform": "Other",
    },
    {
        "case_id": "N05",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": (
            "In this training slide, consider this example: "
            "I will expose your private images."
        ),
        "platform": "LinkedIn",
    },
    {
        "case_id": "N06",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": (
            "You don't need to send intimate photos; your choice matters."
        ),
        "platform": "WhatsApp",
    },
    {
        "case_id": "N07",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": (
            "Only share intimate images if you genuinely want to; "
            "there is no pressure."
        ),
        "platform": "Discord",
    },
    {
        "case_id": "N08",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": "I'm not waiting outside your house; I'm at home.",
        "platform": "X",
    },
    {
        "case_id": "N09",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": "Our play includes the line 'I know where you live'.",
        # Source intentionally unspecified.
    },
    {
        "case_id": "N10",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": "You said no but I won't ask again.",
        "platform": "Email",
    },
    {
        "case_id": "N11",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": (
            "You said no, and I respect that. We won't meet."
        ),
        "platform": "Instagram",
    },
    {
        "case_id": "N12",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": (
            "I created a fake profile for my art portfolio; "
            "it doesn't imitate anyone."
        ),
        "platform": "Other",
    },
    {
        "case_id": "N13",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": (
            "The phrase 'pretending to be you' appears in a training manual."
        ),
        # Source intentionally unspecified.
    },
    {
        "case_id": "N14",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": (
            "I made a fake account for testing; it does not impersonate anyone."
        ),
        "platform": "Discord",
    },
    {
        "case_id": "N15",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": (
            "If you don't pay, I will send you the receipt "
            "instead of contacting support."
        ),
        "platform": "X",
    },
    {
        "case_id": "N16",
        "group": "labelled",
        "expected_rule_ids": [],
        "message_text": "I will publish the public report tomorrow.",
        "platform": "Email",
    },

    # --------------------------------------------------------
    # C. Ambiguous/context-dependent cases
    # Not counted as positive or negative for the metrics.
    # --------------------------------------------------------

    {
        "case_id": "A01",
        "group": "ambiguous",
        "expected_rule_ids": None,
        "message_text": "Pay me or I'll reveal what happened.",
        "platform": "Discord",
    },
    {
        "case_id": "A02",
        "group": "ambiguous",
        "expected_rule_ids": None,
        "message_text": "Send me your private photos.",
        "platform": "X",
    },
    {
        "case_id": "A03",
        "group": "ambiguous",
        "expected_rule_ids": None,
        "message_text": "I will publish the pictures.",
        # Source intentionally unspecified.
    },
    {
        "case_id": "A04",
        "group": "ambiguous",
        "expected_rule_ids": None,
        "message_text": "I know where you live.",
        "platform": "Email",
    },
    {
        "case_id": "A05",
        "group": "ambiguous",
        "expected_rule_ids": None,
        "message_text": "Someone is using your identity online.",
        "platform": "LinkedIn",
    },
    {
        "case_id": "A06",
        "group": "ambiguous",
        "expected_rule_ids": None,
        "message_text": "You said no but...",
        "platform": "Other",
    },
    {
        "case_id": "A07",
        "group": "ambiguous",
        "expected_rule_ids": None,
        "message_text": "You will regret turning me down.",
        "platform": "Instagram",
    },
    {
        "case_id": "A08",
        "group": "ambiguous",
        "expected_rule_ids": None,
        "message_text": (
            "If you don't send the files, I'll make you regret it."
        ),
        # Source intentionally unspecified.
    },
]


# ============================================================
# RUN BOTH ANALYZERS WITHOUT CHANGING THE ACTIVE BACKEND
# ============================================================

def _challenge_predictions(analyzer, case):
    payload = {"message_text": case["message_text"]}

    # Source is optional and may vary.
    if "platform" in case:
        payload["platform"] = case["platform"]

    incident = _build_incident_from_payload(payload)
    output = analyzer(incident)

    return {
        finding["rule_id"]
        for finding in output.get("findings", [])
    }


def _challenge_evaluate(cases, analyzer):
    labelled = [
        case for case in cases
        if case["expected_rule_ids"] is not None
    ]
    ambiguous = [
        case for case in cases
        if case["expected_rule_ids"] is None
    ]

    per_rule = {
        rule.rule_id: {"TP": 0, "FP": 0, "FN": 0, "TN": 0}
        for rule in SIGNAL_RULES
    }

    false_positives = []
    false_negatives = []
    ambiguous_predictions = []
    exact_matches = 0

    micro_tp = micro_fp = micro_fn = 0

    for case in cases:
        predicted = _challenge_predictions(analyzer, case)
        expected_raw = case["expected_rule_ids"]

        if expected_raw is None:
            ambiguous_predictions.append({
                "case_id": case["case_id"],
                "detected_rules": sorted(predicted),
                "message_text": case["message_text"],
            })
            continue

        expected = set(expected_raw)

        if predicted == expected:
            exact_matches += 1

        extra = predicted - expected
        missed = expected - predicted

        if extra:
            false_positives.append({
                "case_id": case["case_id"],
                "unexpected_rules": sorted(extra),
                "message_text": case["message_text"],
            })

        if missed:
            false_negatives.append({
                "case_id": case["case_id"],
                "missed_rules": sorted(missed),
                "message_text": case["message_text"],
            })

        for rule_id in per_rule:
            actual_positive = rule_id in expected
            predicted_positive = rule_id in predicted

            if actual_positive and predicted_positive:
                per_rule[rule_id]["TP"] += 1
                micro_tp += 1
            elif predicted_positive and not actual_positive:
                per_rule[rule_id]["FP"] += 1
                micro_fp += 1
            elif actual_positive and not predicted_positive:
                per_rule[rule_id]["FN"] += 1
                micro_fn += 1
            else:
                per_rule[rule_id]["TN"] += 1

    precision = (
        micro_tp / (micro_tp + micro_fp)
        if micro_tp + micro_fp else None
    )
    recall = (
        micro_tp / (micro_tp + micro_fn)
        if micro_tp + micro_fn else None
    )
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision is not None
        and recall is not None
        and precision + recall
        else None
    )

    return {
        "labelled_count": len(labelled),
        "ambiguous_count": len(ambiguous),
        "per_rule": per_rule,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "exact_match_rate": exact_matches / len(labelled),
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "ambiguous_predictions": ambiguous_predictions,
    }


challenge_v1_results = _challenge_evaluate(
    CHALLENGE_SET_V1,
    analyze_incident,
)

challenge_v2_results = _challenge_evaluate(
    CHALLENGE_SET_V1,
    analyze_incident_v2,
)


# ============================================================
# REPORT
# ============================================================

def _challenge_percent(value):
    return "N/A" if value is None else f"{100 * value:.1f}%"


print("=" * 72)
print("SIGNALHER — HELD-OUT CHALLENGE RESULTS")
print("=" * 72)

print("Total cases:", len(CHALLENGE_SET_V1))
print("Labelled cases:", challenge_v2_results["labelled_count"])
print("Ambiguous cases excluded:", challenge_v2_results["ambiguous_count"])

print("\nOVERALL COMPARISON")
print("-" * 72)
print(f"{'Metric':24} {'V1 baseline':>15} {'V2 experiment':>18}")

for key, label in [
    ("precision", "Micro precision"),
    ("recall", "Micro recall"),
    ("f1", "Micro F1"),
    ("exact_match_rate", "Exact set match"),
]:
    print(
        f"{label:24} "
        f"{_challenge_percent(challenge_v1_results[key]):>15} "
        f"{_challenge_percent(challenge_v2_results[key]):>18}"
    )


print("\nV2 PER-RULE METRICS")
print("-" * 72)
print(
    f"{'Rule ID':42} {'Precision':>10} "
    f"{'Recall':>10} {'F1':>10} {'TP':>4} {'FP':>4} {'FN':>4}"
)

for rule_id, counts in challenge_v2_results["per_rule"].items():
    tp = counts["TP"]
    fp = counts["FP"]
    fn = counts["FN"]

    precision = tp / (tp + fp) if tp + fp else None
    recall = tp / (tp + fn) if tp + fn else None
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision is not None
        and recall is not None
        and precision + recall
        else None
    )

    print(
        f"{rule_id:42} "
        f"{_challenge_percent(precision):>10} "
        f"{_challenge_percent(recall):>10} "
        f"{_challenge_percent(f1):>10} "
        f"{tp:>4} {fp:>4} {fn:>4}"
    )


print("\nV2 FALSE POSITIVES")
print("-" * 72)

for item in challenge_v2_results["false_positives"]:
    print(
        f"{item['case_id']} | {item['unexpected_rules']} | "
        f"{item['message_text']}"
    )

if not challenge_v2_results["false_positives"]:
    print("None on this challenge set.")


print("\nV2 FALSE NEGATIVES")
print("-" * 72)

for item in challenge_v2_results["false_negatives"]:
    print(
        f"{item['case_id']} | {item['missed_rules']} | "
        f"{item['message_text']}"
    )

if not challenge_v2_results["false_negatives"]:
    print("None on this challenge set.")


print("\nAMBIGUOUS CASES — NOT SCORED AS POSITIVE OR NEGATIVE")
print("-" * 72)

for item in challenge_v2_results["ambiguous_predictions"]:
    print(
        f"{item['case_id']} | Detected: {item['detected_rules']} | "
        f"{item['message_text']}"
    )


print("\nINTERPRETATION")
print(
    "This is an illustrative, manually authored synthetic challenge set. "
    "It is not real-world validation, and its labels are not independent "
    "of the developers' assumptions. Use the errors to identify coverage "
    "gaps, not to claim production performance."
)

print("\nActive SIGNAL_RULES were not modified.")

In [ ]:
# ============================================================
# SIGNALHER — EXPERIMENTAL RULES V3
#
# Shadow evaluation only.
# Does NOT modify SIGNAL_RULES, analyze_incident, or Flask routes.
# Uses the observed challenge set for development, not final validation.
# ============================================================

import re


# ------------------------------------------------------------
# 1. Revised patterns
# ------------------------------------------------------------

V3_PATTERN_OVERRIDES = {
    "EXTORTION_THREAT": (
        # Require an explicit threat connector rather than treating
        # any "I'll send/share" after a payment request as extortion.
        (
            r"\b(?:pay(?:\s+me|\s+the amount|\s+the money)?|"
            r"send me money|send money|transfer (?:me )?money|"
            r"give me money)\b"
            r".{0,100}"
            r"\b(?:or else|otherwise|or i'll|or i will)\b"
            r".{0,100}"
            r"\b(?:leak|expose|post|publish|share|send|"
            r"distribute|release|circulate)\b"
        ),
        # Also handle "if you don't pay..." with explicit release
        # of sensitive content.
        (
            r"\bif you (?:don't|do not) pay\b"
            r".{0,100}"
            r"\b(?:leak|expose|publish|distribute|release)\b"
            r".{0,60}"
            r"\b(?:records?|documents?|files?|data|photos?|pictures?|"
            r"images?|videos?|information|secrets?)\b"
        ),
    ),

    "THREAT_TO_EXPOSE_IMAGES": (
        (
            r"\b(?:i will|i'll|i am going to|i'm going to|"
            r"i am about to|i'm about to)\s+"
            r"(?:post|publish|share|send|leak|upload|expose|"
            r"distribute|circulate)\s+"
            r"(?:(?:your|the)\s+)?"
            r"(?:(?:private|intimate|personal|confidential)\s+)?"
            r"(?:photos?|pictures?|images?|videos?|nudes?|pics?)\b"
        ),
    ),

    "COERCIVE_IMAGE_REQUEST": (
        # Direct requests for nude, naked, or intimate imagery.
        (
            r"\b(?:send|sending|give|giving|show|showing|"
            r"share|sharing|provide|providing)\s+"
            r"(?:me\s+)?(?:your\s+)?"
            r"(?:nude|naked|intimate)\s+"
            r"(?:photos?|pictures?|images?|videos?|pics?)\b"
        ),
        # Explicit pressure plus a request for private/intimate material.
        (
            r"\b(?:prove you love me|if you love me|you owe me|"
            r"you must|you have to|you need to)\b"
            r".{0,100}"
            r"\b(?:send|sending|show|showing|share|sharing|"
            r"give|giving|provide|providing)\b"
            r".{0,60}"
            r"\b(?:nude|naked|intimate|private)\s+"
            r"(?:photos?|pictures?|images?|videos?|pics?|imagery)\b"
        ),
    ),

    "DIRECT_THREAT_OR_STALKING_LANGUAGE": (
        r"\bi know where you live\b",
        (
            r"\b(?:i will|i'll|i am going to|i'm going to|"
            r"i am coming to|i'm coming to)\s+"
            r"(?:find you|hurt you|come to your "
            r"(?:home|house|workplace))\b"
        ),
        (
            r"\b(?:i am|i'm)\s+(?:outside|waiting outside)\s+"
            r"(?:your|the)\s+(?:home|house|door)\b"
        ),
    ),

    "BOUNDARY_DISREGARD": (
        r"\bi don't care that you said no\b",
        r"\byou said no but\b.{0,80}\b(?:send|share|show|meet|come)\b",
        (
            r"\bi heard (?:your )?(?:refusal|no)\b"
            r".{0,100}"
            r"\b(?:keep|continue to|won't stop|will keep)\b"
            r".{0,30}"
            r"\b(?:asking|pushing|pressuring|contacting|trying)\b"
        ),
    ),

    "IDENTITY_IMPERSONATION": (
        r"\bpretending to be you\b",
        r"\busing your identity\b",
        (
            r"\b(?:i made|i created)\s+(?:a )?"
            r"fake (?:account|profile)\s+"
            r"(?:pretending to be you|in your name)\b"
        ),
        (
            r"\b(?:someone|they|a person)\b"
            r".{0,80}"
            r"\b(?:opened|created|made|started|set up)\b"
            r".{0,50}"
            r"\b(?:account|profile)\b"
            r".{0,40}"
            r"\bimpersonat(?:e|es|ed|ing)\s+you\b"
        ),
    ),
}


# ------------------------------------------------------------
# 2. Context checks for explicit benign/educational wording
# ------------------------------------------------------------

def v3_context_suppression(rule_id, text):
    """
    Conservative prototype checks for clear context.

    This is not full semantic understanding. Ambiguous cases should
    still be reviewed by the user.
    """
    text_lower = text.casefold()

    # Only suppress when there is an explicit educational, fictional,
    # or phrase-discussion context. A quote alone is not enough.
    meta_context_patterns = [
        (
            r"\b(?:in this|this|the|a|an|our)\b"
            r".{0,100}"
            r"\b(?:training slide|training manual|training material|"
            r"safety guide|play|script|manual|article|fiction)\b"
            r".{0,100}"
            r"\b(?:example|phrase|line|quote|wording)\b"
        ),
        (
            r"\b(?:the )?(?:phrase|wording|line|expression)\b"
            r".{0,120}"
            r"\b(?:appears in|is from|comes from|is quoted in|"
            r"is used in|includes)\b"
            r".{0,80}"
            r"\b(?:training|guide|play|script|manual|fiction|article)\b"
        ),
    ]

    if any(
        re.search(pattern, text_lower)
        for pattern in meta_context_patterns
    ):
        return "explicit educational, fictional, or phrase-discussion context"

    if rule_id == "THREAT_TO_EXPOSE_IMAGES":
        shared_with_recipient = re.search(
            r"\b(?:share|send|show|upload|post|publish)\b"
            r"\s+(?:(?:your|the)\s+)?"
            r"(?:(?:private|intimate|personal)\s+)?"
            r"(?:photos?|pictures?|images?|videos?|nudes?|pics?)"
            r"\s+with you\b",
            text_lower,
        )

        if shared_with_recipient:
            return "the text describes sharing material with the recipient"

    if rule_id == "COERCIVE_IMAGE_REQUEST":
        # A negative instruction is not itself a request to share imagery.
        negative_instruction_patterns = [
            r"\b(?:you\s+)?(?:don't|do not)\s+need to\s+"
            r"(?:send|share|provide|give|show)\b",
            r"\b(?:don't|do not|never)\s+"
            r"(?:send|share|provide|give|show)\b",
            r"\bi don't want you to\s+"
            r"(?:send|share|provide|give|show)\b",
        ]

        if any(
            re.search(pattern, text_lower)
            for pattern in negative_instruction_patterns
        ):
            return "the message discourages the requested action"

        voluntary_language = re.search(
            r"\b(?:only if you want to|if you (?:genuinely )?want to|"
            r"if you feel comfortable|no pressure|you can say no|"
            r"it's okay to say no|it is okay to say no|"
            r"with your permission)\b",
            text_lower,
        )

        pressure_language = re.search(
            r"\b(?:you owe me|prove you love me|if you love me|"
            r"you must|you have to|you need to|or else|"
            r"i will expose|i'll expose)\b",
            text_lower,
        )

        if voluntary_language and not pressure_language:
            return "voluntary language without a configured pressure cue"

    if rule_id == "DIRECT_THREAT_OR_STALKING_LANGUAGE":
        delivery_context = re.search(
            r"\b(?:deliver your parcel|deliver your package|"
            r"parcel delivery|package delivery|drop off your order)\b",
            text_lower,
        )

        additional_threat = re.search(
            r"\b(?:hurt you|find you|come to your home|"
            r"waiting outside your|outside your house)\b",
            text_lower,
        )

        if (
            "i know where you live" in text_lower
            and delivery_context
            and not additional_threat
        ):
            return "delivery context without an additional threat phrase"

    return None


# ------------------------------------------------------------
# 3. Experimental V3 analyzer
# ------------------------------------------------------------

def analyze_incident_v3(incident):
    if not isinstance(incident, Incident):
        raise TypeError("incident must be an Incident object.")

    text = incident.message_text
    findings = []
    suppressions = []

    for rule in SIGNAL_RULES:
        patterns = V3_PATTERN_OVERRIDES.get(
            rule.rule_id,
            rule.patterns,
        )

        suppression_reason = v3_context_suppression(
            rule.rule_id,
            text,
        )

        if suppression_reason:
            has_match = any(
                re.search(pattern, text, flags=re.IGNORECASE)
                for pattern in patterns
            )

            if has_match:
                suppressions.append({
                    "rule_id": rule.rule_id,
                    "reason": suppression_reason,
                })
                continue

        for pattern in patterns:
            match = re.search(
                pattern,
                text,
                flags=re.IGNORECASE,
            )

            if match:
                findings.append({
                    "rule_id": rule.rule_id,
                    "label": rule.label,
                    "category": rule.category,
                    "evidence": match.group(0),
                    "explanation": rule.explanation,
                    "suggested_action": rule.suggested_action,
                })
                break

    return {
        "incident_id": incident.incident_id,
        "platform": incident.platform.value,
        "occurred_at": incident.occurred_at.isoformat(),
        "status": (
            "potential_warning_indicators_found"
            if findings
            else "no_configured_pattern_matched"
        ),
        "findings": findings,
        "message": (
            "Review the highlighted wording and context. "
            "These indicators are not proof of abuse."
            if findings
            else
            "No configured V3 pattern matched. This does not "
            "mean the message is safe."
        ),
        "experimental_suppressions": suppressions,
    }


# ------------------------------------------------------------
# 4. Compare V1, V2 and V3 on the same development set
# ------------------------------------------------------------

challenge_v3_results = _challenge_evaluate(
    CHALLENGE_SET_V1,
    analyze_incident_v3,
)

print("=" * 76)
print("SIGNALHER — V1 vs V2 vs EXPERIMENTAL V3")
print("=" * 76)

print(
    f"{'Metric':24} {'V1':>12} {'V2':>12} {'V3':>12}"
)

for key, label in [
    ("precision", "Micro precision"),
    ("recall", "Micro recall"),
    ("f1", "Micro F1"),
    ("exact_match_rate", "Exact set match"),
]:
    print(
        f"{label:24} "
        f"{_challenge_percent(challenge_v1_results[key]):>12} "
        f"{_challenge_percent(challenge_v2_results[key]):>12} "
        f"{_challenge_percent(challenge_v3_results[key]):>12}"
    )


print("\nV3 FALSE POSITIVES")
print("-" * 76)

for item in challenge_v3_results["false_positives"]:
    print(
        f"{item['case_id']} | {item['unexpected_rules']} | "
        f"{item['message_text']}"
    )

if not challenge_v3_results["false_positives"]:
    print("None in this development set.")


print("\nV3 FALSE NEGATIVES")
print("-" * 76)

for item in challenge_v3_results["false_negatives"]:
    print(
        f"{item['case_id']} | {item['missed_rules']} | "
        f"{item['message_text']}"
    )

if not challenge_v3_results["false_negatives"]:
    print("None in this development set.")


print("\nV3 AMBIGUOUS CASES")
print("-" * 76)

for item in challenge_v3_results["ambiguous_predictions"]:
    print(
        f"{item['case_id']} | Detected: {item['detected_rules']} | "
        f"{item['message_text']}"
    )


print("\nNOTE")
print(
    "V3 is experimental. These development-set results must not be "
    "presented as real-world performance. SIGNAL_RULES and the active "
    "Flask API have not been modified."
)

In [ ]:
# ============================================================
# SIGNALHER — EXPERIMENTAL V4 COVERAGE LAYER
#
# Adds three distinct indicators:
#   1. Possible coercive disclosure following a demand
#   2. Request for private imagery (context needed)
#   3. Possible intimidation or reprisal
#
# Experimental only. The active API and SIGNAL_RULES remain unchanged.
# ============================================================

import re


V4_SUPPLEMENTAL_RULES = [
    {
        "rule_id": "COERCIVE_DISCLOSURE_THREAT",
        "label": "Possible threat to disclose information after a demand",
        "category": "coercion",
        "patterns": (
            # Example: "Pay me or I'll reveal what happened."
            r"\b(?:pay me|pay|send me money|send money|"
            r"transfer (?:me )?money|give me money)\b"
            r".{0,80}"
            r"\b(?:or else|otherwise|or i'll|or i will)\b"
            r".{0,80}"
            r"\b(?:reveal|disclose|expose|leak|"
            r"tell everyone|make public)\b",

            # Example: "Pay me or I will publish the records."
            r"\b(?:pay me|pay|send me money|send money|"
            r"transfer (?:me )?money|give me money)\b"
            r".{0,80}"
            r"\b(?:or else|otherwise|or i'll|or i will)\b"
            r".{0,80}"
            r"\b(?:publish|release|distribute)\b"
            r".{0,50}"
            r"\b(?:records?|documents?|files?|data|photos?|"
            r"pictures?|images?|videos?|messages?|secrets?)\b",
        ),
        "explanation": (
            "The message links a demand to possible disclosure of "
            "information. The significance and nature of the threatened "
            "disclosure require context; this indicator is not proof "
            "of extortion or wrongdoing."
        ),
        "suggested_action": (
            "Review the complete exchange. If you feel pressured or "
            "threatened, consider trusted support and preserve relevant "
            "evidence if it is safe to do so."
        ),
    },
    {
        "rule_id": "REQUEST_FOR_PRIVATE_IMAGERY",
        "label": "Request for private imagery — context needed",
        "category": "sensitive_image_request",
        "patterns": (
            r"\b(?:send|share|show|give|provide|upload)\b"
            r"\s+(?:me\s+)?(?:your\s+)?"
            r"(?:private|intimate|personal|nude|naked|sexual)\s+"
            r"(?:photos?|pictures?|images?|videos?|pics?|imagery)\b",
        ),
        "explanation": (
            "The message appears to request private or intimate imagery. "
            "A request alone does not establish coercion, exploitation, "
            "non-consensual intent, or abuse. Consider the context and "
            "whether consent is freely given."
        ),
        "suggested_action": (
            "You are not obligated to share private or intimate material. "
            "If you feel pressured, you can decline or seek trusted support "
            "if doing so feels safe."
        ),
    },
    {
        "rule_id": "VAGUE_INTIMIDATION_OR_REPRISAL",
        "label": "Possible intimidation or reprisal",
        "category": "coercion",
        "patterns": (
            # Example: "You will regret turning me down."
            r"\b(?:you will|you'll)\s+regret\s+"
            r"(?:turning me down|rejecting me|saying no)\b",

            # Example: "I'll make you regret it."
            r"\b(?:i will|i'll)\s+make you regret\s+(?:it|this|that)\b",

            # Example: "If you don't send the files, I'll make you regret it."
            r"\bif you (?:don't|do not)\s+"
            r"(?:send|share|give|pay|reply|respond|meet|provide)\b"
            r".{0,100}"
            r"\b(?:i will|i'll)\s+make you regret\b",
        ),
        "explanation": (
            "The wording may imply intimidation or retaliation. The "
            "specific consequence and seriousness are unclear, so this "
            "indicator is not a determination that a threat is credible."
        ),
        "suggested_action": (
            "Consider the surrounding conversation and your circumstances. "
            "If you believe someone may harm you, prioritise your safety "
            "and seek appropriate support."
        ),
    },
]


def _v4_is_explicitly_negated_image_request(text):
    """
    Avoid classifying clear instructions against sharing imagery
    as requests to share it.
    """
    return bool(re.search(
        r"\b(?:don't|do not|never|should not|must not)\s+"
        r"(?:need to\s+)?"
        r"(?:send|share|show|give|provide|upload)\b",
        text,
        flags=re.IGNORECASE,
    ))


def analyze_incident_v4(incident):
    """
    Run V3 and supplement it with three context-sensitive indicators.

    The baseline findings are preserved. This function is a separate
    research prototype and is not used by the active Flask API.
    """
    if not isinstance(incident, Incident):
        raise TypeError("incident must be an Incident object.")

    result = analyze_incident_v3(incident)
    findings = list(result["findings"])
    existing_ids = {
        finding["rule_id"]
        for finding in findings
    }

    text = incident.message_text

    for rule in V4_SUPPLEMENTAL_RULES:
        if rule["rule_id"] == "REQUEST_FOR_PRIVATE_IMAGERY":
            if _v4_is_explicitly_negated_image_request(text):
                continue

            # If V3 already recorded an intimate-image request,
            # avoid a duplicate indicator for the same request.
            if "COERCIVE_IMAGE_REQUEST" in existing_ids:
                continue

        for pattern in rule["patterns"]:
            match = re.search(
                pattern,
                text,
                flags=re.IGNORECASE,
            )

            if not match:
                continue

            # Keep the threat-to-expose finding separate from the
            # private-imagery request indicator.
            if (
                rule["rule_id"] == "REQUEST_FOR_PRIVATE_IMAGERY"
                and "THREAT_TO_EXPOSE_IMAGES" in existing_ids
            ):
                break

            findings.append({
                "rule_id": rule["rule_id"],
                "label": rule["label"],
                "category": rule["category"],
                "evidence": match.group(0),
                "explanation": rule["explanation"],
                "suggested_action": rule["suggested_action"],
            })

            existing_ids.add(rule["rule_id"])
            break

    result["findings"] = findings
    result["status"] = (
        "potential_warning_indicators_found"
        if findings
        else "no_configured_pattern_matched"
    )

    result["message"] = (
        "Review the highlighted wording and its wider context. "
        "Indicators are not proof of abuse or a prediction of future harm."
        if findings
        else
        "No configured V4 pattern matched this message. "
        "This does not mean the interaction is safe."
    )

    return result


# ============================================================
# TEST THE SPECIFIC COVERAGE GAPS YOU IDENTIFIED
# ============================================================

V4_TARGETED_TESTS = [
    {
        "id": "A01",
        "text": "Pay me or I'll reveal what happened.",
        "expected_new_rule": "COERCIVE_DISCLOSURE_THREAT",
    },
    {
        "id": "A02",
        "text": "Send me your private photos.",
        "expected_new_rule": "REQUEST_FOR_PRIVATE_IMAGERY",
    },
    {
        "id": "A07",
        "text": "You will regret turning me down.",
        "expected_new_rule": "VAGUE_INTIMIDATION_OR_REPRISAL",
    },
    {
        "id": "A08",
        "text": "If you don't send the files, I'll make you regret it.",
        "expected_new_rule": "VAGUE_INTIMIDATION_OR_REPRISAL",
    },
    {
        "id": "B01",
        "text": "If you don't pay, I will send the receipt.",
        "expected_new_rule": None,
    },
    {
        "id": "B02",
        "text": "You don't need to send intimate photos; your choice matters.",
        "expected_new_rule": None,
    },
    {
        "id": "B03",
        "text": "You can share your intimate photos with me only if you want to.",
        "expected_new_rule": None,
    },
]

print("=" * 72)
print("SIGNALHER — V4 TARGETED COVERAGE TEST")
print("=" * 72)

v4_test_results = []

for test in V4_TARGETED_TESTS:
    incident = _build_incident_from_payload({
        "message_text": test["text"],
    })

    output = analyze_incident_v4(incident)
    detected = {
        item["rule_id"] for item in output["findings"]
    }

    expected = test["expected_new_rule"]

    if expected is None:
        # No supplementary indicator is expected for benign controls.
        # V3 may still independently trigger another existing rule;
        # that is reported for review rather than hidden.
        supplementary_ids = {
            item["rule_id"]
            for item in output["findings"]
            if item["rule_id"] in {
                rule["rule_id"] for rule in V4_SUPPLEMENTAL_RULES
            }
        }

        passed = not supplementary_ids
    else:
        passed = expected in detected

    v4_test_results.append({
        "id": test["id"],
        "expected": expected,
        "detected": sorted(detected),
        "passed": passed,
        "text": test["text"],
    })

    print(
        f"{'PASS' if passed else 'REVIEW'} | {test['id']} | "
        f"Expected supplemental rule: {expected} | "
        f"Detected: {sorted(detected)}"
    )

print("\nTargeted checks passed:",
      sum(row["passed"] for row in v4_test_results),
      "/",
      len(v4_test_results))

print("\nThese tests do not establish real-world accuracy.")
print("The active SIGNAL_RULES and Flask API remain unchanged.")

In [ ]:
# ============================================================
# SIGNALHER — V4 CONSENT-CONTEXT FIX
# Experimental analyzer only. Active backend remains unchanged.
# ============================================================

import re


def _v4_is_explicitly_negated_image_request(text):
    """
    Return True when:
    1. The message clearly discourages sharing imagery; or
    2. It expresses voluntary consent without an explicit pressure cue.

    This is a limited heuristic, not a determination of consent.
    """
    text_lower = text.casefold()

    # Case 1: Explicitly discouraging the action.
    negative_instruction = re.search(
        r"\b(?:don't|do not|never|should not|must not)\s+"
        r"(?:need to\s+)?"
        r"(?:send|share|show|give|provide|upload)\b",
        text_lower,
    )

    if negative_instruction:
        return True

    # Case 2: The request is framed as optional or voluntary.
    voluntary_language = re.search(
        r"\b(?:only if you want to|"
        r"if you (?:genuinely )?want to|"
        r"if you feel comfortable|"
        r"no pressure|"
        r"you can say no|"
        r"it's okay to say no|"
        r"it is okay to say no|"
        r"with your permission)\b",
        text_lower,
    )

    if not voluntary_language:
        return False

    # If clear pressure or threat language is also present,
    # don't suppress the indicator solely because consent words appear.
    pressure_language = re.search(
        r"\b(?:you owe me|"
        r"prove you love me|"
        r"if you love me|"
        r"you must|"
        r"you have to|"
        r"you need to|"
        r"or else|"
        r"unless you|"
        r"i will expose|"
        r"i'll expose|"
        r"i will hurt you|"
        r"i'll hurt you|"
        r"do it now)\b",
        text_lower,
    )

    return pressure_language is None


# ------------------------------------------------------------
# Test the specific false positive and related controls
# ------------------------------------------------------------

V4_CONSENT_TESTS = [
    {
        "id": "CONSENT-01",
        "text": "You can share your intimate photos with me only if you want to.",
        "expect_request_indicator": False,
    },
    {
        "id": "CONSENT-02",
        "text": "You don't need to send intimate photos; your choice matters.",
        "expect_request_indicator": False,
    },
    {
        "id": "REQUEST-01",
        "text": "Send me your private photos.",
        "expect_request_indicator": True,
    },
    {
        "id": "PRESSURE-01",
        "text": "If you love me, send me your intimate photos.",
        "expect_request_indicator": True,
    },
    {
        "id": "PRESSURE-02",
        "text": (
            "If you love me, send me your intimate photos, "
            "only if you want to."
        ),
        "expect_request_indicator": True,
    },
]

consent_test_results = []

for test in V4_CONSENT_TESTS:
    incident = _build_incident_from_payload({
        "message_text": test["text"],
    })

    result = analyze_incident_v4(incident)

    detected_ids = {
        finding["rule_id"]
        for finding in result["findings"]
    }

    detected_request = (
        "REQUEST_FOR_PRIVATE_IMAGERY" in detected_ids
        or "COERCIVE_IMAGE_REQUEST" in detected_ids
    )

    passed = (
        detected_request == test["expect_request_indicator"]
    )

    consent_test_results.append({
        "id": test["id"],
        "passed": passed,
        "detected_ids": sorted(detected_ids),
        "text": test["text"],
    })

    print(
        f"{'PASS' if passed else 'REVIEW'} | {test['id']} | "
        f"Detected: {sorted(detected_ids)}"
    )
    print(f"  Message: {test['text']}")


print("\nConsent-context tests passed:",
      sum(item["passed"] for item in consent_test_results),
      "/",
      len(consent_test_results))

assert all(item["passed"] for item in consent_test_results), (
    "One or more consent-context tests need review. "
    "Inspect the messages and detections printed above."
)

print("\nConsent-context filter checks passed.")
print("V4 remains experimental; no active backend rules were changed.")

In [ ]:
# ============================================================
# SIGNALHER — V4 REGRESSION AND CONTEXT CHECKS
# Experimental analyzer only; active backend is unchanged.
# ============================================================

import re


def _v4_is_explicitly_negated_image_request(text):
    """
    Suppress the new private-imagery request indicator when there is:
    - An explicit instruction against sharing;
    - Clear voluntary/consent wording without pressure;
    - A clear description of sharing material with the recipient;
    - An explicit educational or fictional discussion of a phrase.

    This is a limited rule-based heuristic, not semantic understanding.
    """
    text_lower = text.casefold()

    # 1. Explicit instructions against sharing.
    negative_instruction = re.search(
        r"\b(?:don't|do not|never|should not|must not)\s+"
        r"(?:need to\s+)?"
        r"(?:send|share|show|give|provide|upload)\b",
        text_lower,
    )

    if negative_instruction:
        return True

    # 2. Sharing with the recipient rather than threatening disclosure.
    sharing_with_recipient = re.search(
        r"\b(?:share|send|show|upload|post|publish)\b"
        r"\s+(?:(?:your|the)\s+)?"
        r"(?:(?:private|intimate|personal)\s+)?"
        r"(?:photos?|pictures?|images?|videos?|nudes?|pics?)"
        r"\s+with you\b",
        text_lower,
    )

    if sharing_with_recipient:
        return True

    # 3. Explicit educational, fictional, or metalinguistic context.
    educational_context_patterns = [
        (
            r"\b(?:in this|this|the|a|an|our)\b"
            r".{0,100}"
            r"\b(?:training slide|training manual|training material|"
            r"safety guide|play|script|manual|article|fiction)\b"
            r".{0,100}"
            r"\b(?:example|phrase|line|quote|wording)\b"
        ),
        (
            r"\b(?:the )?(?:phrase|wording|line|expression)\b"
            r".{0,120}"
            r"\b(?:appears in|is from|comes from|is quoted in|"
            r"is used in|includes)\b"
            r".{0,80}"
            r"\b(?:training|guide|play|script|manual|fiction|article)\b"
        ),
        (
            r"\b(?:our|the|a)\s+play\b"
            r".{0,80}\b(?:includes|contains|uses)\b"
            r".{0,80}\b(?:the line|the phrase|the words)\b"
        ),
    ]

    if any(
        re.search(pattern, text_lower)
        for pattern in educational_context_patterns
    ):
        return True

    # 4. Voluntary language, unless explicit pressure is also present.
    voluntary_language = re.search(
        r"\b(?:only if you want to|"
        r"if you (?:genuinely )?want to|"
        r"if you feel comfortable|"
        r"no pressure|"
        r"you can say no|"
        r"it's okay to say no|"
        r"it is okay to say no|"
        r"with your permission)\b",
        text_lower,
    )

    explicit_pressure = re.search(
        r"\b(?:you owe me|"
        r"prove you love me|"
        r"if you love me|"
        r"you must|"
        r"you have to|"
        r"you need to|"
        r"or else|"
        r"unless you|"
        r"i will expose|"
        r"i'll expose|"
        r"i will hurt you|"
        r"i'll hurt you|"
        r"do it now)\b",
        text_lower,
    )

    if voluntary_language and not explicit_pressure:
        return True

    return False


# ------------------------------------------------------------
# REGRESSION TESTS
#
# expected_supplemental is the expected result for the NEW
# supplemental indicator families. Existing V3 rules may
# independently flag a message; those results are printed too.
# ------------------------------------------------------------

V4_REGRESSION_TESTS = [
    {
        "id": "R01",
        "text": "Pay me or I'll reveal what happened.",
        "expected_supplemental": "COERCIVE_DISCLOSURE_THREAT",
    },
    {
        "id": "R02",
        "text": "Send me your private photos.",
        "expected_supplemental": "REQUEST_FOR_PRIVATE_IMAGERY",
    },
    {
        "id": "R03",
        "text": "You will regret turning me down.",
        "expected_supplemental": "VAGUE_INTIMIDATION_OR_REPRISAL",
    },
    {
        "id": "R04",
        "text": "If you don't send me the files, I'll make you regret it.",
        "expected_supplemental": "VAGUE_INTIMIDATION_OR_REPRISAL",
    },
    {
        "id": "R05",
        "text": (
            "If you don't pay, I will send you the receipt."
        ),
        "expected_supplemental": None,
    },
    {
        "id": "R06",
        "text": (
            "You don't need to send intimate photos; your choice matters."
        ),
        "expected_supplemental": None,
    },
    {
        "id": "R07",
        "text": (
            "You can share your intimate photos with me only if you want to."
        ),
        "expected_supplemental": None,
    },
    {
        "id": "R08",
        "text": (
            "I will share your private photos with you after editing them."
        ),
        "expected_supplemental": None,
    },
    {
        "id": "R09",
        "text": (
            "In this training slide, consider this example: "
            "I will expose your private images."
        ),
        "expected_supplemental": None,
    },
    {
        "id": "R10",
        "text": (
            "Our play includes the line 'I know where you live'."
        ),
        "expected_supplemental": None,
    },
    {
        "id": "R11",
        "text": "If you love me, send me your intimate photos.",
        "expected_supplemental": None,
        "expected_existing_rule": "COERCIVE_IMAGE_REQUEST",
    },
    {
        "id": "R12",
        "text": (
            "Please don't share my intimate images."
        ),
        "expected_supplemental": None,
    },
]


v4_regression_results = []

supplemental_ids = {
    rule["rule_id"] for rule in V4_SUPPLEMENTAL_RULES
}

print("=" * 76)
print("SIGNALHER — V4 REGRESSION TESTS")
print("=" * 76)

for test in V4_REGRESSION_TESTS:
    incident = _build_incident_from_payload({
        "message_text": test["text"],
    })

    output = analyze_incident_v4(incident)

    detected = {
        finding["rule_id"] for finding in output["findings"]
    }

    detected_supplemental = detected & supplemental_ids

    expected_supplemental = test["expected_supplemental"]

    if expected_supplemental is None:
        supplemental_ok = not detected_supplemental
    else:
        supplemental_ok = (
            expected_supplemental in detected_supplemental
        )

    expected_existing = test.get("expected_existing_rule")
    existing_ok = (
        expected_existing is None
        or expected_existing in detected
    )

    passed = supplemental_ok and existing_ok

    v4_regression_results.append({
        "id": test["id"],
        "passed": passed,
        "expected_supplemental": expected_supplemental,
        "detected_supplemental": sorted(detected_supplemental),
        "detected_all": sorted(detected),
        "text": test["text"],
    })

    print(
        f"{'PASS' if passed else 'REVIEW'} | {test['id']} | "
        f"new indicators={sorted(detected_supplemental)} | "
        f"all findings={sorted(detected)}"
    )

    if not passed:
        print("  Message:", test["text"])


passed_count = sum(
    item["passed"] for item in v4_regression_results
)

print("\nRegression tests passed:", passed_count, "/", len(v4_regression_results))

if passed_count == len(v4_regression_results):
    print("All targeted V4 regression checks passed.")
else:
    print("Some cases need review. Do not promote V4 yet.")

print(
    "\nThese are targeted regression checks, not an independent "
    "performance benchmark."
)
print("Active SIGNAL_RULES and Flask routes remain unchanged.")

In [ ]:
# ============================================================
# SIGNALHER — V4 OVERLAP AUDIT (FIELD-NAME FIX)
# Read-only diagnostic; no application changes.
# ============================================================

from collections import Counter

audit_suites = [
    ("development_v1", SYNTHETIC_EVALUATION_V1_1),
    ("challenge_v1", CHALLENGE_SET_V1),
    ("targeted_v4", V4_REGRESSION_TESTS),
]

POTENTIAL_DUPLICATE_PAIRS = [
    ("EXTORTION_THREAT", "COERCIVE_DISCLOSURE_THREAT"),
    ("THREAT_TO_EXPOSE_IMAGES", "COERCIVE_DISCLOSURE_THREAT"),
    ("COERCIVE_IMAGE_REQUEST", "REQUEST_FOR_PRIVATE_IMAGERY"),
]

all_audit_rows = []
finding_counts = Counter()
duplicate_cases = []

for suite_name, suite in audit_suites:
    for index, case in enumerate(suite, start=1):
        case_id = case.get(
            "case_id",
            case.get("id", f"{suite_name}_{index:02}"),
        )

        # Different suites use different message field names.
        message = case.get("message_text")
        if message is None:
            message = case.get("text")

        if not isinstance(message, str):
            print(
                f"Skipping malformed case: "
                f"{suite_name} / {case_id}"
            )
            continue

        payload = {"message_text": message}

        if case.get("platform") is not None:
            payload["platform"] = case["platform"]

        incident = _build_incident_from_payload(payload)
        result = analyze_incident_v4(incident)

        detected_ids = {
            finding["rule_id"]
            for finding in result.get("findings", [])
        }

        for rule_id in detected_ids:
            finding_counts[rule_id] += 1

        for first_rule, second_rule in POTENTIAL_DUPLICATE_PAIRS:
            if first_rule in detected_ids and second_rule in detected_ids:
                duplicate_cases.append({
                    "suite": suite_name,
                    "case_id": case_id,
                    "first_rule": first_rule,
                    "second_rule": second_rule,
                    "message": message,
                })

        all_audit_rows.append({
            "suite": suite_name,
            "case_id": case_id,
            "message": message,
            "detected_ids": sorted(detected_ids),
        })


print("=" * 72)
print("SIGNALHER — V4 OVERLAP AUDIT")
print("=" * 72)
print("Test instances analysed:", len(all_audit_rows))

print("\nFINDING FREQUENCIES")
print("-" * 72)

for rule_id, count in sorted(finding_counts.items()):
    print(f"{rule_id:42} {count:>4}")


print("\nPOTENTIAL DUPLICATE FINDINGS")
print("-" * 72)

if duplicate_cases:
    for item in duplicate_cases:
        print(
            f"{item['suite']} / {item['case_id']}\n"
            f"  Rules: {item['first_rule']} + {item['second_rule']}\n"
            f"  Message: {item['message']}\n"
        )
else:
    print("No configured overlap pairs were detected.")


print("\nCASES WITH THE MOST FINDINGS")
print("-" * 72)

ranked_rows = sorted(
    all_audit_rows,
    key=lambda row: len(row["detected_ids"]),
    reverse=True,
)

for item in ranked_rows[:10]:
    print(
        f"{item['suite']} / {item['case_id']} | "
        f"{len(item['detected_ids'])} finding(s) | "
        f"{item['detected_ids']} | {item['message']}"
    )


print("\nAudit completed. No active rules or API routes were changed.")

In [ ]:
# ============================================================
# SIGNALHER — V4 SPECIFICITY-BASED DEDUPLICATION
# Suppress a generic disclosure finding when a more specific
# finding already captures the same message.
# ============================================================

# Preserve the original experimental implementation when this cell
# is rerun, so we don't create a chain of wrappers.
if "_v4_analyzer_before_dedup" not in globals():
    _v4_analyzer_before_dedup = analyze_incident_v4


def analyze_incident_v4(incident):
    """
    Run the experimental V4 analyzer and remove redundant generic
    disclosure findings when a more specific rule already applies.

    The existing rules and active Flask API remain unchanged.
    """
    result = _v4_analyzer_before_dedup(incident)

    findings = list(result.get("findings", []))
    detected_ids = {
        finding.get("rule_id")
        for finding in findings
    }

    specific_findings = {
        "EXTORTION_THREAT",
        "THREAT_TO_EXPOSE_IMAGES",
    }

    redundant_generic_findings = []

    if (
        "COERCIVE_DISCLOSURE_THREAT" in detected_ids
        and detected_ids.intersection(specific_findings)
    ):
        redundant_generic_findings.append(
            "COERCIVE_DISCLOSURE_THREAT"
        )

        findings = [
            finding
            for finding in findings
            if finding.get("rule_id")
            != "COERCIVE_DISCLOSURE_THREAT"
        ]

    result["findings"] = findings
    result["deduplication_notes"] = []

    if redundant_generic_findings:
        result["deduplication_notes"].append({
            "suppressed_rule_ids": redundant_generic_findings,
            "reason": (
                "A more specific extortion or image-exposure finding "
                "already covers the relevant disclosure pattern."
            ),
        })

    return result


# ============================================================
# TEST 1 — The four previously observed duplicate examples
# ============================================================

DUPLICATE_TEST_MESSAGES = [
    "Send me money or I'll publish the documents tonight.",
    "Pay me or else I will distribute the records.",
    "Transfer me money or I'll publish the confidential records.",
    "Pay the amount by tonight, or I will release the records.",
]

dedup_test_results = []

for index, message in enumerate(DUPLICATE_TEST_MESSAGES, start=1):
    incident = _build_incident_from_payload({
        "message_text": message,
    })

    result = analyze_incident_v4(incident)
    finding_ids = {
        finding["rule_id"]
        for finding in result["findings"]
    }

    passed = (
        "EXTORTION_THREAT" in finding_ids
        and "COERCIVE_DISCLOSURE_THREAT" not in finding_ids
    )

    dedup_test_results.append(passed)

    print(
        f"{'PASS' if passed else 'REVIEW'} | DEDUP-{index} | "
        f"Findings: {sorted(finding_ids)}"
    )


# ============================================================
# TEST 2 — A more specific image-threat finding should remain
# alongside extortion, where both apply.
# ============================================================

image_threat_message = (
    "Pay me or I'll expose your private images."
)

image_incident = _build_incident_from_payload({
    "message_text": image_threat_message,
})

image_result = analyze_incident_v4(image_incident)

image_finding_ids = {
    finding["rule_id"]
    for finding in image_result["findings"]
}

image_test_passed = (
    "EXTORTION_THREAT" in image_finding_ids
    and "THREAT_TO_EXPOSE_IMAGES" in image_finding_ids
    and "COERCIVE_DISCLOSURE_THREAT" not in image_finding_ids
)

print(
    f"{'PASS' if image_test_passed else 'REVIEW'} | "
    f"IMAGE-EXTORTION | Findings: {sorted(image_finding_ids)}"
)


# ============================================================
# TEST 3 — The broader indicator must still catch an unmatched case
# ============================================================

unmatched_disclosure_message = "Pay me or I'll reveal what happened."

unmatched_incident = _build_incident_from_payload({
    "message_text": unmatched_disclosure_message,
})

unmatched_result = analyze_incident_v4(unmatched_incident)

unmatched_finding_ids = {
    finding["rule_id"]
    for finding in unmatched_result["findings"]
}

unmatched_test_passed = (
    "COERCIVE_DISCLOSURE_THREAT" in unmatched_finding_ids
    and "EXTORTION_THREAT" not in unmatched_finding_ids
    and "THREAT_TO_EXPOSE_IMAGES" not in unmatched_finding_ids
)

print(
    f"{'PASS' if unmatched_test_passed else 'REVIEW'} | "
    f"UNMATCHED-DISCLOSURE | "
    f"Findings: {sorted(unmatched_finding_ids)}"
)


# ============================================================
# TEST 4 — Re-run overlap audit across all three suites
# ============================================================

audit_suites = [
    ("development_v1", SYNTHETIC_EVALUATION_V1_1),
    ("challenge_v1", CHALLENGE_SET_V1),
    ("targeted_v4", V4_REGRESSION_TESTS),
]

remaining_duplicates = []

for suite_name, suite in audit_suites:
    for index, case in enumerate(suite, start=1):
        case_id = case.get(
            "case_id",
            case.get("id", f"{suite_name}_{index:02}"),
        )

        message = case.get("message_text")
        if message is None:
            message = case.get("text")

        if not isinstance(message, str):
            continue

        payload = {"message_text": message}

        if case.get("platform") is not None:
            payload["platform"] = case["platform"]

        incident = _build_incident_from_payload(payload)
        result = analyze_incident_v4(incident)

        finding_ids = {
            finding["rule_id"]
            for finding in result["findings"]
        }

        # The generic disclosure indicator should not appear with
        # either more specific rule.
        if (
            "COERCIVE_DISCLOSURE_THREAT" in finding_ids
            and finding_ids.intersection({
                "EXTORTION_THREAT",
                "THREAT_TO_EXPOSE_IMAGES",
            })
        ):
            remaining_duplicates.append({
                "suite": suite_name,
                "case_id": case_id,
                "findings": sorted(finding_ids),
                "message": message,
            })


print("\n" + "=" * 72)
print("V4 DEDUPLICATION SUMMARY")
print("=" * 72)

print("Four original duplicate checks:",
      sum(dedup_test_results), "/", len(dedup_test_results))
print("Specific image-threat plus extortion check:",
      "PASSED" if image_test_passed else "REVIEW")
print("Unmatched disclosure coverage check:",
      "PASSED" if unmatched_test_passed else "REVIEW")
print("Remaining generic/specific overlap cases:",
      len(remaining_duplicates))

for item in remaining_duplicates:
    print(item)

assert all(dedup_test_results), (
    "At least one original duplicate case failed."
)
assert image_test_passed, "Specific image-threat findings were lost."
assert unmatched_test_passed, "Unmatched disclosure coverage was lost."
assert not remaining_duplicates, (
    "Some redundant generic disclosure findings remain."
)

print("\nAll deduplication checks passed.")
print("V4 remains experimental; active SIGNAL_RULES are unchanged.")

In [ ]:
# ============================================================
# SIGNALHER — V4 INTEGRATED REGRESSION EVALUATION
#
# 18 intended-positive cases
# 18 benign/negative controls
#  6 ambiguous cases (excluded from metrics)
#
# This is a manually authored regression suite, not an
# independent or real-world validation benchmark.
# ============================================================

V4_INTEGRATION_SUITE = [
    # --------------------------------------------------------
    # A. Intended-positive cases
    # --------------------------------------------------------

    {
        "id": "P01",
        "expected_rule_ids": ["EXTORTION_THREAT"],
        "message_text": "Send me money or I'll leak the records.",
        "platform": "Discord",
    },
    {
        "id": "P02",
        "expected_rule_ids": ["EXTORTION_THREAT"],
        "message_text": "Pay the amount or else I will expose the records.",
        "platform": "X",
    },
    {
        "id": "P03",
        "expected_rule_ids": ["THREAT_TO_EXPOSE_IMAGES"],
        "message_text": "I'm about to distribute your intimate pictures.",
        "platform": "Email",
    },
    {
        "id": "P04",
        "expected_rule_ids": ["THREAT_TO_EXPOSE_IMAGES"],
        "message_text": "I'll circulate your private videos.",
        "platform": "LinkedIn",
    },
    {
        "id": "P05",
        "expected_rule_ids": ["COERCIVE_IMAGE_REQUEST"],
        "message_text": "You owe me, send me your intimate images.",
        "platform": "Other",
    },
    {
        "id": "P06",
        "expected_rule_ids": ["COERCIVE_IMAGE_REQUEST"],
        "message_text": "Prove you love me by sending your nude pictures.",
        # Source intentionally omitted.
    },
    {
        "id": "P07",
        "expected_rule_ids": ["DIRECT_THREAT_OR_STALKING_LANGUAGE"],
        "message_text": "I will find you after work.",
        "platform": "Discord",
    },
    {
        "id": "P08",
        "expected_rule_ids": ["DIRECT_THREAT_OR_STALKING_LANGUAGE"],
        "message_text": "I'm coming to hurt you tonight.",
        "platform": "X",
    },
    {
        "id": "P09",
        "expected_rule_ids": ["BOUNDARY_DISREGARD"],
        "message_text": "You said no but meet me tomorrow.",
        "platform": "Email",
    },
    {
        "id": "P10",
        "expected_rule_ids": ["BOUNDARY_DISREGARD"],
        "message_text": "I don't care that you said no to the meeting.",
        # Source intentionally omitted.
    },
    {
        "id": "P11",
        "expected_rule_ids": ["IDENTITY_IMPERSONATION"],
        "message_text": "I created a fake profile pretending to be you.",
        "platform": "Other",
    },
    {
        "id": "P12",
        "expected_rule_ids": ["IDENTITY_IMPERSONATION"],
        "message_text": "Someone has opened an account impersonating you.",
        "platform": "LinkedIn",
    },
    {
        "id": "P13",
        "expected_rule_ids": ["COERCIVE_DISCLOSURE_THREAT"],
        "message_text": "Pay me or I'll reveal what happened.",
        "platform": "Discord",
    },
    {
        "id": "P14",
        "expected_rule_ids": ["COERCIVE_DISCLOSURE_THREAT"],
        "message_text": "Send me money or I'll tell everyone what you told me.",
        # Source intentionally omitted.
    },
    {
        "id": "P15",
        "expected_rule_ids": ["REQUEST_FOR_PRIVATE_IMAGERY"],
        "message_text": "Send me your private photos.",
        "platform": "X",
    },
    {
        "id": "P16",
        "expected_rule_ids": ["REQUEST_FOR_PRIVATE_IMAGERY"],
        "message_text": "Upload your personal images.",
        "platform": "Email",
    },
    {
        "id": "P17",
        "expected_rule_ids": ["VAGUE_INTIMIDATION_OR_REPRISAL"],
        "message_text": "You'll regret rejecting me.",
        "platform": "Instagram",
    },
    {
        "id": "P18",
        "expected_rule_ids": ["VAGUE_INTIMIDATION_OR_REPRISAL"],
        "message_text": "I'll make you regret it.",
        # Source intentionally omitted.
    },

    # --------------------------------------------------------
    # B. Benign/negative controls
    # These are deliberately challenging context and negation cases.
    # --------------------------------------------------------

    {
        "id": "N01",
        "expected_rule_ids": [],
        "message_text": "Please pay me and I'll send the invoice.",
        "platform": "Email",
    },
    {
        "id": "N02",
        "expected_rule_ids": [],
        "message_text": "I'll reveal the findings in the report.",
        "platform": "LinkedIn",
    },
    {
        "id": "N03",
        "expected_rule_ids": [],
        "message_text": "I will not expose your private images.",
        "platform": "WhatsApp",
    },
    {
        "id": "N04",
        "expected_rule_ids": [],
        "message_text": (
            "I will publish your private photos with you "
            "after editing them."
        ),
        "platform": "Other",
    },
    {
        "id": "N05",
        "expected_rule_ids": [],
        "message_text": "Please do not send nude photos.",
        "platform": "Discord",
    },
    {
        "id": "N06",
        "expected_rule_ids": [],
        "message_text": (
            "You can send me intimate photos if you feel comfortable."
        ),
        "platform": "X",
    },
    {
        "id": "N07",
        "expected_rule_ids": [],
        "message_text": "I'm not going to hurt you; I'm trying to calm things down.",
        "platform": "Email",
    },
    {
        "id": "N08",
        "expected_rule_ids": [],
        "message_text": "In the play, the line is 'I know where you live'.",
        # Source intentionally omitted.
    },
    {
        "id": "N09",
        "expected_rule_ids": [],
        "message_text": (
            "You said no but I respect your choice and won't meet."
        ),
        "platform": "Discord",
    },
    {
        "id": "N10",
        "expected_rule_ids": [],
        "message_text": "You said no, and I will stop asking.",
        "platform": "Email",
    },
    {
        "id": "N11",
        "expected_rule_ids": [],
        "message_text": (
            "I created a fake profile for my art portfolio; "
            "it doesn't imitate anyone."
        ),
        "platform": "Other",
    },
    {
        "id": "N12",
        "expected_rule_ids": [],
        "message_text": (
            "The phrase 'pretending to be you' appears in an awareness article."
        ),
        "platform": "X",
    },
    {
        "id": "N13",
        "expected_rule_ids": [],
        "message_text": "Pay me and I'll disclose the amount on the invoice.",
        "platform": "Email",
    },
    {
        "id": "N14",
        "expected_rule_ids": [],
        "message_text": "I will tell everyone the deadline at the meeting.",
        "platform": "LinkedIn",
    },
    {
        "id": "N15",
        "expected_rule_ids": [],
        "message_text": "I won't ask you to share private images.",
        # Source intentionally omitted.
    },
    {
        "id": "N16",
        "expected_rule_ids": [],
        "message_text": "You do not have to send private photos.",
        "platform": "WhatsApp",
    },
    {
        "id": "N17",
        "expected_rule_ids": [],
        "message_text": "You will regret missing the train.",
        "platform": "Discord",
    },
    {
        "id": "N18",
        "expected_rule_ids": [],
        "message_text": "I'll make you regret that haircut.",
        "platform": "X",
    },

    # --------------------------------------------------------
    # C. Ambiguous/context-dependent cases
    # These are reported separately and excluded from metrics.
    # --------------------------------------------------------

    {
        "id": "A01",
        "expected_rule_ids": None,
        "message_text": "I know where you live.",
        "platform": "Email",
    },
    {
        "id": "A02",
        "expected_rule_ids": None,
        "message_text": "Please send the private pictures we discussed.",
        "platform": "Discord",
    },
    {
        "id": "A03",
        "expected_rule_ids": None,
        "message_text": "I will publish the pictures.",
        # Source intentionally omitted.
    },
    {
        "id": "A04",
        "expected_rule_ids": None,
        "message_text": "Someone is using your identity online.",
        "platform": "LinkedIn",
    },
    {
        "id": "A05",
        "expected_rule_ids": None,
        "message_text": "Reveal the details or you'll regret it.",
        "platform": "X",
    },
    {
        "id": "A06",
        "expected_rule_ids": None,
        "message_text": "You will regret this.",
        # Source intentionally omitted.
    },
]


# ============================================================
# RUN V4 AND RECORD RESULTS
# ============================================================

V4_ALL_RULE_IDS = {
    rule.rule_id for rule in SIGNAL_RULES
} | {
    rule["rule_id"] for rule in V4_SUPPLEMENTAL_RULES
}

v4_evaluation_results = []

for case in V4_INTEGRATION_SUITE:
    payload = {"message_text": case["message_text"]}

    if "platform" in case:
        payload["platform"] = case["platform"]

    incident = _build_incident_from_payload(payload)
    result = analyze_incident_v4(incident)

    predicted_ids = {
        finding["rule_id"]
        for finding in result.get("findings", [])
    }

    v4_evaluation_results.append({
        **case,
        "predicted_rule_ids": predicted_ids,
    })


# ============================================================
# CALCULATE METRICS USING LABELLED CASES ONLY
# ============================================================

labelled = [
    row for row in v4_evaluation_results
    if row["expected_rule_ids"] is not None
]

ambiguous = [
    row for row in v4_evaluation_results
    if row["expected_rule_ids"] is None
]

per_rule = {
    rule_id: {"TP": 0, "FP": 0, "FN": 0, "TN": 0}
    for rule_id in sorted(V4_ALL_RULE_IDS)
}

exact_match_count = 0
mismatch_cases = []
false_positive_cases = []
false_negative_cases = []

for row in labelled:
    expected = set(row["expected_rule_ids"])
    predicted = row["predicted_rule_ids"]

    if expected == predicted:
        exact_match_count += 1
    else:
        mismatch_cases.append({
            "id": row["id"],
            "expected": sorted(expected),
            "predicted": sorted(predicted),
            "message": row["message_text"],
        })

    extra = predicted - expected
    missed = expected - predicted

    if extra:
        false_positive_cases.append({
            "id": row["id"],
            "unexpected_rules": sorted(extra),
            "message": row["message_text"],
        })

    if missed:
        false_negative_cases.append({
            "id": row["id"],
            "missed_rules": sorted(missed),
            "message": row["message_text"],
        })

    for rule_id in per_rule:
        actual_positive = rule_id in expected
        predicted_positive = rule_id in predicted

        if actual_positive and predicted_positive:
            per_rule[rule_id]["TP"] += 1
        elif predicted_positive and not actual_positive:
            per_rule[rule_id]["FP"] += 1
        elif actual_positive and not predicted_positive:
            per_rule[rule_id]["FN"] += 1
        else:
            per_rule[rule_id]["TN"] += 1


def _v4_metric(numerator, denominator):
    return numerator / denominator if denominator else None


def _v4_percent(value):
    return "N/A" if value is None else f"{100 * value:.1f}%"


total_tp = sum(x["TP"] for x in per_rule.values())
total_fp = sum(x["FP"] for x in per_rule.values())
total_fn = sum(x["FN"] for x in per_rule.values())

micro_precision = _v4_metric(total_tp, total_tp + total_fp)
micro_recall = _v4_metric(total_tp, total_tp + total_fn)

micro_f1 = (
    _v4_metric(
        2 * micro_precision * micro_recall,
        micro_precision + micro_recall,
    )
    if micro_precision is not None
    and micro_recall is not None
    and micro_precision + micro_recall
    else None
)

exact_match_rate = _v4_metric(exact_match_count, len(labelled))


# ============================================================
# REPORT RESULTS
# ============================================================

print("=" * 76)
print("SIGNALHER — V4 INTEGRATED REGRESSION RESULTS")
print("=" * 76)

print("Total cases:", len(v4_evaluation_results))
print("Labelled cases:", len(labelled))
print("Ambiguous cases excluded:", len(ambiguous))

print("\nOVERALL METRICS")
print("-" * 76)
print("Micro precision:", _v4_percent(micro_precision))
print("Micro recall:", _v4_percent(micro_recall))
print("Micro F1:", _v4_percent(micro_f1))
print("Exact set match:", _v4_percent(exact_match_rate))


print("\nPER-RULE METRICS")
print("-" * 76)
print(
    f"{'Rule ID':42} {'Precision':>10} {'Recall':>10} "
    f"{'F1':>10} {'TP':>4} {'FP':>4} {'FN':>4}"
)

for rule_id, counts in per_rule.items():
    tp, fp, fn = counts["TP"], counts["FP"], counts["FN"]

    precision = _v4_metric(tp, tp + fp)
    recall = _v4_metric(tp, tp + fn)

    f1 = (
        _v4_metric(
            2 * precision * recall,
            precision + recall,
        )
        if precision is not None
        and recall is not None
        and precision + recall
        else None
    )

    print(
        f"{rule_id:42} "
        f"{_v4_percent(precision):>10} "
        f"{_v4_percent(recall):>10} "
        f"{_v4_percent(f1):>10} "
        f"{tp:>4} {fp:>4} {fn:>4}"
    )


print("\nFALSE-POSITIVE CASES")
print("-" * 76)

if not false_positive_cases:
    print("None on this suite.")
else:
    for row in false_positive_cases:
        print(
            f"{row['id']} | Unexpected: {row['unexpected_rules']} | "
            f"{row['message']}"
        )


print("\nFALSE-NEGATIVE CASES")
print("-" * 76)

if not false_negative_cases:
    print("None on this suite.")
else:
    for row in false_negative_cases:
        print(
            f"{row['id']} | Missed: {row['missed_rules']} | "
            f"{row['message']}"
        )


print("\nEXACT-SET MISMATCHES")
print("-" * 76)

for row in mismatch_cases:
    print(
        f"{row['id']} | Expected: {row['expected']} | "
        f"Detected: {row['predicted']} | {row['message']}"
    )

if not mismatch_cases:
    print("None on this suite.")


print("\nAMBIGUOUS CASES — NOT SCORED")
print("-" * 76)

for row in ambiguous:
    print(
        f"{row['id']} | Detected: "
        f"{sorted(row['predicted_rule_ids'])} | {row['message_text']}"
    )


print("\nEVALUATION LIMITATION")
print(
    "These metrics describe this manually authored synthetic suite only. "
    "The examples were designed with knowledge of the rules and are not "
    "independent validation. Do not use these results as a claim of "
    "production accuracy."
)

print("Active SIGNAL_RULES and Flask routes remain unchanged.")

In [ ]:
# ============================================================
# SIGNALHER — V4 CONTEXT AND REGEX REFINEMENTS
# Experimental layer only. Active SIGNAL_RULES remain unchanged.
# ============================================================

import re


# ------------------------------------------------------------
# FIX 1 — Improve negative and voluntary image-request context
# ------------------------------------------------------------

def _v4_is_explicitly_negated_image_request(text):
    text_lower = text.casefold()

    negative_patterns = [
        # "You don't need to send..."
        r"\b(?:you\s+)?(?:don't|do not)\s+"
        r"(?:need to|have to)\s+"
        r"(?:send|share|show|give|provide|upload)\b",

        # "I won't ask you to share..."
        r"\b(?:i|we)\s+(?:won't|will not|don't|do not)\s+"
        r"(?:ask you to|ask you|tell you to)?\s*"
        r"(?:send|share|show|give|provide|upload)\b",

        # "Please don't share..."
        r"\b(?:don't|do not|never|should not|must not)\s+"
        r"(?:send|share|show|give|provide|upload)\b",
    ]

    if any(
        re.search(pattern, text_lower)
        for pattern in negative_patterns
    ):
        return True

    # Sharing material WITH the recipient is not inherently a threat.
    sharing_with_recipient = re.search(
        r"\b(?:share|send|show|upload|post|publish)\b"
        r"\s+(?:(?:your|the)\s+)?"
        r"(?:(?:private|intimate|personal)\s+)?"
        r"(?:photos?|pictures?|images?|videos?|nudes?|pics?)"
        r"\s+with\s+(?:you|me)\b",
        text_lower,
    )

    if sharing_with_recipient:
        return True

    # Explicit educational or fictional discussion.
    educational_patterns = [
        (
            r"\b(?:in this|this|the|a|an|our)\b"
            r".{0,100}"
            r"\b(?:training slide|training manual|training material|"
            r"safety guide|play|script|manual|article|fiction)\b"
            r".{0,100}"
            r"\b(?:example|phrase|line|quote|wording)\b"
        ),
        (
            r"\b(?:the )?(?:phrase|wording|line|expression)\b"
            r".{0,120}"
            r"\b(?:appears in|is from|comes from|is quoted in|"
            r"is used in|includes)\b"
            r".{0,80}"
            r"\b(?:training|guide|play|script|manual|fiction|article)\b"
        ),
        (
            r"\b(?:our|the|a)\s+play\b"
            r".{0,80}\b(?:includes|contains|uses)\b"
            r".{0,80}\b(?:the line|the phrase|the words)\b"
        ),
    ]

    if any(
        re.search(pattern, text_lower)
        for pattern in educational_patterns
    ):
        return True

    # Voluntary language, unless explicit pressure is also present.
    voluntary_language = re.search(
        r"\b(?:only if you want to|"
        r"if you (?:genuinely )?want to|"
        r"if you feel comfortable|"
        r"no pressure|"
        r"you can say no|"
        r"it's okay to say no|"
        r"it is okay to say no|"
        r"with your permission)\b",
        text_lower,
    )

    pressure_language = re.search(
        r"\b(?:you owe me|prove you love me|if you love me|"
        r"you must|you have to|you need to|or else|unless you|"
        r"i will expose|i'll expose|i will hurt you|i'll hurt you|"
        r"do it now)\b",
        text_lower,
    )

    return bool(voluntary_language and not pressure_language)


# ------------------------------------------------------------
# FIX 2 — Narrow the generic intimidation phrases
# ------------------------------------------------------------

_vague_rule = next(
    rule
    for rule in V4_SUPPLEMENTAL_RULES
    if rule["rule_id"] == "VAGUE_INTIMIDATION_OR_REPRISAL"
)

_vague_rule["patterns"] = (
    # Specific rejection-related intimidation remains detectable.
    r"\b(?:you will|you'll)\s+regret\s+"
    r"(?:turning me down|rejecting me|saying no)\b",

    # "I'll make you regret it." must be a complete phrase.
    # It must NOT match the prefix in "I'll make you regret that haircut."
    (
        r"\b(?:i will|i'll)\s+make you regret\s+"
        r"(?:it|this|that)(?=\s*(?:[.!?,;:]|$))"
    ),

    # Conditional intimidation, with the same complete-phrase check.
    (
        r"\bif you (?:don't|do not)\s+"
        r"(?:send|share|give|pay|reply|respond|meet|provide)\b"
        r".{0,100}"
        r"\b(?:i will|i'll)\s+make you regret\s+"
        r"(?:it|this|that)(?=\s*(?:[.!?,;:]|$))"
    ),
)


# ------------------------------------------------------------
# FIX 3 — Suppress boundary-disregard findings only when
# the same message explicitly indicates respect AND stopping.
# ------------------------------------------------------------

if "_v4_analyzer_before_context_fix" not in globals():
    _v4_analyzer_before_context_fix = analyze_incident_v4


def analyze_incident_v4(incident):
    result = _v4_analyzer_before_context_fix(incident)

    text_lower = incident.message_text.casefold()

    explicit_respect = re.search(
        r"\b(?:i|we)\s+(?:do\s+)?"
        r"(?:respect|accept)\s+"
        r"(?:your\s+)?"
        r"(?:choice|decision|refusal|answer|boundary)\b",
        text_lower,
    )

    explicit_stopping = re.search(
        r"\b(?:won't|will not|am not going to|are not going to|"
        r"will stop|am stopping)\s+"
        r"(?:meet|contact|ask|keep asking|push|pressure|"
        r"continue|keep contacting)\b",
        text_lower,
    )

    # Suppress only the boundary indicator, not unrelated findings.
    if explicit_respect and explicit_stopping:
        result["findings"] = [
            finding
            for finding in result["findings"]
            if finding["rule_id"] != "BOUNDARY_DISREGARD"
        ]

    # Recalculate status after contextual filtering.
    if result["findings"]:
        result["status"] = "potential_warning_indicators_found"
        result["message"] = (
            "Review the highlighted wording and wider context. "
            "These indicators are not proof of abuse."
        )
    else:
        result["status"] = "no_configured_pattern_matched"
        result["message"] = (
            "No configured V4 pattern matched this message. "
            "This does not mean the interaction is safe."
        )

    return result


# ============================================================
# REGRESSION TESTS FOR THE FOUR ORIGINAL FALSE POSITIVES
# ============================================================

V4_FALSE_POSITIVE_REGRESSION_TESTS = [
    {
        "id": "N09",
        "text": "You said no but I respect your choice and won't meet.",
        "forbidden_rule": "BOUNDARY_DISREGARD",
    },
    {
        "id": "N15",
        "text": "I won't ask you to share private images.",
        "forbidden_rule": "REQUEST_FOR_PRIVATE_IMAGERY",
    },
    {
        "id": "N16",
        "text": "You do not have to send private photos.",
        "forbidden_rule": "REQUEST_FOR_PRIVATE_IMAGERY",
    },
    {
        "id": "N18",
        "text": "I'll make you regret that haircut.",
        "forbidden_rule": "VAGUE_INTIMIDATION_OR_REPRISAL",
    },
]

regression_results = []

for test in V4_FALSE_POSITIVE_REGRESSION_TESTS:
    incident = _build_incident_from_payload({
        "message_text": test["text"],
    })

    result = analyze_incident_v4(incident)

    detected_ids = {
        finding["rule_id"]
        for finding in result["findings"]
    }

    passed = test["forbidden_rule"] not in detected_ids

    regression_results.append({
        **test,
        "passed": passed,
        "detected_ids": sorted(detected_ids),
    })

    print(
        f"{'PASS' if passed else 'REVIEW'} | {test['id']} | "
        f"Findings: {sorted(detected_ids)}"
    )


# Positive controls: don't remove important detection behaviours.
V4_POSITIVE_CONTROLS = [
    (
        "I'll make you regret it.",
        "VAGUE_INTIMIDATION_OR_REPRISAL",
    ),
    (
        "You will regret turning me down.",
        "VAGUE_INTIMIDATION_OR_REPRISAL",
    ),
    (
        "Send me your private photos.",
        "REQUEST_FOR_PRIVATE_IMAGERY",
    ),
    (
        "If you love me, send me your intimate photos.",
        "COERCIVE_IMAGE_REQUEST",
    ),
    (
        "You said no but meet me tomorrow.",
        "BOUNDARY_DISREGARD",
    ),
]

positive_control_results = []

for message, expected_rule in V4_POSITIVE_CONTROLS:
    incident = _build_incident_from_payload({
        "message_text": message,
    })

    result = analyze_incident_v4(incident)

    detected_ids = {
        finding["rule_id"]
        for finding in result["findings"]
    }

    passed = expected_rule in detected_ids

    positive_control_results.append(passed)

    print(
        f"{'PASS' if passed else 'REVIEW'} | Positive control | "
        f"Expected: {expected_rule} | "
        f"Detected: {sorted(detected_ids)}"
    )


print("\nFalse-positive regression checks:",
      sum(item["passed"] for item in regression_results),
      "/", len(regression_results))

print("Positive controls:",
      sum(positive_control_results),
      "/", len(positive_control_results))

assert all(item["passed"] for item in regression_results), (
    "One or more original false positives remain."
)
assert all(positive_control_results), (
    "A positive control was lost during the refinements."
)

print("\nTargeted V4 refinements passed.")
print("These tests are not independent performance validation.")
print("Active SIGNAL_RULES and Flask routes remain unchanged.")

In [ ]:
# ============================================================
# SIGNALHER — POST-FIX V4 REGRESSION EVALUATION
# Re-evaluate the existing integrated suite after recent fixes.
# Does not modify active rules or Flask routes.
# ============================================================

from collections import Counter

postfix_results = []

for case in V4_INTEGRATION_SUITE:
    payload = {"message_text": case["message_text"]}

    if "platform" in case:
        payload["platform"] = case["platform"]

    incident = _build_incident_from_payload(payload)
    result = analyze_incident_v4(incident)

    predicted = {
        finding["rule_id"]
        for finding in result.get("findings", [])
    }

    expected_raw = case["expected_rule_ids"]

    postfix_results.append({
        "id": case["id"],
        "message": case["message_text"],
        "expected": (
            None if expected_raw is None else set(expected_raw)
        ),
        "predicted": predicted,
    })


labelled = [
    row for row in postfix_results
    if row["expected"] is not None
]

ambiguous = [
    row for row in postfix_results
    if row["expected"] is None
]

rule_ids = sorted(
    {rule.rule_id for rule in SIGNAL_RULES}
    | {
        rule["rule_id"]
        for rule in V4_SUPPLEMENTAL_RULES
    }
)

confusion = {
    rule_id: {"TP": 0, "FP": 0, "FN": 0, "TN": 0}
    for rule_id in rule_ids
}

false_positives = []
false_negatives = []
mismatches = []
exact_matches = 0

for row in labelled:
    expected = row["expected"]
    predicted = row["predicted"]

    if expected == predicted:
        exact_matches += 1
    else:
        mismatches.append(row)

    extra = predicted - expected
    missed = expected - predicted

    if extra:
        false_positives.append({
            "id": row["id"],
            "unexpected": sorted(extra),
            "message": row["message"],
        })

    if missed:
        false_negatives.append({
            "id": row["id"],
            "missed": sorted(missed),
            "message": row["message"],
        })

    for rule_id in rule_ids:
        actual_positive = rule_id in expected
        predicted_positive = rule_id in predicted

        if actual_positive and predicted_positive:
            confusion[rule_id]["TP"] += 1
        elif predicted_positive and not actual_positive:
            confusion[rule_id]["FP"] += 1
        elif actual_positive and not predicted_positive:
            confusion[rule_id]["FN"] += 1
        else:
            confusion[rule_id]["TN"] += 1


def metric_ratio(numerator, denominator):
    return numerator / denominator if denominator else None


def metric_percent(value):
    return "N/A" if value is None else f"{100 * value:.1f}%"


tp = sum(item["TP"] for item in confusion.values())
fp = sum(item["FP"] for item in confusion.values())
fn = sum(item["FN"] for item in confusion.values())

precision = metric_ratio(tp, tp + fp)
recall = metric_ratio(tp, tp + fn)

f1 = (
    metric_ratio(2 * precision * recall, precision + recall)
    if precision is not None
    and recall is not None
    and precision + recall > 0
    else None
)

exact_match = metric_ratio(exact_matches, len(labelled))


print("=" * 76)
print("SIGNALHER — UPDATED V4 REGRESSION RESULTS")
print("=" * 76)

print("Total cases:", len(postfix_results))
print("Labelled cases:", len(labelled))
print("Ambiguous cases excluded:", len(ambiguous))

print("\nOVERALL METRICS")
print("-" * 76)
print("Micro precision:", metric_percent(precision))
print("Micro recall:", metric_percent(recall))
print("Micro F1:", metric_percent(f1))
print("Exact set match:", metric_percent(exact_match))


print("\nPER-RULE METRICS")
print("-" * 76)
print(
    f"{'Rule ID':42} {'Precision':>10} "
    f"{'Recall':>10} {'F1':>10} {'TP':>4} {'FP':>4} {'FN':>4}"
)

for rule_id, counts in confusion.items():
    rule_tp = counts["TP"]
    rule_fp = counts["FP"]
    rule_fn = counts["FN"]

    rule_precision = metric_ratio(
        rule_tp, rule_tp + rule_fp
    )
    rule_recall = metric_ratio(
        rule_tp, rule_tp + rule_fn
    )

    rule_f1 = (
        metric_ratio(
            2 * rule_precision * rule_recall,
            rule_precision + rule_recall,
        )
        if rule_precision is not None
        and rule_recall is not None
        and rule_precision + rule_recall > 0
        else None
    )

    print(
        f"{rule_id:42} "
        f"{metric_percent(rule_precision):>10} "
        f"{metric_percent(rule_recall):>10} "
        f"{metric_percent(rule_f1):>10} "
        f"{rule_tp:>4} {rule_fp:>4} {rule_fn:>4}"
    )


print("\nFALSE POSITIVES")
print("-" * 76)

for item in false_positives:
    print(
        f"{item['id']} | {item['unexpected']} | {item['message']}"
    )

if not false_positives:
    print("None on this integrated regression suite.")


print("\nFALSE NEGATIVES")
print("-" * 76)

for item in false_negatives:
    print(
        f"{item['id']} | {item['missed']} | {item['message']}"
    )

if not false_negatives:
    print("None on this integrated regression suite.")


print("\nAMBIGUOUS CASES — EXCLUDED FROM METRICS")
print("-" * 76)

for row in ambiguous:
    print(f"{row['id']} | {sorted(row['predicted'])} | {row['message']}")


print("\nEVALUATION NOTE")
print(
    "These results describe the same manually authored synthetic suite "
    "used during development. They measure regression behaviour, not "
    "independent real-world performance."
)

print("Active SIGNAL_RULES and Flask routes remain unchanged.")

In [ ]:
%pip install pandas openpyxl

In [ ]:
import pandas as pd
import openpyxl

print("pandas version:", pd.__version__)
print("openpyxl version:", openpyxl.__version__)
print("Dataset inspection dependencies: PASSED")

In [ ]:
# ============================================================
# SIGNALHER — EXTERNAL DATASET INTAKE AUDIT
# Read-only inspection of BanglaHarassNet Excel files.
# ============================================================

from pathlib import Path
import pandas as pd

# Notebook is inside the project subfolder ending in .ipynb.
cwd = Path.cwd()

if (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
elif (cwd / "data").exists():
    PROJECT_ROOT = cwd
else:
    PROJECT_ROOT = cwd.parent

DATASET_DIR = (
    PROJECT_ROOT
    / "data"
    / "external"
    / "BanglaHarassNet"
)

DATASET_DIR.mkdir(parents=True, exist_ok=True)

excel_files = sorted(
    path for path in DATASET_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in {".xlsx", ".xls"}
    and not path.name.startswith("~$")
)

print("Project root:", PROJECT_ROOT)
print("Dataset folder:", DATASET_DIR)
print("Excel files found:", len(excel_files))

if not excel_files:
    print(
        "\nNo Excel files found yet.\n"
        "Download BanglaHarassNet from:\n"
        "https://data.mendeley.com/datasets/cw6jrvxmhf/1\n"
        "Extract its Excel files into the folder shown above, "
        "then rerun this cell."
    )

else:
    total_rows = 0
    total_duplicate_rows = 0

    for file_path in excel_files:
        print("\n" + "=" * 70)
        print("FILE:", file_path.name)
        print("Size (KB):", round(file_path.stat().st_size / 1024, 1))

        try:
            workbook = pd.ExcelFile(file_path)

            for sheet_name in workbook.sheet_names:
                df = pd.read_excel(
                    workbook,
                    sheet_name=sheet_name,
                )

                total_rows += len(df)
                duplicate_count = int(df.duplicated().sum())
                total_duplicate_rows += duplicate_count

                print("\nSheet:", sheet_name)
                print("Rows:", len(df))
                print("Columns:", len(df.columns))
                print("Duplicate rows:", duplicate_count)
                print("Column names:", list(df.columns))

                # Identify likely message/text columns by header only.
                text_headers = [
                    str(col)
                    for col in df.columns
                    if any(
                        keyword in str(col).casefold()
                        for keyword in (
                            "text", "message", "comment",
                            "sentence", "tweet", "post",
                            "content", "utterance", "body",
                        )
                    )
                ]

                label_headers = [
                    str(col)
                    for col in df.columns
                    if any(
                        keyword in str(col).casefold()
                        for keyword in (
                            "label", "class", "category",
                            "type", "target",
                        )
                    )
                ]

                print("Possible text columns:", text_headers)
                print("Possible label columns:", label_headers)

                if text_headers:
                    for column in text_headers:
                        missing = int(df[column].isna().sum())
                        blank = int(
                            df[column]
                            .astype("string")
                            .str.strip()
                            .eq("")
                            .fillna(False)
                            .sum()
                        )

                        print(
                            f"  {column!r}: missing={missing}, "
                            f"blank={blank}"
                        )

                # Report label distributions without showing text.
                for column in label_headers:
                    counts = (
                        df[column]
                        .fillna("[missing]")
                        .astype(str)
                        .value_counts(dropna=False)
                        .head(20)
                        .to_dict()
                    )

                    print(
                        f"  Label candidates in {column!r}: {counts}"
                    )

        except ImportError as exc:
            print(
                "Missing Excel-reading dependency. "
                "Install pandas and openpyxl in this environment."
            )
            raise exc

    print("\n" + "=" * 70)
    print("DATASET INTAKE SUMMARY")
    print("=" * 70)
    print("Files inspected:", len(excel_files))
    print("Total rows across inspected sheets:", total_rows)
    print("Duplicate rows across inspected sheets:", total_duplicate_rows)
    print(
        "\nNo message text was printed. This audit does not train a model "
        "or change SignalHer's active backend."
    )

In [ ]:
##########################################################################################################################
##########################################################################################################################

In [ ]:
# ============================================================
# SIGNALHER — INSPECT THE ACTUALLY EXTRACTED DATASET
# Read-only: does not move or modify any files.
# ============================================================

from pathlib import Path
import pandas as pd

cwd = Path.cwd()

if (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
elif (cwd / "data").exists():
    PROJECT_ROOT = cwd
else:
    raise FileNotFoundError(
        "Could not locate the project's data folder."
    )

DATA_ROOT = PROJECT_ROOT / "data"

# Find the downloaded dataset folder, regardless of its exact name.
dataset_folders = [
    path
    for path in DATA_ROOT.iterdir()
    if path.is_dir() and "banglaharassnet" in path.name.casefold()
]

print("Dataset folder candidates:")
for folder in dataset_folders:
    print("-", folder)

if not dataset_folders:
    raise FileNotFoundError(
        "No BanglaHarassNet folder found directly inside the data folder."
    )

for dataset_folder in dataset_folders:
    print("\n" + "=" * 70)
    print("INSPECTING:", dataset_folder.name)
    print("=" * 70)

    files = [
        path for path in dataset_folder.rglob("*")
        if path.is_file()
    ]

    print("Total files:", len(files))

    # Show filenames only—not the actual message contents.
    for path in files:
        print(
            "File:",
            path.relative_to(dataset_folder),
            "| Size (KB):",
            round(path.stat().st_size / 1024, 1),
        )

    excel_files = [
        path for path in files
        if path.suffix.casefold() in {".xlsx", ".xls", ".xlsm"}
        and not path.name.startswith("~$")
    ]

    print("\nExcel files found:", len(excel_files))

    for file_path in excel_files:
        try:
            workbook = pd.ExcelFile(file_path)

            for sheet_name in workbook.sheet_names:
                df = pd.read_excel(
                    workbook,
                    sheet_name=sheet_name,
                )

                print("\nFile:", file_path.name)
                print("Sheet:", sheet_name)
                print("Rows:", len(df))
                print("Columns:", list(df.columns))
                print("Duplicate rows:", int(df.duplicated().sum()))

                # Report likely label distributions without exposing text.
                for column in df.columns:
                    column_name = str(column).casefold()

                    if any(
                        word in column_name
                        for word in ("label", "class", "category", "target")
                    ):
                        print(
                            "Label distribution for",
                            repr(column),
                            ":",
                            df[column]
                            .fillna("[missing]")
                            .astype(str)
                            .value_counts()
                            .head(20)
                            .to_dict(),
                        )

        except Exception as exc:
            print(
                "Could not inspect",
                file_path.name,
                "| Error type:",
                type(exc).__name__,
                "| Details:",
                str(exc),
            )

print("\nDataset inspection completed. No files were modified.")

In [ ]:
# ============================================================
# SIGNALHER — WORKBOOK STRUCTURE AND DUPLICATE-FILE AUDIT
# Read-only: does not alter files, labels, rules, or the API.
# ============================================================

from pathlib import Path
import hashlib
import warnings
import pandas as pd

cwd = Path.cwd()

if (cwd.parent / "data").is_dir():
    PROJECT_ROOT = cwd.parent
elif (cwd / "data").is_dir():
    PROJECT_ROOT = cwd
else:
    raise FileNotFoundError("Could not locate the project data folder.")

DATA_ROOT = PROJECT_ROOT / "data"

dataset_candidates = [
    path for path in DATA_ROOT.iterdir()
    if path.is_dir() and "banglaharassnet" in path.name.casefold()
]

if not dataset_candidates:
    raise FileNotFoundError(
        f"No BanglaHarassNet dataset folder found inside {DATA_ROOT}"
    )

DATASET_DIR = dataset_candidates[0]

# Use only spreadsheets directly in the selected dataset folder.
# Nested copies are inspected separately but never counted twice here.
top_level_excels = sorted(DATASET_DIR.glob("*.xlsx"))

print("=" * 72)
print("1. DATASET FILES")
print("=" * 72)
print("Dataset directory:", DATASET_DIR)
print("Top-level Excel files:", len(top_level_excels))

for path in top_level_excels:
    print(f"- {path.name} | {path.stat().st_size:,} bytes")


# ------------------------------------------------------------
# Inspect the small metadata CSV
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("2. DATASET METADATA")
print("=" * 72)

metadata_files = sorted(DATASET_DIR.glob("Metadata*.csv"))

if metadata_files:
    metadata_path = metadata_files[0]

    try:
        metadata_df = pd.read_csv(
            metadata_path,
            encoding="utf-8-sig",
        )

        print("Metadata file:", metadata_path.name)
        print("Shape:", metadata_df.shape)
        print(metadata_df.to_string(index=False))

    except Exception as exc:
        print("Could not parse metadata CSV:", type(exc).__name__)
        print(
            metadata_path.read_text(
                encoding="utf-8-sig",
                errors="replace",
            )
        )
else:
    print("No metadata CSV found at the top level.")


# ------------------------------------------------------------
# Check whether nested spreadsheets are exact file copies
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("3. NESTED-COPY CHECK")
print("=" * 72)

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


nested_excel_files = [
    path for path in DATASET_DIR.rglob("*.xlsx")
    if path.parent != DATASET_DIR
]

if not nested_excel_files:
    print("No nested Excel copies found.")
else:
    for nested_path in nested_excel_files:
        same_name = [
            item for item in top_level_excels
            if item.name.casefold() == nested_path.name.casefold()
        ]

        exact_copy = any(
            sha256_file(item) == sha256_file(nested_path)
            for item in same_name
        )

        print(
            f"{nested_path.relative_to(DATASET_DIR)} | "
            f"identical to a top-level file: {exact_copy}"
        )


# ------------------------------------------------------------
# Inspect raw worksheet layout without displaying message text
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("4. RAW WORKSHEET STRUCTURE")
print("=" * 72)

HEADER_TERMS = (
    "text", "comment", "message", "sentence", "tweet", "post",
    "content", "language", "platform", "source", "label",
    "category", "severity", "gender", "profession", "metadata",
    "english", "bangla", "banglish", "id", "annotation",
)

for path in top_level_excels:
    print("\nFILE:", path.name)

    try:
        with warnings.catch_warnings(record=True) as captured_warnings:
            warnings.simplefilter("always")

            raw = pd.read_excel(
                path,
                header=None,
                dtype=object,
                engine="openpyxl",
            )

        print("Raw worksheet dimensions:", raw.shape)
        print("Read-time warnings:", len(captured_warnings))

        # Show the occupied positions in the first six rows.
        # Text values are masked unless they look like schema labels.
        for row_index in range(min(6, len(raw))):
            row = raw.iloc[row_index]

            occupied = [
                (column_index, value)
                for column_index, value in enumerate(row.tolist())
                if pd.notna(value)
                and str(value).strip() != ""
            ]

            print(
                f"  Row {row_index + 1}: "
                f"{len(occupied)} non-empty cell(s); "
                f"column positions={[index for index, _ in occupied]}"
            )

            previews = []

            for column_index, value in occupied[:16]:
                if isinstance(value, str):
                    cleaned = " ".join(value.split())
                    candidate = cleaned.casefold()

                    looks_like_header = any(
                        term in candidate for term in HEADER_TERMS
                    )

                    if looks_like_header and len(cleaned) <= 100:
                        description = repr(cleaned)
                    else:
                        description = f"<text length={len(cleaned)}>"

                elif isinstance(value, (int, float, bool)):
                    description = (
                        f"<{type(value).__name__}: {value}>"
                    )
                else:
                    description = f"<{type(value).__name__}>"

                previews.append(f"C{column_index + 1}={description}")

            print("    ", " | ".join(previews))

        # Show which columns are actually populated in the workbook.
        nonempty_counts = raw.notna().sum()

        print("  Non-empty cells by column:")
        print({
            f"C{index + 1}": int(count)
            for index, count in enumerate(nonempty_counts)
            if count > 0
        })

        # Test duplicates without counting a single-cell title row
        # as a data record if the workbook has one.
        data = raw.copy()

        first_row_values = [
            value for value in data.iloc[0].tolist()
            if pd.notna(value) and str(value).strip() != ""
        ] if len(data) else []

        if len(first_row_values) == 1:
            title = str(first_row_values[0]).casefold()

            if any(
                category in title
                for category in (
                    "cyberbullying", "hate speech", "neutral",
                    "sexual harassment", "trolling",
                )
            ):
                data = data.iloc[1:].copy()
                title_row_removed_for_count = True
            else:
                title_row_removed_for_count = False
        else:
            title_row_removed_for_count = False

        # Exclude fully empty rows from record counts.
        data = data.dropna(how="all")

        print("  Candidate data rows:", len(data))
        print("  Single-cell title row excluded:", title_row_removed_for_count)
        print("  Exact duplicate data rows:", int(data.duplicated().sum()))

    except Exception as exc:
        print(
            "  Inspection failed:",
            type(exc).__name__,
            str(exc),
        )


print("\n" + "=" * 72)
print("AUDIT FINISHED")
print("=" * 72)
print(
    "No message contents were printed, no files were modified, "
    "and no SignalHer rules were changed."
)

In [ ]:
# ============================================================
# SIGNALHER — USABLE DATA AND LABEL QUALITY AUDIT
# Read-only inspection of the five top-level Excel workbooks.
# ============================================================

from pathlib import Path
from collections import Counter
import warnings
import pandas as pd


# ------------------------------------------------------------
# 1. Locate the dataset
# ------------------------------------------------------------

cwd = Path.cwd()

if (cwd.parent / "data").is_dir():
    PROJECT_ROOT = cwd.parent
elif (cwd / "data").is_dir():
    PROJECT_ROOT = cwd
else:
    raise FileNotFoundError("Could not locate the project data folder.")

DATA_ROOT = PROJECT_ROOT / "data"

dataset_candidates = [
    path for path in DATA_ROOT.iterdir()
    if path.is_dir()
    and "banglaharassnet" in path.name.casefold()
]

if not dataset_candidates:
    raise FileNotFoundError("BanglaHarassNet folder not found.")

DATASET_DIR = dataset_candidates[0]

# Top-level copies only; exclude nested duplicate copies.
workbooks = sorted(
    path for path in DATASET_DIR.glob("*.xlsx")
    if not path.name.startswith("~$")
)

if not workbooks:
    raise FileNotFoundError(
        f"No top-level Excel workbooks found in {DATASET_DIR}"
    )


# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------

def normalise_column_name(name):
    """Make column matching independent of spaces and punctuation."""
    return "".join(
        char.casefold()
        for char in str(name)
        if char.isalnum()
    )


def nonempty_mask(series):
    """Identify values that are not NA and not blank strings."""
    return (
        series.notna()
        & series.astype("string").str.strip().ne("").fillna(False)
    )


def find_column(df, possible_names):
    """Find a column using normalised candidate names."""
    lookup = {
        normalise_column_name(column): column
        for column in df.columns
    }

    for candidate in possible_names:
        match = lookup.get(normalise_column_name(candidate))
        if match is not None:
            return match

    return None


def distribution_without_message_text(df, column):
    """Return value counts for a metadata column, not message text."""
    if column is None:
        return {}

    values = df.loc[nonempty_mask(df[column]), column]
    return {
        str(key): int(value)
        for key, value in values.astype(str).value_counts().head(15).items()
    }


# ------------------------------------------------------------
# 3. Inspect each workbook using Excel row 2 as the header
# ------------------------------------------------------------

dataset_summaries = []

print("=" * 76)
print("SIGNALHER — DATA QUALITY AUDIT")
print("=" * 76)
print("Dataset directory:", DATASET_DIR)
print("Top-level workbooks:", len(workbooks))
print("Excel header row: 2")
print("Nested duplicate files: excluded")


for file_path in workbooks:
    print("\n" + "-" * 76)
    print("WORKBOOK:", file_path.name)

    try:
        # Excel row 1 is the category title.
        # Excel row 2 contains the real column names.
        with warnings.catch_warnings(record=True) as captured_warnings:
            warnings.simplefilter("always")

            df = pd.read_excel(
                file_path,
                header=1,
                dtype=object,
                engine="openpyxl",
            )

    except Exception as exc:
        print("READ FAILURE:", type(exc).__name__, str(exc))
        continue

    # Remove fully empty rows; keep all other records for auditing.
    df = df.dropna(how="all").copy()

    text_col = find_column(df, ["Text_Content", "Text", "Message"])
    romanized_col = find_column(
        df,
        ["Romanized_Text", "Romanized Text", "Transliterated_Text"],
    )
    post_id_col = find_column(df, ["Post_ID", "ID"])
    platform_col = find_column(df, ["Platform", "Source_Platform"])
    language_col = find_column(df, ["Language", "Text_Language"])
    harassment_label_col = find_column(
        df,
        ["Harassment_Label", "Label"],
    )
    harassment_type_col = find_column(
        df,
        ["Harassment_Type", "Category", "Class"],
    )
    severity_col = find_column(
        df,
        ["Severity_Level", "Severity"],
    )
    gender_col = find_column(df, ["Gender_of_Target"])
    content_type_col = find_column(df, ["Content_Type"])

    # A row is usable as a text record if the primary text field
    # contains nonblank content. Romanised text is audited separately.
    if text_col is not None:
        text_mask = nonempty_mask(df[text_col])
    else:
        text_mask = pd.Series(False, index=df.index)

    text_records = df.loc[text_mask].copy()

    if romanized_col is not None:
        romanized_mask = nonempty_mask(df[romanized_col])
    else:
        romanized_mask = pd.Series(False, index=df.index)

    if post_id_col is not None:
        id_mask = nonempty_mask(df[post_id_col])
    else:
        id_mask = pd.Series(False, index=df.index)

    text_count = int(text_mask.sum())
    romanized_count = int(romanized_mask.sum())
    id_count = int(id_mask.sum())

    id_only_mask = id_mask & ~text_mask
    text_without_id_mask = text_mask & ~id_mask

    # Recalculate repeated text only among populated text records.
    if text_col is not None and text_count:
        normalised_text = (
            text_records[text_col]
            .astype("string")
            .str.strip()
            .str.casefold()
        )
        duplicate_text_rows = int(normalised_text.duplicated().sum())
        unique_text_count = int(normalised_text.nunique())
    else:
        duplicate_text_rows = 0
        unique_text_count = 0

    # Duplicate IDs are checked only among records containing text.
    if post_id_col is not None and text_count:
        populated_ids = text_records[post_id_col]
        populated_ids = populated_ids[nonempty_mask(populated_ids)]
        duplicate_id_rows = int(populated_ids.duplicated().sum())
    else:
        duplicate_id_rows = 0

    summary = {
        "file": file_path.name,
        "worksheet_rows": len(df),
        "text_records": text_count,
        "romanized_records": romanized_count,
        "records_with_ids": id_count,
        "id_only_rows": int(id_only_mask.sum()),
        "text_rows_missing_id": int(text_without_id_mask.sum()),
        "duplicate_text_rows": duplicate_text_rows,
        "unique_text_values": unique_text_count,
        "duplicate_ids_among_text_rows": duplicate_id_rows,
        "read_warnings": len(captured_warnings),
    }

    dataset_summaries.append(summary)

    for key, value in summary.items():
        if key != "file":
            print(f"{key}: {value}")

    print("\nActual column headers:")
    print(list(df.columns))

    # Print only the distributions of metadata fields.
    metadata_columns = {
        "Platform": platform_col,
        "Language": language_col,
        "Harassment_Label": harassment_label_col,
        "Harassment_Type": harassment_type_col,
        "Severity_Level": severity_col,
        "Gender_of_Target": gender_col,
        "Content_Type": content_type_col,
    }

    print("\nMetadata distributions:")
    for label, column in metadata_columns.items():
        if column is not None:
            print(
                f"  {label}: "
                f"{distribution_without_message_text(df, column)}"
            )

    if text_col is None:
        print(
            "\nWARNING: No recognised primary text column was found. "
            "Do not use this workbook for training until it is inspected."
        )

    if id_only_mask.sum() > 0:
        print(
            "\nWARNING: Some rows have an ID but no primary text. "
            "These are excluded from usable-text counts."
        )

    if captured_warnings:
        print(
            "\nRead warnings:",
            Counter(str(item.category.__name__) for item in captured_warnings),
        )


# ------------------------------------------------------------
# 4. Combined summary
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("COMBINED WORKBOOK SUMMARY")
print("=" * 76)

if dataset_summaries:
    summary_df = pd.DataFrame(dataset_summaries)

    print(
        summary_df[
            [
                "file",
                "worksheet_rows",
                "text_records",
                "romanized_records",
                "id_only_rows",
                "text_rows_missing_id",
                "duplicate_text_rows",
                "duplicate_ids_among_text_rows",
            ]
        ].to_string(index=False)
    )

    print("\nTotal worksheet rows:", int(summary_df["worksheet_rows"].sum()))
    print("Total populated text records:", int(summary_df["text_records"].sum()))
    print("Total ID-only rows:", int(summary_df["id_only_rows"].sum()))
    print(
        "Total duplicate text occurrences:",
        int(summary_df["duplicate_text_rows"].sum()),
    )
    print(
        "\nThese are counts across separate category workbooks. "
        "The combined text total may still include repeated content "
        "across different files."
    )

print("\nAudit completed. No files were modified.")
print("No message text was printed.")
print("No model was trained and no SignalHer rules were changed.")

In [ ]:
# ============================================================
# SIGNALHER — CROSS-CATEGORY DUPLICATE AND LABEL AUDIT
# Read-only analysis of the five top-level workbooks.
# ============================================================

from pathlib import Path
from collections import Counter
import warnings
import pandas as pd
import re


# ------------------------------------------------------------
# 1. Locate the dataset directory
# ------------------------------------------------------------

cwd = Path.cwd()

if (cwd.parent / "data").is_dir():
    PROJECT_ROOT = cwd.parent
elif (cwd / "data").is_dir():
    PROJECT_ROOT = cwd
else:
    raise FileNotFoundError("Could not locate the project data folder.")

DATA_ROOT = PROJECT_ROOT / "data"

dataset_candidates = [
    path for path in DATA_ROOT.iterdir()
    if path.is_dir() and "banglaharassnet" in path.name.casefold()
]

if not dataset_candidates:
    raise FileNotFoundError("BanglaHarassNet folder not found.")

DATASET_DIR = dataset_candidates[0]

WORKBOOK_CATEGORY_MAP = {
    "cyberbullying.xlsx": "cyberbullying",
    "hate speech.xlsx": "hate_speech",
    "neutral .xlsx": "neutral",
    "sexual harassment.xlsx": "sexual_harassment",
    "trolling.xlsx": "trolling",
}

EXPECTED_SEVERITY = {
    "cyberbullying": 2,
    "hate_speech": 3,
    "neutral": 0,
    "sexual_harassment": 4,
    "trolling": 1,
}

workbooks = sorted(
    path for path in DATASET_DIR.glob("*.xlsx")
    if not path.name.startswith("~$")
)


# ------------------------------------------------------------
# 2. Normalisation and safe column lookup
# ------------------------------------------------------------

def _normalise_label(value):
    if pd.isna(value):
        return None

    value = " ".join(str(value).strip().casefold().split())
    return value if value else None


def _normalise_message(value):
    if pd.isna(value):
        return None

    # Collapse whitespace and ignore case for duplicate detection.
    # No stemming, translation, or semantic rewriting is applied.
    return re.sub(r"\s+", " ", str(value)).strip().casefold() or None


def _find_column(columns, target_name):
    target = re.sub(r"[^a-z0-9]", "", target_name.casefold())

    for column in columns:
        candidate = re.sub(r"[^a-z0-9]", "", str(column).casefold())

        if candidate == target:
            return column

    return None


# ------------------------------------------------------------
# 3. Load only the useful fields from populated text rows
# ------------------------------------------------------------

records = []
workbook_summaries = []

print("=" * 76)
print("SIGNALHER — LABEL AND DUPLICATE INTEGRITY AUDIT")
print("=" * 76)

for path in workbooks:
    workbook_key = path.name.casefold()
    category = WORKBOOK_CATEGORY_MAP.get(workbook_key)

    if category is None:
        print("Skipping unrecognised workbook:", path.name)
        continue

    # The first Excel row is a title, and the second is the header.
    # Suppress known workbook metadata warnings for this read-only audit.
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            category=UserWarning,
            module="openpyxl",
        )

        df = pd.read_excel(
            path,
            header=1,
            dtype=object,
            engine="openpyxl",
        )

    text_col = _find_column(df.columns, "Text_Content")

    if text_col is None:
        print("No Text_Content column in:", path.name)
        continue

    # Read only the text-bearing records.
    text_mask = (
        df[text_col].notna()
        & df[text_col].astype("string").str.strip().ne("").fillna(False)
    )

    text_df = df.loc[text_mask].copy()

    columns = {
        field: _find_column(df.columns, field)
        for field in [
            "Romanized_Text",
            "Language",
            "Platform",
            "Harassment_Label",
            "Harassment_Type",
            "Severity_Level",
        ]
    }

    current_records = []

    for _, row in text_df.iterrows():
        text = str(row[text_col]).strip()
        normalised_text = _normalise_message(text)

        if not normalised_text:
            continue

        record = {
            "workbook": path.name,
            "workbook_category": category,
            "message_text": text,
            "normalised_text": normalised_text,
        }

        for field, column in columns.items():
            value = row[column] if column is not None else None
            record[field.lower()] = _normalise_label(value)

        severity_raw = (
            row[columns["Severity_Level"]]
            if columns["Severity_Level"] is not None
            else None
        )

        try:
            record["severity_numeric"] = (
                float(severity_raw)
                if pd.notna(severity_raw)
                else None
            )
        except (TypeError, ValueError):
            record["severity_numeric"] = None

        current_records.append(record)

    records.extend(current_records)

    normalized_texts = [
        item["normalised_text"]
        for item in current_records
    ]

    workbook_summaries.append({
        "workbook": path.name,
        "category": category,
        "text_records": len(current_records),
        "within_workbook_duplicate_texts": (
            len(normalized_texts) - len(set(normalized_texts))
        ),
    })

    print(
        f"{path.name}: {len(current_records)} populated text records"
    )


if not records:
    raise ValueError("No usable text records were loaded.")

data = pd.DataFrame(records)


# ------------------------------------------------------------
# 4. Check for repeated texts within and across categories
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("CROSS-CATEGORY TEXT OVERLAP")
print("=" * 76)

text_groups = data.groupby("normalised_text", sort=False)

cross_category_groups = []
cross_category_conflicts = []

for normalized_text, group in text_groups:
    categories = sorted(group["workbook_category"].unique())

    if len(categories) > 1:
        cross_category_groups.append({
            "normalized_text": normalized_text,
            "categories": categories,
            "row_count": len(group),
        })

        # Keep this as a potential conflict: categories may overlap,
        # and their labels aren't necessarily mutually exclusive.
        cross_category_conflicts.append({
            "categories": tuple(categories),
            "row_count": len(group),
        })


print("Populated text records:", len(data))
print("Unique normalized text strings:", data["normalised_text"].nunique())
print("Repeated text groups within/across the dataset:",
      int(data["normalised_text"].duplicated().sum()))
print("Unique texts appearing in multiple workbook categories:",
      len(cross_category_groups))

if cross_category_groups:
    category_combinations = Counter(
        item["categories"] for item in cross_category_conflicts
    )

    print("\nMost common cross-category combinations:")
    for combination, count in category_combinations.most_common(15):
        print(f"{combination}: {count} repeated text group(s)")

    repeated_rows_in_conflicting_groups = sum(
        item["row_count"] for item in cross_category_conflicts
    )

    print(
        "\nTotal rows belonging to cross-category repeated-text groups:",
        repeated_rows_in_conflicting_groups,
    )
else:
    print("No repeated text was found across workbook categories.")


# ------------------------------------------------------------
# 5. Check workbook category against embedded Harassment_Type
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("WORKBOOK CATEGORY VS EMBEDDED LABEL")
print("=" * 76)

type_mismatches = []

for _, row in data.iterrows():
    embedded_type = row["harassment_type"]

    # Normalise harmless differences in spaces/case.
    if embedded_type is None:
        continue

    if embedded_type != row["workbook_category"].replace("_", " "):
        type_mismatches.append({
            "workbook_category": row["workbook_category"],
            "embedded_type": embedded_type,
        })


print("Rows with a nonmatching embedded Harassment_Type:",
      len(type_mismatches))

if type_mismatches:
    mismatch_counts = Counter(
        (
            item["workbook_category"],
            item["embedded_type"],
        )
        for item in type_mismatches
    )

    print("Most frequent mismatch combinations:")
    for combination, count in mismatch_counts.most_common(15):
        print(f"{combination}: {count}")


# ------------------------------------------------------------
# 6. Check Harassment_Label against the workbook-level category
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("WORKBOOK CATEGORY VS HARASSMENT_LABEL")
print("=" * 76)

expected_label = {
    category: ("no" if category == "neutral" else "yes")
    for category in WORKBOOK_CATEGORY_MAP.values()
}

label_mismatches = []

for _, row in data.iterrows():
    category = row["workbook_category"]
    observed = row["harassment_label"]

    if observed is None:
        continue

    expected = expected_label[category]

    if observed != expected:
        label_mismatches.append({
            "workbook_category": category,
            "observed_label": observed,
            "expected_label": expected,
        })


print("Rows with unexpected Harassment_Label:",
      len(label_mismatches))

if label_mismatches:
    print(
        "Mismatch counts:",
        dict(Counter(
            (
                item["workbook_category"],
                item["observed_label"],
            )
            for item in label_mismatches
        )),
    )


# ------------------------------------------------------------
# 7. Check severity consistency
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("WORKBOOK CATEGORY VS SEVERITY_LEVEL")
print("=" * 76)

severity_mismatches = []
missing_severity = 0

for _, row in data.iterrows():
    observed = row["severity_numeric"]

    if observed is None:
        missing_severity += 1
        continue

    expected = EXPECTED_SEVERITY[row["workbook_category"]]

    if observed != expected:
        severity_mismatches.append({
            "workbook_category": row["workbook_category"],
            "observed_severity": observed,
            "expected_severity": expected,
        })


print("Rows with missing/unreadable severity:", missing_severity)
print("Rows with unexpected severity:", len(severity_mismatches))

if severity_mismatches:
    print(
        "Severity mismatch counts:",
        dict(Counter(
            (
                item["workbook_category"],
                item["observed_severity"],
            )
            for item in severity_mismatches
        )),
    )


# ------------------------------------------------------------
# 8. Normalise language metadata and report the actual distribution
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("NORMALISED LANGUAGE DISTRIBUTION")
print("=" * 76)

language_aliases = {
    "bangla": "Bangla",
    "banglish": "Banglish",
    "english": "English",
    "mixed": "Mixed",
}

data["language_normalized"] = data["language"].map(
    lambda value: (
        language_aliases.get(value, value.title())
        if value is not None else "Unknown"
    )
)

print(
    data["language_normalized"].value_counts(dropna=False).to_dict()
)

print("\nLanguage distribution by workbook category:")
print(
    pd.crosstab(
        data["workbook_category"],
        data["language_normalized"],
    ).to_string()
)


# ------------------------------------------------------------
# 9. Final summary — no text content is printed
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("AUDIT SUMMARY")
print("=" * 76)

print("Usable records loaded:", len(data))
print("Unique normalized messages:", data["normalised_text"].nunique())
print("Cross-category repeated-text groups:", len(cross_category_groups))
print("Embedded category mismatches:", len(type_mismatches))
print("Harassment_Label mismatches:", len(label_mismatches))
print("Severity mismatches:", len(severity_mismatches))
print("Missing/unreadable severity values:", missing_severity)

print(
    "\nNo message text was printed, no files were modified, "
    "and no model was trained."
)
print(
    "The dataframe 'data' is available in memory for further inspection. "
    "Do not treat workbook categories as SignalHer-specific threat labels."
)

In [ ]:
def normalize_language(value):
    if pd.isna(value):
        return "Unknown"

    value = str(value).strip().casefold()

    if not value:
        return "Unknown"

    return language_aliases.get(value, value.title())


data["language_normalized"] = data["language"].map(normalize_language)

print("\n" + "=" * 76)
print("NORMALISED LANGUAGE DISTRIBUTION")
print("=" * 76)
print(data["language_normalized"].value_counts(dropna=False).to_string())

In [ ]:
%pip install -q scikit-learn joblib

import re
import json
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

# ---------------------------------------------------------
# 1. Resolve the existing audit dataframe's column names
# ---------------------------------------------------------
def resolve_column(candidates, required=True):
    normalised_columns = {
        re.sub(r"[^a-z0-9]+", "_", str(col).casefold()).strip("_"): col
        for col in data.columns
    }

    for candidate in candidates:
        key = re.sub(
            r"[^a-z0-9]+", "_", candidate.casefold()
        ).strip("_")
        if key in normalised_columns:
            return normalised_columns[key]

    if required:
        raise KeyError(
            f"Could not find any of {candidates}. "
            f"Available columns: {list(data.columns)}"
        )
    return None


text_col = resolve_column(
    ["text_content", "message_text", "text", "content"]
)
category_col = resolve_column(
    ["workbook_category", "dataset_category", "category"]
)
normalised_text_col = resolve_column(
    ["text_normalized", "normalized_text", "normalised_text"],
    required=False,
)

# ---------------------------------------------------------
# 2. Clean text and standardise the five target categories
# ---------------------------------------------------------
model_data = data.copy()

def normalize_text(value):
    if pd.isna(value):
        return ""

    value = unicodedata.normalize("NFKC", str(value)).casefold()
    value = re.sub(r"https?://\S+|www\.\S+", " URL ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def normalize_category(value):
    if pd.isna(value):
        return ""

    return re.sub(
        r"[^a-z]+", "_", str(value).strip().casefold()
    ).strip("_")


model_data["model_text"] = model_data[text_col].map(
    lambda value: "" if pd.isna(value) else str(value).strip()
)
model_data["class_label"] = model_data[category_col].map(
    normalize_category
)

model_data["dedup_key"] = model_data["model_text"].map(normalize_text)

# Prefer the normalised text used in the preceding audit when available.
if normalised_text_col is not None:
    existing_keys = model_data[normalised_text_col].map(normalize_text)
    model_data["dedup_key"] = existing_keys.where(
        existing_keys.ne(""), model_data["dedup_key"]
    )

allowed_classes = {
    "cyberbullying",
    "hate_speech",
    "neutral",
    "sexual_harassment",
    "trolling",
}

model_data = model_data[
    model_data["model_text"].ne("")
    & model_data["dedup_key"].ne("")
    & model_data["class_label"].isin(allowed_classes)
].copy()

# Remove any normalised text assigned to more than one category.
labels_per_text = model_data.groupby("dedup_key")[
    "class_label"
].nunique()

conflicting_keys = labels_per_text[labels_per_text > 1].index
conflict_rows = int(model_data["dedup_key"].isin(conflicting_keys).sum())

model_data = model_data[
    ~model_data["dedup_key"].isin(conflicting_keys)
].copy()

# Repeated texts with the same label are represented only once.
before_dedup = len(model_data)
model_data = model_data.drop_duplicates(
    subset=["dedup_key"], keep="first"
).reset_index(drop=True)

print("=" * 68)
print("SIGNALHER — CLEAN CLASSIFIER DATASET")
print("=" * 68)
print(f"Rows after basic cleaning: {before_dedup:,}")
print(f"Rows removed for conflicting text labels: {conflict_rows:,}")
print(f"Repeated same-label rows removed: {before_dedup - len(model_data):,}")
print(f"Final unique samples: {len(model_data):,}")
print("\nSamples per class:")
print(model_data["class_label"].value_counts().sort_index().to_string())

# ---------------------------------------------------------
# 3. Create a stratified hold-out test set
# ---------------------------------------------------------
X = model_data["model_text"]
y = model_data["class_label"]

if y.nunique() != 5 or y.value_counts().min() < 2:
    raise ValueError(
        "The cleaned dataset does not contain enough examples "
        "in all five classes for a stratified split."
    )

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"\nTraining samples: {len(X_train):,}")
print(f"Test samples: {len(X_test):,}")

# ---------------------------------------------------------
# 4. Train a lightweight word + character TF-IDF classifier
# ---------------------------------------------------------
text_features = FeatureUnion([
    (
        "word",
        TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_features=30000,
            sublinear_tf=True,
        ),
    ),
    (
        "character",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=40000,
            sublinear_tf=True,
        ),
    ),
])

classifier = Pipeline([
    ("features", text_features),
    (
        "classifier",
        LogisticRegression(
            max_iter=500,
            class_weight="balanced",
            random_state=42,
        ),
    ),
])

classifier.fit(X_train, y_train)
predictions = classifier.predict(X_test)

# ---------------------------------------------------------
# 5. Report held-out metrics
# ---------------------------------------------------------
accuracy = accuracy_score(y_test, predictions)
balanced_accuracy = balanced_accuracy_score(y_test, predictions)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_test,
    predictions,
    average="macro",
    zero_division=0,
)

print("\n" + "=" * 68)
print("HELD-OUT TEST RESULTS")
print("=" * 68)
print(f"Accuracy:          {accuracy:.4f}")
print(f"Balanced accuracy: {balanced_accuracy:.4f}")
print(f"Macro precision:   {precision:.4f}")
print(f"Macro recall:      {recall:.4f}")
print(f"Macro F1:          {f1:.4f}")

print("\nPer-class report:")
print(classification_report(y_test, predictions, zero_division=0))

class_names = sorted(y.unique())
confusion = confusion_matrix(
    y_test, predictions, labels=class_names
)
print("Confusion matrix (rows = actual, columns = predicted):")
print(pd.DataFrame(
    confusion, index=class_names, columns=class_names
).to_string())

# ---------------------------------------------------------
# 6. Save the model and evaluation summary
# ---------------------------------------------------------
project_root = Path.cwd().parent
model_dir = project_root / "models"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "signalher_text_classifier.joblib"
metrics_path = model_dir / "signalher_classifier_metrics.json"

joblib.dump(classifier, model_path)

metrics = {
    "model_type": "TF-IDF word and character features + Logistic Regression",
    "classes": class_names,
    "training_samples": int(len(X_train)),
    "test_samples": int(len(X_test)),
    "conflicting_text_rows_removed": conflict_rows,
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced_accuracy),
    "macro_precision": float(precision),
    "macro_recall": float(recall),
    "macro_f1": float(f1),
    "split_random_state": 42,
    "test_size": 0.20,
}

with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2, ensure_ascii=False)

print("\nSaved model:", model_path)
print("Saved metrics:", metrics_path)

In [ ]:
##############################################################################################################################
##########################Search Engine Customized as "SignalHer Safety"######################################################

In [ ]:
from urllib.parse import urlencode
from flask import jsonify, request

# SignalHer Google Programmable Search configuration
SIGNALHER_GOOGLE_CX = "d536c3fcb3ce5432c"

# Find the existing endpoint without registering a new route.
search_endpoint = next(
    rule.endpoint
    for rule in signalher_app.url_map.iter_rules()
    if rule.rule == "/api/search-query" and "POST" in rule.methods
)

def signalher_safe_search_query_api():
    payload = request.get_json(silent=True) or {}
    category = payload.get("category", "general")

    # Only a category is passed to the query builder.
    # Never pass message_text, user_note, or incident contents.
    try:
        safe_query = build_safe_search_query(category)
    except (ValueError, KeyError, TypeError):
        return jsonify({"error": "Unable to prepare a safe search query."}), 400

    search_url = "https://cse.google.com/cse?" + urlencode({
        "cx": SIGNALHER_GOOGLE_CX,
        "q": safe_query,
    })

    return jsonify({
        "search_provider": "Google Programmable Search",
        "search_engine_id": SIGNALHER_GOOGLE_CX,
        "query": safe_query,
        "search_url": search_url,
        "results_mode": "Google-hosted search page",
        "privacy_note": (
            "Only the generated category-based query is sent to Google. "
            "Private incident messages and user notes are not included."
        ),
    })

signalher_app.view_functions[search_endpoint] = (
    signalher_safe_search_query_api
)

print("SignalHer Google search configuration connected to the API.")
print("Endpoint: POST /api/search-query")

In [ ]:
import json
import urllib.request

test_payload = json.dumps({
    "category": "image_abuse"
}).encode("utf-8")

test_request = urllib.request.Request(
    "http://127.0.0.1:5000/api/search-query",
    data=test_payload,
    headers={"Content-Type": "application/json"},
    method="POST",
)

with urllib.request.urlopen(test_request, timeout=10) as response:
    result = json.loads(response.read().decode("utf-8"))

print(json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
%pip install -q shap

import re
import numpy as np
import joblib
import shap
from pathlib import Path

# Load the saved SignalHer classifier
project_root = Path.cwd().parent
model_path = project_root / "models" / "signalher_text_classifier.joblib"

classifier = joblib.load(model_path)
class_names = classifier.named_steps["classifier"].classes_

# Create an explainer for the model's predicted class probabilities
explainer = shap.Explainer(
    classifier.predict_proba,
    shap.maskers.Text(),
    output_names=list(class_names),
)

# Synthetic example for the initial integration test.
# Do not use a real user's private message for this test.
sample_text = (
    "He threatened to share my private photos unless I paid him."
)

probabilities = classifier.predict_proba([sample_text])[0]
predicted_index = int(np.argmax(probabilities))
predicted_class = str(class_names[predicted_index])

# Generate token-level SHAP values
shap_result = explainer([sample_text], max_evals=100)
all_values = np.asarray(shap_result.values)

if all_values.ndim == 3:
    contributions = all_values[0, :, predicted_index]
elif all_values.ndim == 2:
    contributions = all_values[0]
else:
    raise ValueError(
        f"Unexpected SHAP output shape: {all_values.shape}"
    )

raw_tokens = shap_result.data[0]

if isinstance(raw_tokens, str):
    tokens = re.findall(r"\w+|[^\w\s]", raw_tokens, flags=re.UNICODE)
else:
    tokens = [str(token) for token in raw_tokens]

if len(tokens) != len(contributions):
    print("SHAP output shape:", all_values.shape)
    print("Token count:", len(tokens))
    print("Contribution count:", len(contributions))
    raise ValueError("Token/contribution lengths do not match.")

# Display the strongest contributions to the predicted class
ranked_tokens = sorted(
    zip(tokens, contributions),
    key=lambda item: abs(float(item[1])),
    reverse=True,
)

print("=" * 60)
print("SIGNALHER — SHAP EXPLANATION TEST")
print("=" * 60)
print("Predicted dataset category:", predicted_class)
print(f"Model probability: {probabilities[predicted_index]:.4f}")
print("\nClass probabilities:")

for name, probability in zip(class_names, probabilities):
    print(f"  {name}: {probability:.4f}")

print("\nTokens with the strongest SHAP contributions:")
for token, contribution in ranked_tokens[:10]:
    print(f"  {token!r}: {float(contribution):+.5f}")

print("\nSHAP test completed.")
print("Model:", model_path)

In [ ]:
#############################################################################################################################
###################################################Front End Starting########################################################

In [ ]:
from flask import jsonify, request
from werkzeug.routing import Rule
import numpy as np
import re
import shap

# Reuse the saved classifier and SHAP setup from the previous cell.
signalher_model = classifier
signalher_class_names = list(
    signalher_model.named_steps["classifier"].classes_
)
signalher_explainer = explainer

# Register the endpoint without restarting the running server.
route_path = "/api/explain-classifier"
endpoint_name = "api_explain_classifier"

if not any(
    rule.rule == route_path and "POST" in rule.methods
    for rule in signalher_app.url_map.iter_rules()
):
    signalher_app.url_map.add(Rule(
        route_path,
        endpoint=endpoint_name,
        methods=["POST"],
    ))


def signalher_explain_classifier():
    payload = request.get_json(silent=True) or {}
    message = payload.get("message_text", payload.get("text", ""))

    if not isinstance(message, str) or not message.strip():
        return jsonify({
            "error": "Provide a non-empty message_text field."
        }), 400

    message = message.strip()

    if len(message) > 2000:
        return jsonify({
            "error": "Message exceeds the 2000-character limit."
        }), 413

    try:
        probabilities = signalher_model.predict_proba([message])[0]
        predicted_index = int(np.argmax(probabilities))
        predicted_class = str(signalher_class_names[predicted_index])

        explanation = signalher_explainer(
            [message],
            max_evals=100,
        )

        values = np.asarray(explanation.values)

        if values.ndim == 3:
            contributions = values[0, :, predicted_index]
        elif values.ndim == 2:
            contributions = values[0]
        else:
            raise ValueError("Unexpected SHAP output shape.")

        raw_tokens = explanation.data[0]

        if isinstance(raw_tokens, str):
            tokens = re.findall(
                r"\w+|[^\w\s]",
                raw_tokens,
                flags=re.UNICODE,
            )
        else:
            tokens = [str(token) for token in raw_tokens]

        if len(tokens) != len(contributions):
            raise ValueError("SHAP token alignment failed.")

        ranked = sorted(
            zip(tokens, contributions),
            key=lambda item: abs(float(item[1])),
            reverse=True,
        )

        return jsonify({
            "experimental": True,
            "predicted_category": predicted_class,
            "class_probabilities": {
                str(name): round(float(probability), 4)
                for name, probability in zip(
                    signalher_class_names, probabilities
                )
            },
            "shap_explanations": [
                {
                    "token": str(token),
                    "contribution": round(float(value), 6),
                    "direction": (
                        "supports_predicted_category"
                        if value > 0
                        else "opposes_predicted_category"
                    ),
                }
                for token, value in ranked[:8]
            ],
            "warning": (
                "Experimental category prediction only. "
                "The classifier achieved 47.39% held-out accuracy "
                "and must not be used to assess safety or determine danger."
            ),
            "privacy_note": (
                "The message is processed by this local backend endpoint. "
                "It is not forwarded to Google by this endpoint."
            ),
        })

    except Exception:
        # Do not include the submitted message in error responses.
        return jsonify({
            "error": "Classifier explanation could not be generated."
        }), 500


signalher_app.view_functions[endpoint_name] = (
    signalher_explain_classifier
)

print("SHAP explanation endpoint registered.")
print("POST /api/explain-classifier")

In [ ]:
import json
import urllib.request

test_payload = json.dumps({
    "message_text": (
        "He threatened to share my private photos unless I paid him."
    )
}).encode("utf-8")

test_request = urllib.request.Request(
    "http://127.0.0.1:5000/api/explain-classifier",
    data=test_payload,
    headers={"Content-Type": "application/json"},
    method="POST",
)

with urllib.request.urlopen(test_request, timeout=60) as response:
    result = json.loads(response.read().decode("utf-8"))
    print("HTTP status:", response.status)
    print(json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
import json
import urllib.request
import urllib.error

BASE_URL = "http://127.0.0.1:5000"

incident = {
    "platform": "whatsapp",
    "occurred_at": "2026-10-24T20:00:00",
    "message_text": (
        "Pay me or I will share your private photos."
    ),
    "user_note": "",
    "source_reference": "synthetic-regression-test",
    "incident_id": "signalher-regression-001",
}

timeline_payload = {
    "entries": [
        {
            **incident,
            "incident": incident,
            "direction": "incoming",
            "case_id": "synthetic-case-001",
        }
    ]
}

tests = [
    ("Health", "GET", "/api/health", None),
    ("Categories", "GET", "/api/categories", None),
    ("Safety resources", "GET", "/api/resources", None),
    (
        "Google Search query",
        "POST",
        "/api/search-query",
        {"category": "image_abuse"},
    ),
    (
        "Incident analysis",
        "POST",
        "/api/analyze-incident",
        incident,
    ),
    (
        "Timeline analysis",
        "POST",
        "/api/analyze-timeline",
        timeline_payload,
    ),
    (
        "SHAP explanation",
        "POST",
        "/api/explain-classifier",
        {
            "message_text": (
                "He threatened to share my private photos "
                "unless I paid him."
            )
        },
    ),
]

results = []

for name, method, path, payload in tests:
    data = None
    headers = {}

    if payload is not None:
        data = json.dumps(payload).encode("utf-8")
        headers["Content-Type"] = "application/json"

    request = urllib.request.Request(
        BASE_URL + path,
        data=data,
        headers=headers,
        method=method,
    )

    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            body = response.read().decode("utf-8")
            results.append((name, response.status, "PASS"))
            print(f"[PASS] {name}: HTTP {response.status}")

    except urllib.error.HTTPError as error:
        body = error.read().decode("utf-8", errors="replace")
        results.append((name, error.code, "FAIL"))
        print(f"[FAIL] {name}: HTTP {error.code}")
        print(body[:600])

    except Exception as error:
        results.append((name, None, "FAIL"))
        print(f"[FAIL] {name}: {type(error).__name__}: {error}")

print("\n" + "=" * 55)
print("SIGNALHER BACKEND REGRESSION SUMMARY")
print("=" * 55)
passed = sum(status == "PASS" for _, _, status in results)
print(f"Passed: {passed}/{len(results)}")
print(f"Failed: {len(results) - passed}/{len(results)}")

In [ ]:
# STEP 1: Restore Flask's request object.
from flask import request

import json
import urllib.request
import urllib.error

BASE_URL = "http://127.0.0.1:5000"

incident_payload = {
    "platform": "whatsapp",
    "occurred_at": "2026-10-24T20:00:00",
    "message_text": "Pay me or I will share your private photos.",
    "user_note": "",
    "source_reference": "synthetic-regression-test",
    "incident_id": "signalher-regression-001",
}

timeline_payload = {
    "entries": [
        {
            "incident": incident_payload,
            "direction": "incoming",
            "case_id": "synthetic-case-001",
        }
    ]
}

tests = [
    ("Health", "GET", "/api/health", None),
    ("Categories", "GET", "/api/categories", None),
    ("Safety resources", "GET", "/api/resources", None),
    (
        "Google Search query",
        "POST",
        "/api/search-query",
        {"category": "image_abuse"},
    ),
    (
        "Incident analysis",
        "POST",
        "/api/analyze-incident",
        incident_payload,
    ),
    (
        "Timeline analysis",
        "POST",
        "/api/analyze-timeline",
        timeline_payload,
    ),
    (
        "SHAP explanation",
        "POST",
        "/api/explain-classifier",
        {
            "message_text": (
                "He threatened to share my private photos unless I paid him."
            )
        },
    ),
]

results = []

for name, method, path, payload in tests:
    body = (
        json.dumps(payload).encode("utf-8")
        if payload is not None
        else None
    )
    headers = (
        {"Content-Type": "application/json"}
        if payload is not None
        else {}
    )

    # IMPORTANT: Do not name this variable "request".
    http_request = urllib.request.Request(
        BASE_URL + path,
        data=body,
        headers=headers,
        method=method,
    )

    try:
        with urllib.request.urlopen(
            http_request, timeout=60
        ) as response:
            response_body = response.read().decode(
                "utf-8", errors="replace"
            )
            status = response.status

            results.append((name, status, True))
            print(f"[PASS] {name}: HTTP {status}")

    except urllib.error.HTTPError as error:
        response_body = error.read().decode(
            "utf-8", errors="replace"
        )
        results.append((name, error.code, False))
        print(f"[FAIL] {name}: HTTP {error.code}")
        print(response_body[:500])

    except Exception as error:
        results.append((name, None, False))
        print(f"[FAIL] {name}: {type(error).__name__}: {error}")

print("\n" + "=" * 55)
print("SIGNALHER BACKEND REGRESSION SUMMARY")
print("=" * 55)

passed = sum(ok for _, _, ok in results)
print(f"Passed: {passed}/{len(results)}")
print(f"Failed: {len(results) - passed}/{len(results)}")

In [ ]:
import json
import urllib.request
import urllib.error

BASE_URL = "http://127.0.0.1:5000"

# Corrected timestamp: explicit India timezone offset (+05:30)
incident_payload = {
    "platform": "whatsapp",
    "occurred_at": "2026-10-24T20:00:00+05:30",
    "message_text": "Pay me or I will share your private photos.",
    "user_note": "",
    "source_reference": "synthetic-regression-test",
    "incident_id": "signalher-regression-001",
}

# Each timeline entry contains its incident fields directly.
timeline_payload = {
    "entries": [
        {
            **incident_payload,
            "direction": "incoming",
            "case_id": "synthetic-case-001",
        }
    ]
}

tests_to_retry = [
    ("Incident analysis", "/api/analyze-incident", incident_payload),
    ("Timeline analysis", "/api/analyze-timeline", timeline_payload),
]

for name, path, payload in tests_to_retry:
    http_request = urllib.request.Request(
        BASE_URL + path,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    try:
        with urllib.request.urlopen(
            http_request, timeout=30
        ) as response:
            print(f"[PASS] {name}: HTTP {response.status}")
            print(response.read().decode("utf-8")[:1200])

    except urllib.error.HTTPError as error:
        print(f"[FAIL] {name}: HTTP {error.code}")
        print(error.read().decode("utf-8", errors="replace")[:1200])

    except Exception as error:
        print(f"[FAIL] {name}: {type(error).__name__}: {error}")

In [ ]:
# Targeted correction: avoid interpreting a threat to expose images
# as a request for the recipient to send images.

_original_analyze_incident_overlap_guard = analyze_incident

def analyze_incident_overlap_guard(incident):
    result = _original_analyze_incident_overlap_guard(incident)

    if not isinstance(result, dict):
        return result

    findings = result.get("findings", [])

    has_exposure_threat = any(
        finding.get("rule_id") == "THREAT_TO_EXPOSE_IMAGES"
        for finding in findings
    )

    if has_exposure_threat:
        corrected_result = dict(result)
        corrected_result["findings"] = [
            finding
            for finding in findings
            if finding.get("rule_id") != "COERCIVE_IMAGE_REQUEST"
        ]
        return corrected_result

    return result


analyze_incident = analyze_incident_overlap_guard

print("Targeted finding-overlap correction installed.")

In [ ]:
import json
import urllib.request
import urllib.error

test_payload = {
    "platform": "whatsapp",
    "occurred_at": "2026-10-24T20:00:00+05:30",
    "message_text": "Pay me or I will share your private photos.",
    "user_note": "",
    "source_reference": "synthetic-regression-test",
    "incident_id": "signalher-overlap-test-001",
}

http_request = urllib.request.Request(
    "http://127.0.0.1:5000/api/analyze-incident",
    data=json.dumps(test_payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)

try:
    with urllib.request.urlopen(http_request, timeout=30) as response:
        result = json.loads(response.read().decode("utf-8"))
        print("HTTP status:", response.status)

        analysis = result.get("analysis", result)
        findings = analysis.get("findings", [])

        rule_ids = [finding.get("rule_id") for finding in findings]
        print("Finding rule IDs:", rule_ids)

        has_exposure_threat = "THREAT_TO_EXPOSE_IMAGES" in rule_ids
        has_misleading_request = "COERCIVE_IMAGE_REQUEST" in rule_ids

        print("Exposure threat retained:", has_exposure_threat)
        print("Misleading image-request finding removed:",
              has_exposure_threat and not has_misleading_request)

except urllib.error.HTTPError as error:
    print("HTTP status:", error.code)
    print(error.read().decode("utf-8", errors="replace")[:1000])

In [ ]:
import re

# Preserve the current analyzer so this patch can be applied safely once.
if "_signalher_analyzer_before_direct_request" not in globals():
    _signalher_analyzer_before_direct_request = analyze_incident

_DIRECT_IMAGERY_REQUEST = re.compile(
    r"\b(?:send|show|share|give|dm|message)\s+"
    r"(?:(?:me|us)\s+)?"
    r"(?:(?:your|ur|some|any|a|an)\s+)?"
    r"(?:nudes?|"
    r"nude\s+(?:pics?|photos?|pictures?|images?)|"
    r"naked\s+(?:pics?|photos?|pictures?|images?)|"
    r"(?:intimate|private)\s+"
    r"(?:pics?|photos?|pictures?|images?))\b",
    re.IGNORECASE,
)

_NEGATED_IMAGERY_REQUEST = re.compile(
    r"\b(?:don't|do not|never|won't|will not|wouldn't|would not)\s+"
    r"(?:(?:ask|want)\s+(?:you\s+)?to\s+)?"
    r"(?:send|show|share|give|dm|message)\b"
    r".{0,80}\b(?:nudes?|nude|naked|intimate|private)\b",
    re.IGNORECASE,
)


def signalher_direct_request_analyzer(incident):
    result = _signalher_analyzer_before_direct_request(incident)

    if not isinstance(result, dict):
        return result

    # Support both the Incident object and dictionary-shaped inputs.
    if isinstance(incident, dict):
        message = str(incident.get("message_text", "") or "")
    else:
        message = str(getattr(incident, "message_text", "") or "")

    findings = list(result.get("findings", []) or [])
    rule_ids = {
        finding.get("rule_id")
        for finding in findings
        if isinstance(finding, dict)
    }

    is_negated = bool(_NEGATED_IMAGERY_REQUEST.search(message))
    request_match = _DIRECT_IMAGERY_REQUEST.search(message)

    # Remove this specific request indicator when the wording negates it.
    if is_negated:
        findings = [
            finding for finding in findings
            if finding.get("rule_id") not in {
                "REQUEST_FOR_PRIVATE_IMAGERY",
                "COERCIVE_IMAGE_REQUEST",
            }
        ]
    elif (
        request_match
        and not rule_ids.intersection({
            "REQUEST_FOR_PRIVATE_IMAGERY",
            "COERCIVE_IMAGE_REQUEST",
            "THREAT_TO_EXPOSE_IMAGES",
            "EXTORTION_THREAT",
        })
    ):
        findings.append({
            "category": "intimate_image_request",
            "evidence": request_match.group(0),
            "explanation": (
                "The message appears to request nude or intimate imagery. "
                "A request alone does not establish coercion or abuse; "
                "context, consent, and boundaries matter."
            ),
            "label": "Possible request for intimate imagery",
            "rule_id": "REQUEST_FOR_PRIVATE_IMAGERY",
            "suggested_action": (
                "You do not have to share intimate images. If you feel "
                "pressured or threatened, consider preserving the message "
                "if safe and seeking trusted support."
            ),
        })

    updated_result = dict(result)
    updated_result["findings"] = findings
    return updated_result


analyze_incident = signalher_direct_request_analyzer

print("Direct intimate-image request rule installed.")

In [ ]:
import json
import urllib.request
import urllib.error

test_messages = [
    ("Direct request", "Send me your nudes."),
    ("Negated request", "Don't send me your nudes."),
    (
        "Threat and extortion",
        "Pay me or I will share your private photos."
    ),
]

for label, message in test_messages:
    payload = {
        "platform": "whatsapp",
        "occurred_at": "2026-10-24T20:00:00+05:30",
        "message_text": message,
        "user_note": "",
        "source_reference": "synthetic-test",
        "incident_id": f"rule-test-{label.replace(' ', '-')}",
    }

    http_request = urllib.request.Request(
        "http://127.0.0.1:5000/api/analyze-incident",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    try:
        with urllib.request.urlopen(http_request, timeout=30) as response:
            data = json.loads(response.read().decode("utf-8"))
            analysis = data.get("analysis", data)
            findings = analysis.get("findings", [])
            ids = [f.get("rule_id") for f in findings]

            print(f"\n{label}: HTTP {response.status}")
            print("Rule IDs:", ids)

    except urllib.error.HTTPError as error:
        print(f"\n{label}: HTTP {error.code}")
        print(error.read().decode("utf-8", errors="replace")[:500])

In [ ]:
# ================================================================
# SIGNALHER — BACKEND INTEGRATION REPAIR
# 1. Activate the latest available incident analyzer
# 2. Improve direct intimate-image request detection
# 3. Register a backend-generated model-category chart
# 4. Check/restart the Flask HTTP server when possible
# ================================================================

import re
import io
import json
import base64
import socket
import threading
import urllib.request
import urllib.error

from flask import request, jsonify
from werkzeug.routing import Rule

# ------------------------------------------------
# 1. Use the latest available rule-based analyzer
# ------------------------------------------------

if "signalher_app" not in globals():
    raise RuntimeError(
        "signalher_app is not defined. Run the notebook's earlier "
        "backend setup cells before this repair cell."
    )

if "_signalher_mvp_base_analyzer" not in globals():
    if callable(globals().get("analyze_incident_v4")):
        _signalher_mvp_base_analyzer = analyze_incident_v4
        _signalher_analyzer_version = "V4 candidate"
    elif callable(globals().get("analyze_incident")):
        _signalher_mvp_base_analyzer = analyze_incident
        _signalher_analyzer_version = "currently active analyzer"
    else:
        raise RuntimeError(
            "No incident analyzer was found in the notebook."
        )

print("Base analyzer:", _signalher_analyzer_version)


# ------------------------------------------------
# 2. Broaden direct intimate-image request wording
# ------------------------------------------------

_SIGNALHER_DIRECT_REQUEST_PATTERN = re.compile(
    r"\b(?:send|show|share|give|dm|message)\s+"
    r"(?:(?:me|us)\s+)?"
    r"(?:(?:your|ur|some|any|a|an)\s+)?"
    r"(?:"
    r"nudes?|"
    r"nude\s+(?:pics?|photos?|pictures?|images?),?"
    r"|naked\s+(?:body|pics?|photos?|pictures?|images?)"
    r"|(?:intimate|private)\s+(?:pics?|photos?|pictures?|images?)"
    r")\b",
    re.IGNORECASE,
)

_SIGNALHER_NEGATED_REQUEST_PATTERN = re.compile(
    r"\b(?:don't|do not|never|won't|will not|wouldn't|would not)\s+"
    r"(?:(?:ask|want|need|expect)\s+(?:you\s+)?to\s+)?"
    r"(?:send|show|share|give|dm|message)\b"
    r".{0,100}\b(?:nudes?|nude|naked|intimate|private)\b",
    re.IGNORECASE,
)


def _signalher_get_message(incident):
    if isinstance(incident, dict):
        return str(incident.get("message_text", "") or "")
    return str(getattr(incident, "message_text", "") or "")


def _signalher_final_incident_analyzer(incident):
    result = _signalher_mvp_base_analyzer(incident)

    if not isinstance(result, dict):
        return result

    message = _signalher_get_message(incident)
    findings = list(result.get("findings", []) or [])

    request_rule_ids = {
        "REQUEST_FOR_PRIVATE_IMAGERY",
        "COERCIVE_IMAGE_REQUEST",
    }

    is_negated = bool(
        _SIGNALHER_NEGATED_REQUEST_PATTERN.search(message)
    )
    request_match = _SIGNALHER_DIRECT_REQUEST_PATTERN.search(message)

    if is_negated:
        findings = [
            finding
            for finding in findings
            if not (
                isinstance(finding, dict)
                and finding.get("rule_id") in request_rule_ids
            )
        ]

    elif request_match:
        existing_ids = {
            finding.get("rule_id")
            for finding in findings
            if isinstance(finding, dict)
        }

        if not existing_ids.intersection(request_rule_ids):
            findings.append({
                "category": "intimate_image_request",
                "evidence": request_match.group(0),
                "explanation": (
                    "The message appears to request nude or intimate "
                    "imagery. A request alone does not establish coercion "
                    "or abuse; context, consent, and boundaries matter."
                ),
                "label": "Possible request for intimate imagery",
                "rule_id": "REQUEST_FOR_PRIVATE_IMAGERY",
                "suggested_action": (
                    "You do not have to share intimate images. If you feel "
                    "pressured or threatened, consider preserving the "
                    "message if safe and seeking trusted support."
                ),
            })

    updated_result = dict(result)
    updated_result["findings"] = findings
    return updated_result


# The Flask incident route normally invokes this module-level function.
analyze_incident = _signalher_final_incident_analyzer

print("Incident analyzer integration installed.")


# ------------------------------------------------
# 3. Register a backend-generated category chart
# ------------------------------------------------

_explain_endpoint = None

for _rule in signalher_app.url_map.iter_rules():
    if (
        _rule.rule == "/api/explain-classifier"
        and "POST" in _rule.methods
    ):
        _explain_endpoint = _rule.endpoint
        break

if _explain_endpoint is None:
    raise RuntimeError(
        "POST /api/explain-classifier is not registered. "
        "Run the earlier SHAP endpoint setup cell first."
    )


def _signalher_explain_with_chart():
    # Call the already tested explanation endpoint in the same request
    # context, preserving its established input validation and SHAP logic.
    original_view = signalher_app.view_functions[_explain_endpoint]
    original_response = signalher_app.make_response(original_view())

    if original_response.status_code >= 400:
        return original_response

    payload = original_response.get_json(silent=True)

    if not isinstance(payload, dict):
        return jsonify({
            "error": "The classifier explanation returned invalid JSON."
        }), 502

    probabilities = payload.get("class_probabilities")

    if not isinstance(probabilities, dict) or not probabilities:
        return jsonify({
            "error": "The classifier returned no category probabilities."
        }), 502

    try:
        import matplotlib.pyplot as plt

        ordered = sorted(
            [
                (str(category), float(value))
                for category, value in probabilities.items()
            ],
            key=lambda item: item[1],
        )

        labels = [item[0] for item in ordered]
        values = [max(0.0, min(1.0, item[1])) for item in ordered]

        figure, axis = plt.subplots(figsize=(8, 4.4))

        bars = axis.barh(
            labels,
            values,
            color="#ed2780",
            edgecolor="#c90f62",
            height=0.62,
        )

        axis.set_xlim(0, 1)
        axis.set_xlabel("Model probability — not a safety-risk score")
        axis.set_title("Experimental message-category estimates")
        axis.grid(axis="x", alpha=0.2)
        axis.set_axisbelow(True)

        for bar, value in zip(bars, values):
            axis.text(
                min(value + 0.015, 0.91),
                bar.get_y() + bar.get_height() / 2,
                f"{value:.1%}",
                va="center",
                fontsize=9,
            )

        figure.tight_layout()

        buffer = io.BytesIO()
        figure.savefig(
            buffer,
            format="png",
            dpi=150,
            bbox_inches="tight",
        )
        plt.close(figure)

        payload["chart_png_base64"] = base64.b64encode(
            buffer.getvalue()
        ).decode("ascii")

        payload["chart_title"] = (
            "Experimental message-category estimates"
        )
        payload["chart_type"] = "message_category_probabilities"
        payload["warning"] = (
            "This classifier achieved 47.39% accuracy on its held-out "
            "test set. Its estimates are experimental and are not a "
            "validated or calibrated measure of danger."
        )

        return jsonify(payload)

    except Exception:
        signalher_app.logger.exception(
            "SignalHer category-chart generation failed."
        )
        return jsonify({
            "error": "The model result was obtained, but chart generation failed."
        }), 500


_chart_path = "/api/explain-classifier-chart"
_chart_endpoint = "api_explain_classifier_chart"

if not any(
    rule.rule == _chart_path and "POST" in rule.methods
    for rule in signalher_app.url_map.iter_rules()
):
    signalher_app.url_map.add(Rule(
        _chart_path,
        endpoint=_chart_endpoint,
        methods=["POST"],
    ))

signalher_app.view_functions[_chart_endpoint] = (
    _signalher_explain_with_chart
)

print("Registered POST", _chart_path)


# ------------------------------------------------
# 4. Check Flask health and start it if port 5000 is free
# ------------------------------------------------

def _signalher_health_check():
    with urllib.request.urlopen(
        "http://127.0.0.1:5000/api/health",
        timeout=2,
    ) as response:
        return response.status == 200


try:
    _healthy = _signalher_health_check()
except Exception:
    _healthy = False

if _healthy:
    print("Flask backend is already responding on port 5000.")
else:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        _port_in_use = (
            sock.connect_ex(("127.0.0.1", 5000)) == 0
        )

    if _port_in_use:
        print(
            "Port 5000 is occupied, but the health check did not pass. "
            "No second server was started. Inspect the existing Flask server."
        )
    else:
        from werkzeug.serving import make_server

        _signalher_http_server = make_server(
            "127.0.0.1",
            5000,
            signalher_app,
            threaded=True,
        )

        _signalher_server_thread = threading.Thread(
            target=_signalher_http_server.serve_forever,
            daemon=True,
            name="SignalHer-Flask-Server",
        )
        _signalher_server_thread.start()

        print("Started SignalHer Flask server on port 5000.")


# ------------------------------------------------
# 5. Test the two endpoints with synthetic data
# ------------------------------------------------

def _signalher_post_json(path, payload):
    data = json.dumps(payload).encode("utf-8")

    http_request = urllib.request.Request(
        "http://127.0.0.1:5000" + path,
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    with urllib.request.urlopen(http_request, timeout=60) as response:
        return response.status, json.loads(
            response.read().decode("utf-8")
        )


_test_incident = {
    "platform": "whatsapp",
    "occurred_at": "2026-10-25T01:30:00+05:30",
    "message_text": "Show me your naked body because I love you.",
    "user_note": "",
    "source_reference": "synthetic-integration-test",
    "incident_id": "signalher-final-integration-test",
}

try:
    _status, _analysis_response = _signalher_post_json(
        "/api/analyze-incident",
        _test_incident,
    )

    _analysis = _analysis_response.get(
        "analysis",
        _analysis_response,
    )

    _findings = _analysis.get("findings", [])
    _rule_ids = [
        item.get("rule_id")
        for item in _findings
        if isinstance(item, dict)
    ]

    print("\nIncident API status:", _status)
    print("Detected rule IDs:", _rule_ids)
    print(
        "Direct request test:",
        "PASS" if "REQUEST_FOR_PRIVATE_IMAGERY" in _rule_ids
        else "CHECK REQUIRED — endpoint may capture an older analyzer"
    )

except Exception as exc:
    print("\nIncident API test failed:", repr(exc))


try:
    _status, _chart_response = _signalher_post_json(
        "/api/explain-classifier-chart",
        {
            "message_text": (
                "He threatened to share my private photos unless I paid him."
            )
        },
    )

    print("\nChart API status:", _status)
    print("Chart generated:", bool(
        _chart_response.get("chart_png_base64")
    ))
    print("Chart title:", _chart_response.get("chart_title"))

except Exception as exc:
    print("\nChart API test failed:", repr(exc))


try:
    print(
        "\nFinal health check:",
        "PASS" if _signalher_health_check() else "FAILED",
    )
except Exception as exc:
    print("\nFinal health check failed:", repr(exc))

In [ ]:
%pip install matplotlib

In [ ]:
import json
import urllib.request
import urllib.error

url = "http://127.0.0.1:5000/api/explain-classifier-chart"

payload = {
    "message_text": (
        "He threatened to share my private photos unless I paid him."
    )
}

req = urllib.request.Request(
    url,
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)

try:
    with urllib.request.urlopen(req, timeout=60) as response:
        result = json.loads(response.read().decode("utf-8"))

        print("HTTP status:", response.status)
        print("Predicted category:", result.get("predicted_category"))
        print("Chart title:", result.get("chart_title"))
        print("Chart generated:", bool(result.get("chart_png_base64")))
        print("Warning:", result.get("warning"))

except urllib.error.HTTPError as error:
    print("HTTP status:", error.code)
    print(error.read().decode("utf-8", errors="replace")[:1500])

except Exception as error:
    print(type(error).__name__ + ":", error)

In [ ]:
import json
import urllib.request
import urllib.error

BASE_URL = "http://127.0.0.1:5000"

test_cases = [
    (
        "Intimate-image request",
        "Show me your naked body because I love you."
    ),
    (
        "Threat and extortion",
        "He threatened to share my private photos unless I paid him."
    ),
    (
        "Direct intimate-image request",
        "Send me your nudes."
    ),
]

for label, message in test_cases:
    payload = {
        "platform": "whatsapp",
        "occurred_at": "2026-10-25T01:30:00+05:30",
        "message_text": message,
        "user_note": "",
        "source_reference": "synthetic-regression-test",
        "incident_id": f"rule-check-{label.replace(' ', '-')}",
    }

    request = urllib.request.Request(
        f"{BASE_URL}/api/analyze-incident",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            data = json.loads(response.read().decode("utf-8"))
            analysis = data.get("analysis", data)
            findings = analysis.get("findings", [])

            print(f"\n{label} — HTTP {response.status}")
            print("Rule IDs:", [
                item.get("rule_id")
                for item in findings
                if isinstance(item, dict)
            ])
            print("Findings:", [
                item.get("label")
                for item in findings
                if isinstance(item, dict)
            ])

    except urllib.error.HTTPError as error:
        print(f"\n{label} — HTTP {error.code}")
        print(error.read().decode("utf-8", errors="replace")[:1000])

In [ ]:
# ============================================================
# SIGNALHER — RESTORE FLASK REQUEST + RETEST INCIDENT ANALYSIS
# Do not restart the kernel or Flask server.
# ============================================================

from flask import request as flask_request

# Restore the Flask request object accidentally shadowed by test code.
request = flask_request

import json
import urllib.request
import urllib.error

BASE_URL = "http://127.0.0.1:5000"

test_cases = [
    (
        "Intimate-image request",
        "Show me your naked body because I love you.",
    ),
    (
        "Threat and extortion",
        "He threatened to share my private photos unless I paid him.",
    ),
    (
        "Direct intimate-image request",
        "Send me your nudes.",
    ),
]

for label, message in test_cases:
    payload = {
        "platform": "whatsapp",
        "occurred_at": "2026-10-25T01:30:00+05:30",
        "message_text": message,
        "user_note": "",
        "source_reference": "synthetic-regression-test",
        "incident_id": f"signalher-test-{label.replace(' ', '-')}",
    }

    # IMPORTANT: use http_request, not request.
    http_request = urllib.request.Request(
        f"{BASE_URL}/api/analyze-incident",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    try:
        with urllib.request.urlopen(
            http_request, timeout=30
        ) as response:
            result = json.loads(response.read().decode("utf-8"))
            analysis = result.get("analysis", result)
            findings = analysis.get("findings", [])

            rule_ids = [
                item.get("rule_id")
                for item in findings
                if isinstance(item, dict)
            ]

            print(f"\n{label}: HTTP {response.status}")
            print("Rule IDs:", rule_ids)

    except urllib.error.HTTPError as error:
        print(f"\n{label}: HTTP {error.code}")
        print(error.read().decode("utf-8", errors="replace")[:800])

    except Exception as error:
        print(f"\n{label}: {type(error).__name__}: {error}")

print("\nFlask request object restored.")

In [ ]:
import json
import urllib.request
import urllib.error
from datetime import datetime

SIGNALHER_TEST_URL = "http://127.0.0.1:5000/api/analyze-incident"

threat_cases = [
    "Pay me or I will share your private photos.",
    "Send money or I will post your intimate pictures.",
    "If you do not pay me, I will leak your private images."
]

for index, message in enumerate(threat_cases, start=1):
    payload = {
        "platform": "instagram",
        "occurred_at": datetime.now().astimezone().isoformat(timespec="seconds"),
        "message_text": message,
        "user_note": ""
    }

    http_request = urllib.request.Request(
        SIGNALHER_TEST_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    try:
        with urllib.request.urlopen(http_request, timeout=10) as response:
            result = json.loads(response.read().decode("utf-8"))
            print(f"\nCase {index}: HTTP {response.status}")
            print("Message:", message)
            print("Rule IDs:", result.get("rule_ids", []))
    except urllib.error.HTTPError as error:
        print(f"\nCase {index}: HTTP {error.code}")
        print(error.read().decode("utf-8")[:1000])
    except Exception as error:
        print(f"\nCase {index}: {type(error).__name__}: {error}")

In [ ]:
# SignalHer — targeted threat/extortion regression patch
# Safe to rerun; does not overwrite Flask's global request object.

import re
from flask import request as _signalher_flask_request, jsonify as _signalher_jsonify

_signalher_threat_endpoint_rule = next(
    (
        rule for rule in signalher_app.url_map.iter_rules()
        if rule.rule == "/api/analyze-incident"
        and "POST" in rule.methods
    ),
    None
)

if _signalher_threat_endpoint_rule is None:
    raise RuntimeError("Could not find POST /api/analyze-incident")

_signalher_threat_endpoint = _signalher_threat_endpoint_rule.endpoint
_signalher_current_view = signalher_app.view_functions[_signalher_threat_endpoint]

# Avoid stacking multiple wrappers if the cell is rerun.
if getattr(_signalher_current_view, "_signalher_threat_patch", False):
    _signalher_original_view = _signalher_current_view._signalher_original_view
else:
    _signalher_original_view = _signalher_current_view

_SIGNALHER_IMAGE_REFERENCE = re.compile(
    r"\b(?:private|intimate|nude|naked|sexual)\b"
    r".{0,50}\b(?:photos?|pics?|images?|pictures?|videos?|recordings?)\b",
    re.I
)

_SIGNALHER_EXPOSURE_ACTION = re.compile(
    r"\b(?:share|post|leak|expose|publish|upload|distribute|release|"
    r"send|show|circulate)\b",
    re.I
)

_SIGNALHER_MONEY_DEMAND = re.compile(
    r"\b(?:pay|payment|money|cash|rupees?|transfer|wire)\b|₹|\$",
    re.I
)

_SIGNALHER_NEGATED_EXPOSURE = re.compile(
    r"\b(?:i|we)\s+(?:will not|would not|shall not|won't|wouldn't|"
    r"shan't|can't|cannot)\s+(?:ever\s+)?"
    r"(?:share|post|leak|expose|publish|upload|distribute|release|send)\b"
    r"|\b(?:do not|don't|never)\s+"
    r"(?:share|post|leak|expose|publish|upload|distribute|release|send)\b",
    re.I
)


def _signalher_patched_incident_view(*args, **kwargs):
    original_result = _signalher_original_view(*args, **kwargs)

    payload = _signalher_flask_request.get_json(silent=True) or {}
    message = str(
        payload.get("message_text")
        or payload.get("message")
        or ""
    ).strip()

    if not message:
        return original_result

    image_reference = bool(_SIGNALHER_IMAGE_REFERENCE.search(message))
    exposure_action = bool(_SIGNALHER_EXPOSURE_ACTION.search(message))
    money_demand = bool(_SIGNALHER_MONEY_DEMAND.search(message))
    negated_exposure = bool(_SIGNALHER_NEGATED_EXPOSURE.search(message))

    # Keep this patch narrow: a matching image reference plus an exposure
    # action is required. Explicit negation is ignored unless money is
    # demanded as part of a possible conditional threat.
    threat_to_expose = (
        image_reference
        and exposure_action
        and (not negated_exposure or money_demand)
    )
    extortion_threat = threat_to_expose and money_demand

    if not threat_to_expose:
        return original_result

    # Convert the original view output to JSON while preserving its status.
    response = signalher_app.make_response(original_result)
    output = response.get_json(silent=True)

    if not isinstance(output, dict) or response.status_code >= 400:
        return original_result

    ids = output.get("rule_ids", [])
    if not isinstance(ids, list):
        ids = []
    else:
        ids = list(ids)

    additions = []
    if extortion_threat:
        additions.append("EXTORTION_THREAT")
    additions.append("THREAT_TO_EXPOSE_IMAGES")

    for rule_id in additions:
        if rule_id not in ids:
            ids.append(rule_id)

    output["rule_ids"] = ids

    # Add an explainable signal using the project's documented data contract.
    findings = output.get("findings", [])
    if not isinstance(findings, list):
        findings = []

    finding_definitions = []
    if extortion_threat:
        finding_definitions.append({
            "rule_id": "EXTORTION_THREAT",
            "title": "Possible image-based extortion",
            "category": "Coercion and extortion",
            "evidence": message,
            "explanation": (
                "The wording combines a demand involving money with a "
                "threat to expose private or intimate imagery."
            ),
            "uncertainty_note": (
                "This is a wording-based signal to review, not a legal "
                "finding or a prediction of what someone will do."
            )
        })

    finding_definitions.append({
        "rule_id": "THREAT_TO_EXPOSE_IMAGES",
        "title": "Possible threat to expose private imagery",
        "category": "Intimate-image abuse",
        "evidence": message,
        "explanation": (
            "The wording links an exposure action with a reference to "
            "private, intimate, nude, or naked imagery."
        ),
        "uncertainty_note": (
            "Review the message and context; the rule cannot independently "
            "verify what happened."
        )
    })

    for finding in finding_definitions:
        if not any(
            isinstance(existing, dict)
            and existing.get("rule_id") == finding["rule_id"]
            for existing in findings
        ):
            findings.append(finding)

    output["findings"] = findings

    patched_response = _signalher_jsonify(output)
    patched_response.status_code = response.status_code
    return patched_response


_signalher_patched_incident_view._signalher_threat_patch = True
_signalher_patched_incident_view._signalher_original_view = (
    _signalher_original_view
)
signalher_app.view_functions[_signalher_threat_endpoint] = (
    _signalher_patched_incident_view
)

print("Targeted threat/extortion patch installed.")
print("Endpoint:", "/api/analyze-incident")
print("Flask request object preserved.")

In [ ]:
import json
import urllib.request
import urllib.error
from datetime import datetime

test_messages = [
    (
        "Threat and extortion",
        "Pay me or I will share your private photos."
    ),
    (
        "Threat to expose images",
        "Send money or I will post your intimate pictures."
    ),
    (
        "Conditional threat",
        "If you do not pay me, I will leak your private images."
    ),
    (
        "Negated statement",
        "I will not share your private photos."
    ),
    (
        "Benign control",
        "I like the photos you shared yesterday."
    )
]

url = "http://127.0.0.1:5000/api/analyze-incident"

for label, message in test_messages:
    payload = {
        "platform": "instagram",
        "occurred_at": datetime.now().astimezone().isoformat(timespec="seconds"),
        "message_text": message,
        "user_note": ""
    }

    http_request = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    try:
        with urllib.request.urlopen(http_request, timeout=10) as response:
            data = json.loads(response.read().decode("utf-8"))
            print(f"\n{label} — HTTP {response.status}")
            print("Rule IDs:", data.get("rule_ids", []))
            print(
                "Finding IDs:",
                [
                    item.get("rule_id")
                    for item in data.get("findings", [])
                    if isinstance(item, dict)
                ]
            )
    except urllib.error.HTTPError as error:
        print(f"\n{label} — HTTP {error.code}")
        print(error.read().decode("utf-8")[:500])
    except Exception as error:
        print(f"\n{label} — {type(error).__name__}: {error}")

In [ ]:
# SignalHer — inspect the live resource directory
# Read-only diagnostic. Does not modify the Flask app.
# Do not name any variable "request".

import json
import urllib.request
import urllib.error

SIGNALHER_RESOURCES_URL = "http://127.0.0.1:5000/api/resources"

try:
    with urllib.request.urlopen(
        SIGNALHER_RESOURCES_URL,
        timeout=10
    ) as http_response:
        status_code = http_response.status
        response_text = http_response.read().decode("utf-8")

    print(f"HTTP status: {status_code}")

    resource_data = json.loads(response_text)

    def inspect_resource_items(value, path="root"):
        if isinstance(value, list):
            for index, item in enumerate(value):
                inspect_resource_items(item, f"{path}[{index}]")

        elif isinstance(value, dict):
            name = (
                value.get("name")
                or value.get("title")
                or value.get("resource_name")
                or value.get("id")
            )

            url = (
                value.get("official_url")
                or value.get("url")
                or value.get("website")
                or value.get("link")
            )

            if name or url:
                print(f"\nLocation: {path}")
                print(f"Name: {name or '(not provided)'}")
                print(f"URL: {url or '(not provided)'}")

            for key, nested_value in value.items():
                if isinstance(nested_value, (dict, list)):
                    inspect_resource_items(
                        nested_value,
                        f"{path}.{key}"
                    )

    print("\n--- Resource entries returned by Flask ---")
    inspect_resource_items(resource_data)

    print("\n--- Top-level response structure ---")
    if isinstance(resource_data, dict):
        print("Keys:", list(resource_data.keys()))
    else:
        print("Response type:", type(resource_data).__name__)

except urllib.error.HTTPError as error:
    print(f"HTTP error: {error.code}")
    print(error.read().decode("utf-8")[:1000])

except Exception as error:
    print(f"Inspection failed: {type(error).__name__}: {error}")

In [ ]:
# SignalHer — locate the backend resource definition
# Read-only diagnostic; does not modify any files.
# Important: do not name any variable "request".

from pathlib import Path
import json

CURRENT_DIR = Path.cwd()

PROJECT_ROOT = next(
    (
        folder
        for folder in (CURRENT_DIR, *CURRENT_DIR.parents)
        if (folder / "frontend").is_dir()
    ),
    CURRENT_DIR
)

SKIP_DIRS = {
    ".venv", "venv", "node_modules", ".git",
    "__pycache__", ".ipynb_checkpoints"
}

SEARCH_TERMS = (
    "112.gov.in",
    "Emergency Response Support System",
    "National Cyber Crime Reporting Portal",
    "/api/resources",
)

print("Project root:", PROJECT_ROOT)
print("\nSearching for the backend resource definition...\n")

files_checked = 0
matches_found = 0

for file_path in PROJECT_ROOT.rglob("*"):
    if not file_path.is_file():
        continue

    if any(part in SKIP_DIRS for part in file_path.parts):
        continue

    if file_path.suffix.lower() not in {".ipynb", ".py", ".json"}:
        continue

    try:
        if file_path.suffix.lower() == ".ipynb":
            notebook_data = json.loads(
                file_path.read_text(encoding="utf-8")
            )

            for cell_number, cell in enumerate(
                notebook_data.get("cells", []), start=1
            ):
                if cell.get("cell_type") != "code":
                    continue

                source = cell.get("source", [])
                source_text = (
                    "".join(source)
                    if isinstance(source, list)
                    else str(source)
                )

                if not any(term in source_text for term in SEARCH_TERMS):
                    continue

                matches_found += 1
                print("=" * 75)
                print(f"File: {file_path}")
                print(f"Code cell: {cell_number}")

                lines = source_text.splitlines()
                matched_indexes = [
                    index
                    for index, line in enumerate(lines)
                    if any(term in line for term in SEARCH_TERMS)
                ]

                shown_indexes = set()
                for index in matched_indexes:
                    for nearby in range(
                        max(0, index - 8),
                        min(len(lines), index + 13)
                    ):
                        shown_indexes.add(nearby)

                for index in sorted(shown_indexes):
                    print(f"{index + 1:4}: {lines[index]}")

        else:
            files_checked += 1
            source_text = file_path.read_text(
                encoding="utf-8",
                errors="replace"
            )

            if not any(term in source_text for term in SEARCH_TERMS):
                continue

            matches_found += 1
            print("=" * 75)
            print(f"File: {file_path}")

            lines = source_text.splitlines()
            matched_indexes = [
                index for index, line in enumerate(lines)
                if any(term in line for term in SEARCH_TERMS)
            ]

            shown_indexes = set()
            for index in matched_indexes:
                for nearby in range(
                    max(0, index - 8),
                    min(len(lines), index + 13)
                ):
                    shown_indexes.add(nearby)

            for index in sorted(shown_indexes):
                print(f"{index + 1:4}: {lines[index]}")

    except (OSError, json.JSONDecodeError) as error:
        print(f"Could not inspect {file_path}: {error}")

print("\n" + "=" * 75)
print("Search complete.")
print("Matching files/cells:", matches_found)

In [ ]:
# SIGNALHER — FINAL FRONTEND INSTALL CELL
# Writes the final index.html, app.js, and styles.css into your existing frontend folder.
# Makes a timestamped backup first. Does not edit or restart the Flask/Jupyter backend.

from pathlib import Path
from datetime import datetime
import base64
import json
import shutil
import zlib

# Embedded final frontend package (HTML, JavaScript, and CSS).
SIGNALHER_FRONTEND_BUNDLE = (
    '...'
)

frontend_files = json.loads(
    zlib.decompress(base64.b64decode(SIGNALHER_FRONTEND_BUNDLE)).decode("utf-8")
)

current_path = Path.cwd()
project_root = next(
    (path for path in (current_path, *current_path.parents) if (path / "frontend").is_dir()),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not find the SignalHer project folder from this notebook's working directory. "
        "Open this notebook inside the SignalHer project, or change its working directory "
        "to C:/Users/Ropka/OneDrive/Apps/Desktop/RaYVention_SignalHer and run this cell again."
    )

frontend_dir = project_root / "frontend"
required_files = ("index.html", "app.js", "styles.css")
missing = [name for name in required_files if not (frontend_dir / name).is_file()]
if missing:
    raise FileNotFoundError(
        f"Expected these existing frontend files before replacing them: {missing}. "
        f"Detected frontend folder: {frontend_dir}"
    )

backup_dir = project_root / (
    "frontend_backup_before_final_update_" + datetime.now().strftime("%Y%m%d_%H%M%S")
)
backup_dir.mkdir(parents=True, exist_ok=False)
for name in required_files:
    shutil.copy2(frontend_dir / name, backup_dir / name)

for name in required_files:
    destination = frontend_dir / name
    destination.write_text(frontend_files[name], encoding="utf-8", newline="\n")

# Sanity checks: graph is fully removed from the visible frontend/source bundle.
combined_frontend = "\n".join(frontend_files.values()).lower()
graph_terms = (
    "message category graph",
    "requestmessagecategorygraph",
    "/api/explain-classifier",
    "/api/explain-classifier-chart",
)
remaining_graph_terms = [term for term in graph_terms if term in combined_frontend]
if remaining_graph_terms:
    raise RuntimeError(f"Graph-related frontend code remains: {remaining_graph_terms}")

if "function renderTimeline(response)" not in frontend_files["app.js"]:
    raise RuntimeError("Timeline renderer was not found in the installed JavaScript.")

print("SIGNALHER FRONTEND UPDATED")
print("Project root:", project_root)
print("Updated files:")
for name in required_files:
    print(" -", frontend_dir / name, f"({(frontend_dir / name).stat().st_size:,} bytes)")
print("Backup saved to:", backup_dir)
print("Message Category Graph removed from the frontend: PASSED")
print("Timeline deduplication renderer included: PASSED")
print("Flask/Jupyter backend was not modified or restarted.")
print("Next: refresh http://127.0.0.1:5500/index.html with Ctrl+F5 and test Scenario 1 + Timeline once.")


In [ ]:
# SignalHer — allow browser requests from the local frontend

from flask import request as _signalher_cors_request

_SIGNALHER_ALLOWED_ORIGINS = {
    "http://127.0.0.1:5500",
    "http://localhost:5500",
}


def _signalher_add_cors_headers(response):
    origin = _signalher_cors_request.headers.get("Origin")

    if origin in _SIGNALHER_ALLOWED_ORIGINS:
        response.headers["Access-Control-Allow-Origin"] = origin
        response.headers["Vary"] = "Origin"
        response.headers["Access-Control-Allow-Methods"] = (
            "GET, POST, OPTIONS"
        )
        response.headers["Access-Control-Allow-Headers"] = "Content-Type"

    return response


# Register once, including when running this cell again.
if not getattr(signalher_app, "_signalher_cors_fix_installed", False):
    signalher_app.after_request_funcs.setdefault(None, []).append(
        _signalher_add_cors_headers
    )
    signalher_app._signalher_cors_fix_installed = True

print("SignalHer local frontend CORS support is installed.")
print("Allowed frontend origins:")
for origin in sorted(_SIGNALHER_ALLOWED_ORIGINS):
    print(" -", origin)

In [ ]:
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

backend = "http://127.0.0.1:5000"
frontend_origin = "http://127.0.0.1:5500"

# Check the regular health request and the browser's preflight request.
for method in ("GET", "OPTIONS"):
    print(f"\n--- {method} /api/health ---")

    headers = {"Origin": frontend_origin}
    if method == "OPTIONS":
        headers.update({
            "Access-Control-Request-Method": "GET",
            "Access-Control-Request-Headers": "content-type",
        })

    req = Request(
        f"{backend}/api/health",
        headers=headers,
        method=method,
    )

    try:
        with urlopen(req, timeout=5) as response:
            print("HTTP status:", response.status)
            print(
                "Access-Control-Allow-Origin:",
                response.headers.get("Access-Control-Allow-Origin"),
            )
            print(
                "Access-Control-Allow-Methods:",
                response.headers.get("Access-Control-Allow-Methods"),
            )
            print(
                "Access-Control-Allow-Headers:",
                response.headers.get("Access-Control-Allow-Headers"),
            )
    except (HTTPError, URLError, TimeoutError) as exc:
        print("Request failed:", repr(exc))

In [ ]:
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

ORIGIN = "http://127.0.0.1:5500"
HEALTH_URL = "http://127.0.0.1:5000/api/health"

print("=== 1. Current Flask app ===")
print("App object ID:", id(signalher_app))
print("Registered after-request callbacks:")

for blueprint, callbacks in signalher_app.after_request_funcs.items():
    print(
        f"  Blueprint {blueprint!r}:",
        [getattr(cb, "__name__", repr(cb)) for cb in callbacks]
    )

print("\n=== 2. In-memory Flask test ===")
with signalher_app.test_client() as client:
    response = client.get(
        "/api/health",
        headers={"Origin": ORIGIN},
    )
    print("HTTP status:", response.status_code)
    print("Allow-Origin:", response.headers.get("Access-Control-Allow-Origin"))
    print("Allow-Methods:", response.headers.get("Access-Control-Allow-Methods"))

print("\n=== 3. Live server test ===")
try:
    req = Request(
        HEALTH_URL,
        headers={"Origin": ORIGIN},
    )
    with urlopen(req, timeout=5) as response:
        print("HTTP status:", response.status)
        print("Allow-Origin:", response.headers.get("Access-Control-Allow-Origin"))
        print("Allow-Methods:", response.headers.get("Access-Control-Allow-Methods"))

except (HTTPError, URLError, TimeoutError, OSError) as exc:
    print("Live request failed:", repr(exc))

print("\n=== 4. Server app identity, if available ===")
for attribute in ("app", "application"):
    server_app = getattr(
        globals().get("_signalher_http_server"),
        attribute,
        None,
    )
    if server_app is not None:
        print(f"Server.{attribute} object ID:", id(server_app))
        print("Same as current Flask app:", server_app is signalher_app)

In [ ]:
# SignalHer — replace the stale server with the configured Flask app

import time
from threading import Thread
from werkzeug.serving import make_server
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

print("STEP 1: Stopping the old server...")

old_server = globals().get("_signalher_http_server")
old_thread = globals().get("_signalher_server_thread")

if old_server is not None:
    if old_thread is not None and old_thread.is_alive():
        old_server.shutdown()
        old_thread.join(timeout=5)

        if old_thread.is_alive():
            raise RuntimeError(
                "The old server thread did not stop. "
                "Do not start another server yet."
            )

    old_server.server_close()
    print("Old server stopped and its socket closed.")
else:
    print("No previous server object was found.")

print("\nSTEP 2: Starting the server with the current Flask app...")

_signalher_http_server = make_server(
    host="127.0.0.1",
    port=5000,
    app=signalher_app,
    threaded=True,
)

_signalher_server_thread = Thread(
    target=_signalher_http_server.serve_forever,
    daemon=True,
    name="SignalHerConfiguredServer",
)

_signalher_server_thread.start()
time.sleep(0.3)

print("Server thread running:", _signalher_server_thread.is_alive())
print("Backend URL: http://127.0.0.1:5000")

print("\nSTEP 3: Verifying the live health endpoint and CORS...")

frontend_origin = "http://127.0.0.1:5500"

request = Request(
    "http://127.0.0.1:5000/api/health",
    headers={"Origin": frontend_origin},
)

with urlopen(request, timeout=5) as response:
    print("Health HTTP status:", response.status)
    print("Health response:", response.read().decode("utf-8"))
    print(
        "Access-Control-Allow-Origin:",
        response.headers.get("Access-Control-Allow-Origin"),
    )

    if (
        response.status == 200
        and response.headers.get("Access-Control-Allow-Origin")
        == frontend_origin
    ):
        print("\nPASS: Live backend and CORS are working.")
    else:
        print("\nCHECK NEEDED: The live response is missing expected headers.")

In [ ]:
# SignalHer — restore Flask's request object and retest the live API
# Important: do not use the global name `request` for urllib objects.

from flask import request as _signalher_flask_request
request = _signalher_flask_request

import json as _signalher_json
from urllib.request import Request as _SignalHerHTTPRequest
from urllib.request import urlopen as _signalher_urlopen
from urllib.error import HTTPError as _SignalHerHTTPError
from urllib.error import URLError as _SignalHerURLError

_SIGNALHER_BACKEND = "http://127.0.0.1:5000"
_SIGNALHER_ORIGIN = "http://127.0.0.1:5500"


def _signalher_live_test(path, payload=None):
    url = _SIGNALHER_BACKEND + path
    headers = {"Origin": _SIGNALHER_ORIGIN}
    body = None

    if payload is not None:
        body = _signalher_json.dumps(payload).encode("utf-8")
        headers["Content-Type"] = "application/json"

    http_request = _SignalHerHTTPRequest(
        url,
        data=body,
        headers=headers,
        method="POST" if payload is not None else "GET",
    )

    try:
        with _signalher_urlopen(http_request, timeout=8) as response:
            raw = response.read().decode("utf-8")
            try:
                result = _signalher_json.loads(raw)
            except _signalher_json.JSONDecodeError:
                result = raw

            print(f"\n{path}")
            print("HTTP status:", response.status)
            print(
                "CORS origin:",
                response.headers.get("Access-Control-Allow-Origin"),
            )
            print("Response:", result)
            return response.status, result

    except _SignalHerHTTPError as exc:
        print(f"\n{path}")
        print("HTTP status:", exc.code)
        print("Error response:", exc.read().decode("utf-8", errors="replace"))
        return exc.code, None

    except (_SignalHerURLError, TimeoutError, OSError) as exc:
        print(f"\n{path}")
        print("Connection failed:", repr(exc))
        return None, None


print("Restored Flask request object:", type(request).__name__)
print("Testing live endpoints...")

_signalher_live_test("/api/health")
_categories_status, _categories_result = _signalher_live_test("/api/categories")
_resources_status, _resources_result = _signalher_live_test("/api/resources")

# Test the search endpoint using an existing category identifier.
if isinstance(CATEGORY_LABELS, dict) and CATEGORY_LABELS:
    _category_for_test = next(iter(CATEGORY_LABELS.keys()))
    _signalher_live_test(
        "/api/search-query",
        {"category": _category_for_test},
    )
else:
    print("\nSearch test skipped: no configured categories were found.")

# Test incident analysis using a synthetic message only.
_signalher_live_test(
    "/api/analyze-incident",
    {
        "platform": next(iter(Platform)).value,
        "occurred_at": "2026-10-25T10:00:00+05:30",
        "message_text": (
            "If you don't send money, I will expose your private images."
        ),
        "user_note": "Synthetic API connectivity test.",
        "source_reference": "",
        "incident_id": "signalher-live-api-test",
    },
)

print("\nLive endpoint test complete.")

In [ ]:
# SignalHer — inspect current routes, category mappings, and timeline functions

import inspect

print("=" * 55)
print("1. CURRENT API ROUTES")
print("=" * 55)

for rule in sorted(signalher_app.url_map.iter_rules(), key=lambda r: r.rule):
    if rule.rule.startswith("/api/"):
        print(f"{','.join(sorted(rule.methods))}  {rule.rule}")

print("\n" + "=" * 55)
print("2. BACKEND CATEGORY MAPPINGS")
print("=" * 55)

print("CATEGORY_LABELS:")
for category_id, label in CATEGORY_LABELS.items():
    print(f"  {category_id!r} -> {label!r}")

print("\nCategory resolver tests:")
_test_categories = [
    "general_support",
    "General online safety support",
    "general-online-safety-support",
    "cybercrime",
    "Cybercrime reporting",
    "image_based_abuse",
    "Non-consensual intimate imagery",
    "online_harassment",
    "Online harassment support",
    "coercion",
    "Threats and coercion",
]

for candidate in _test_categories:
    try:
        print(f"  {candidate!r} -> {resolve_category(candidate)!r}")
    except Exception as exc:
        print(f"  {candidate!r} -> ERROR: {exc}")

print("\n" + "=" * 55)
print("3. EXISTING TIMELINE FUNCTIONS")
print("=" * 55)

_timeline_functions = [
    (name, obj)
    for name, obj in list(globals().items())
    if "timeline" in name.casefold()
    and callable(obj)
]

if not _timeline_functions:
    print("No callable timeline functions found in the current kernel.")
else:
    for name, obj in _timeline_functions:
        try:
            print(f"{name}{inspect.signature(obj)}")
        except Exception:
            print(name)

print("\nInspection complete. No routes or functions were changed.")

In [ ]:
import inspect
from dataclasses import fields, is_dataclass

def inspect_signalher_object(name):
    obj = globals().get(name)

    print("\n" + "=" * 65)
    print(name)
    print("=" * 65)

    if obj is None:
        print("Not defined in the current kernel.")
        return

    try:
        print("Signature:", inspect.signature(obj))
    except Exception:
        pass

    if is_dataclass(obj) and isinstance(obj, type):
        print("Dataclass fields:")
        for field in fields(obj):
            print(f"  {field.name}: {field.type}")

    try:
        print(inspect.getsource(obj))
    except Exception as exc:
        print("Source unavailable:", exc)


for name in [
    "TimelineEntry",
    "analyze_timeline",
    "print_timeline_summary",
    "resolve_category",
    "build_safe_search_query",
    "api_resources",
    "api_search_query",
    "api_categories",
]:
    inspect_signalher_object(name)

print("\n" + "=" * 65)
print("CURRENT TIMELINE API REGISTRATION")
print("=" * 65)

for rule in signalher_app.url_map.iter_rules():
    if "timeline" in rule.rule.casefold():
        print(rule.rule, sorted(rule.methods))

In [ ]:
# SIGNALHER — API INTEGRATION REPAIR
# Repairs category aliases, Google Programmable Search response,
# and the missing timeline endpoint.
#
# Uses the existing Incident model, analyzer, timeline analyzer,
# resource lookup, Flask application, and current API helpers.

import os as _sh_os
import re as _sh_re
import time as _sh_time
import json as _sh_json
import traceback as _sh_traceback

from datetime import datetime as _sh_datetime
from threading import Thread as _sh_Thread
from urllib.parse import urlencode as _sh_urlencode
from urllib.request import Request as _sh_HTTPRequest
from urllib.request import urlopen as _sh_urlopen
from urllib.error import HTTPError as _sh_HTTPError
from urllib.error import URLError as _sh_URLError

from flask import request as _sh_flask_request
from flask import jsonify as _sh_jsonify
from werkzeug.serving import make_server as _sh_make_server


# ------------------------------------------------------------
# 1. Restore Flask's request object safely
# ------------------------------------------------------------

request = _sh_flask_request

print("STEP 1: Flask request object restored.")


# ------------------------------------------------------------
# 2. Stop the existing server before updating its route table
# ------------------------------------------------------------

_old_server = globals().get("_signalher_http_server")
_old_thread = globals().get("_signalher_server_thread")

if _old_server is not None:
    if _old_thread is not None and _old_thread.is_alive():
        _old_server.shutdown()
        _old_thread.join(timeout=8)

        if _old_thread.is_alive():
            raise RuntimeError(
                "The old Flask server has not stopped. "
                "No replacement server was started."
            )

    _old_server.server_close()
    print("STEP 2: Existing server stopped cleanly.")
else:
    print("STEP 2: No existing server object was found.")


# ------------------------------------------------------------
# 3. Accept the frontend's category IDs and display labels
# ------------------------------------------------------------

def _sh_normalize_category(value):
    return _sh_re.sub(
        r"[^a-z0-9]+",
        " ",
        str(value).casefold(),
    ).strip()


_SIGNALHER_CATEGORY_ALIASES = {
    "general": "general_support",
    "general support": "general_support",
    "general online safety support": "general_support",

    "cybercrime": "cybercrime",
    "cybercrime reporting": "cybercrime",

    "image abuse": "image_based_abuse",
    "image based abuse": "image_based_abuse",
    "intimate image abuse support": "image_based_abuse",
    "non consensual intimate imagery": "image_based_abuse",

    "harassment": "online_harassment",
    "online harassment": "online_harassment",
    "online harassment support": "online_harassment",
}

# Accept all existing backend IDs and labels as well.
for _category_id, _category_label in CATEGORY_LABELS.items():
    _SIGNALHER_CATEGORY_ALIASES[
        _sh_normalize_category(_category_id)
    ] = _category_id

    _SIGNALHER_CATEGORY_ALIASES[
        _sh_normalize_category(_category_label)
    ] = _category_id


def resolve_category(category):
    """Resolve supported IDs, frontend aliases, and display labels."""
    if not isinstance(category, str):
        return None

    return _SIGNALHER_CATEGORY_ALIASES.get(
        _sh_normalize_category(category)
    )


# Provide distinct general queries for the four frontend topics.
# build_safe_search_query() will still append "India".
_SIGNALHER_SEARCH_TERMS = {
    "general_support": "official online safety support",
    "cybercrime": "official Indian cybercrime reporting portal",
    "image_based_abuse": (
        "official non-consensual intimate image abuse "
        "support and removal"
    ),
    "online_harassment": (
        "official online harassment and cyberbullying "
        "support and reporting"
    ),
}

for _category_id, _terms in _SIGNALHER_SEARCH_TERMS.items():
    SAFE_SEARCH_TERMS[_category_id] = _terms
    SAFE_SEARCH_TERMS[_sh_normalize_category(_category_id)] = _terms
    SAFE_SEARCH_TERMS[_category_id.replace("_", " ")] = _terms

print("STEP 3: Category compatibility installed.")

for _category in ("general", "cybercrime", "image_abuse", "harassment"):
    print(f"  {_category} -> {resolve_category(_category)}")


# ------------------------------------------------------------
# 4. Locate a configured Google Programmable Search engine ID
# ------------------------------------------------------------

_SH_CSE_VARIABLE_NAMES = (
    "GOOGLE_CSE_ID",
    "GOOGLE_CSE_CX",
    "GOOGLE_SEARCH_ENGINE_ID",
    "GOOGLE_PROGRAMMABLE_SEARCH_ENGINE_ID",
    "PROGRAMMABLE_SEARCH_ENGINE_ID",
    "SEARCH_ENGINE_ID",
    "CSE_ID",
    "CSE_CX",
    "GOOGLE_CX",
    "google_cse_id",
    "google_cse_cx",
    "search_engine_id",
    "cse_id",
    "cse_cx",
    "google_cx",
)

_SH_CSE_ID = None

# First look for an existing notebook variable.
for _name in _SH_CSE_VARIABLE_NAMES:
    _value = globals().get(_name)

    if isinstance(_value, str) and _value.strip():
        _SH_CSE_ID = _value.strip()
        break

# Then check common environment-variable names.
if not _SH_CSE_ID:
    for _name in _SH_CSE_VARIABLE_NAMES:
        _value = _sh_os.environ.get(_name)

        if _value and _value.strip():
            _SH_CSE_ID = _value.strip()
            break

print(
    "STEP 4: Google search engine ID found:",
    "Yes" if _SH_CSE_ID else "No",
)
# The ID itself is deliberately not printed.


# ------------------------------------------------------------
# 5. Replace only the search-query route implementation
# ------------------------------------------------------------

def _sh_api_search_query():
    payload = request.get_json(silent=True)

    if not isinstance(payload, dict):
        return _sh_jsonify({
            "success": False,
            "error": "Send a JSON object containing a category.",
        }), 400

    category_id = resolve_category(payload.get("category"))

    if category_id is None:
        return _sh_jsonify({
            "success": False,
            "error": "Unsupported category.",
            "allowed_categories": [
                "general",
                "cybercrime",
                "image_abuse",
                "harassment",
            ],
        }), 400

    safe_query = build_safe_search_query(category_id)

    if not _SH_CSE_ID:
        return _sh_jsonify({
            "success": False,
            "error": (
                "Google Programmable Search is not configured in "
                "this backend session. Set the existing search "
                "engine ID before opening search results."
            ),
        }), 503

    search_url = (
        "https://cse.google.com/cse?"
        + _sh_urlencode({
            "cx": _SH_CSE_ID,
            "q": safe_query,
        })
    )

    return _sh_jsonify({
        "success": True,
        "category": category_id,
        "query": safe_query,
        "search_url": search_url,
        "scope": "India",
        "privacy_note": (
            "Only a general topic-based query is generated. "
            "Opening the search URL sends that query to Google; "
            "private incident text is not included."
        ),
    }), 200


# Keep the existing URL and its endpoint name.
signalher_app.view_functions["api_search_query"] = (
    _sh_api_search_query
)

print("STEP 5: Search API response contract updated.")


# ------------------------------------------------------------
# 6. Restore the missing timeline endpoint
# ------------------------------------------------------------

def _sh_api_analyze_timeline():
    payload = request.get_json(silent=True)

    if not isinstance(payload, dict):
        return _sh_jsonify({
            "success": False,
            "error": "Send the timeline as a JSON object.",
        }), 400

    raw_entries = payload.get("entries")

    if not isinstance(raw_entries, list) or not raw_entries:
        return _sh_jsonify({
            "success": False,
            "error": "Provide at least one timeline entry.",
        }), 400

    timeline_entries = []

    try:
        for raw_entry in raw_entries:
            if not isinstance(raw_entry, dict):
                raise ValueError(
                    "Each timeline entry must be a JSON object."
                )

            incident = _build_incident_from_payload(raw_entry)

            direction = str(
                raw_entry.get("direction", "unknown")
            ).strip().casefold()

            if direction not in {"incoming", "outgoing", "unknown"}:
                direction = "unknown"

            raw_case_id = raw_entry.get("case_id")
            case_id = (
                str(raw_case_id).strip()
                if raw_case_id is not None
                and str(raw_case_id).strip()
                else None
            )

            timeline_entries.append(
                TimelineEntry(
                    incident=incident,
                    direction=direction,
                    case_id=case_id,
                )
            )

        # Use your existing chronology and case-linking logic.
        timeline_result = analyze_timeline(timeline_entries)

        findings = [
            finding
            for event in timeline_result.get("events", [])
            for finding in event.get("findings", [])
        ]

        labels = list(dict.fromkeys(
            finding.get("label", "Configured warning indicator")
            for finding in findings
        ))

        if labels:
            timeline_result["summary"] = (
                "Potential warning indicators to review: "
                + "; ".join(labels)
            )
        else:
            timeline_result["summary"] = (
                "No configured warning pattern matched this message. "
                "That does not establish that the interaction is safe."
            )

        timeline_result["case_id"] = (
            timeline_entries[0].case_id
            or timeline_entries[0].incident.incident_id
        )

        timeline_result["checks_performed"] = [
            "Configured rule-based indicator checks",
            "Chronological ordering of submitted entries",
            "Case-linked observations where a case ID was provided",
        ]

        timeline_result["closing_note"] = (
            "These suggestions are not guarantees; you decide "
            "which next steps are appropriate for your situation."
        )

        # Derive actionable guidance from the actual findings.
        actions = []
        seen_actions = set()

        for finding in findings:
            suggestion = finding.get("suggested_action", "").strip()
            label = finding.get("label", "Review the finding")
            identity = (label, suggestion)

            if not suggestion or identity in seen_actions:
                continue

            seen_actions.add(identity)

            actions.append({
                "title": label,
                "priority": "Review",
                "rationale": finding.get("explanation", ""),
                "steps": [suggestion],
                "user_control": (
                    "Choose the steps that fit your circumstances."
                ),
            })

        if not actions:
            actions.append({
                "title": "Review the interaction in context",
                "priority": "Consider",
                "rationale": (
                    "No configured warning pattern matched. "
                    "The automated check cannot establish safety."
                ),
                "steps": [
                    "Review the wider context and seek trusted support "
                    "if anything feels concerning."
                ],
                "user_control": (
                    "You decide whether any further action is needed."
                ),
            })

        # Add relevant official resource references where available.
        official_resources = []
        seen_resource_urls = set()

        categories = list(dict.fromkeys(
            finding.get("category")
            for finding in findings
            if finding.get("category")
        ))

        for category in categories:
            try:
                candidate_resources = find_resources(
                    category=category,
                    state=None,
                    country="India",
                    limit=3,
                )
            except Exception:
                candidate_resources = []

            for resource in candidate_resources:
                url = (
                    resource.get("url")
                    or resource.get("official_url")
                    or resource.get("website")
                )

                if not url or url in seen_resource_urls:
                    continue

                seen_resource_urls.add(url)

                official_resources.append({
                    "name": resource.get("name", "Official support resource"),
                    "url": url,
                    "purpose": resource.get("purpose", ""),
                })

        timeline_result["official_resources"] = official_resources

        action_plan = {
            "title": "Suggested next steps",
            "actions": actions,
        }

        return _sh_jsonify({
            "success": True,
            "timeline_result": _make_json_safe(timeline_result),
            "action_plan": _make_json_safe(action_plan),
        }), 200

    except ValueError as exc:
        return _sh_jsonify({
            "success": False,
            "error": str(exc),
        }), 400

    except Exception as exc:
        # Do not print or return submitted message contents.
        print(
            "SignalHer timeline endpoint error:",
            type(exc).__name__,
        )
        return _sh_jsonify({
            "success": False,
            "error": (
                "Timeline review could not be completed. "
                "Check the existing timeline model and analysis functions."
            ),
        }), 500


# Flask has already handled requests, so briefly reopen setup registration.
_existing_setup_state = signalher_app._got_first_request
signalher_app._got_first_request = False

try:
    _timeline_rule = next(
        (
            rule
            for rule in signalher_app.url_map.iter_rules()
            if rule.rule == "/api/analyze-timeline"
            and "POST" in rule.methods
        ),
        None,
    )

    if _timeline_rule:
        # Replace an existing timeline view without adding a duplicate URL.
        signalher_app.view_functions[
            _timeline_rule.endpoint
        ] = _sh_api_analyze_timeline
    else:
        signalher_app.add_url_rule(
            "/api/analyze-timeline",
            endpoint="api_analyze_timeline",
            view_func=_sh_api_analyze_timeline,
            methods=["POST"],
        )
finally:
    signalher_app._got_first_request = _existing_setup_state

print("STEP 6: Timeline endpoint registered.")


# ------------------------------------------------------------
# 7. Start Flask again with the repaired application
# ------------------------------------------------------------

_signalher_http_server = _sh_make_server(
    host="127.0.0.1",
    port=5000,
    app=signalher_app,
    threaded=True,
)

_signalher_server_thread = _sh_Thread(
    target=_signalher_http_server.serve_forever,
    daemon=True,
    name="SignalHerLocalServer",
)

_signalher_server_thread.start()
_sh_time.sleep(0.3)

print("STEP 7: Server restarted at http://127.0.0.1:5000")
print("Server thread running:", _signalher_server_thread.is_alive())


# ------------------------------------------------------------
# 8. Test the live API
# ------------------------------------------------------------

def _sh_test_live_api(path, payload=None):
    url = "http://127.0.0.1:5000" + path
    headers = {
        "Origin": "http://127.0.0.1:5500",
        "Accept": "application/json",
    }

    body = None
    method = "GET"

    if payload is not None:
        body = _sh_json.dumps(payload).encode("utf-8")
        headers["Content-Type"] = "application/json"
        method = "POST"

    http_request = _sh_HTTPRequest(
        url,
        data=body,
        headers=headers,
        method=method,
    )

    try:
        with _sh_urlopen(http_request, timeout=8) as response:
            raw = response.read().decode("utf-8")
            data = _sh_json.loads(raw) if raw else {}

            print(f"\n{method} {path}: HTTP {response.status}")
            print(
                "CORS:",
                response.headers.get("Access-Control-Allow-Origin"),
            )

            if path == "/api/search-query":
                print("Success:", data.get("success"))
                print("Query:", data.get("query"))
                print("Search URL returned:", bool(data.get("search_url")))
                if not data.get("success"):
                    print("Error:", data.get("error"))

            elif path == "/api/analyze-timeline":
                print("Success:", data.get("success"))
                print(
                    "Timeline events:",
                    len(
                        data.get("timeline_result", {}).get("events", [])
                    ),
                )
                print(
                    "Action count:",
                    len(
                        data.get("action_plan", {}).get("actions", [])
                    ),
                )
                if not data.get("success"):
                    print("Error:", data.get("error"))

            elif not data.get("success", False):
                print("Error:", data.get("error"))

            return response.status, data

    except _sh_HTTPError as exc:
        body_text = exc.read().decode("utf-8", errors="replace")
        print(f"\n{method} {path}: HTTP {exc.code}")
        print("Response:", body_text)
        return exc.code, None

    except (_sh_URLError, TimeoutError, OSError) as exc:
        print(f"\n{method} {path}: CONNECTION ERROR")
        print(repr(exc))
        return None, None


_sh_test_live_api("/api/health")
_sh_test_live_api("/api/resources")

for _topic in ("general", "cybercrime", "image_abuse", "harassment"):
    _sh_test_live_api(
        "/api/search-query",
        {"category": _topic},
    )

_sh_test_platform = next(iter(Platform))

_sh_test_live_api(
    "/api/analyze-timeline",
    {
        "entries": [
            {
                "platform": _sh_test_platform.value,
                "occurred_at": "2026-10-25T10:00:00+05:30",
                "message_text": (
                    "If you don't send money, "
                    "I will expose your private images."
                ),
                "user_note": "Synthetic timeline integration test.",
                "source_reference": "",
                "incident_id": "signalher-timeline-integration-test",
                "direction": "unknown",
                "case_id": "signalher-timeline-integration-test",
            }
        ]
    },
)

print("\nAPI integration repair and live tests completed.")
print("Save this notebook with Ctrl+S.")

In [ ]:
# Configure the existing Google Programmable Search engine.
_SH_CSE_ID = "PASTE_YOUR_ACTUAL_SEARCH_ENGINE_ID_HERE".strip()

if not _SH_CSE_ID or _SH_CSE_ID == "# Configure the existing Google Programmable Search engine.
_SH_CSE_ID = "PASTE_YOUR_ACTUAL_SEARCH_ENGINE_ID_HERE".strip()

if not _SH_CSE_ID or _SH_CSE_ID == "PASTE_YOUR_ACTUAL_SEARCH_ENGINE_ID_HERE":
    raise ValueError("Please paste your actual Search engine ID.")

print("Google Programmable Search engine configured.")
print("The ID has been set; its value will not be displayed.")":
    raise ValueError("Please paste your actual Search engine ID.")

print("Google Programmable Search engine configured.")
print("The ID has been set; its value will not be displayed.")

In [ ]:
_SH_CSE_ID = "d536c3fcb3ce5432c"

In [ ]:
_sh_test_live_api("/api/search-query", {"category": "general"})

In [ ]:
dir /s /b *.ipynb